# STEM Digital Twin — v6 (Unified Diffraction · Specimen Realism · Simulation Environments · 13 Samples)

This revision is a **testbed for developing and stress-testing microscope-automation
code before deployment on a real instrument**. It is not a substitute for a full
dynamical image-simulation package; it is an intermediate environment where the
*logic* of acquisition workflows can be exercised against realistic failure modes.

**What changed since the previous version**

1. **One diffraction channel for every sample.** Diffraction is now computed
   directly from atomic positions in the illuminated region (`|Σ_j f_j e^{2πi q·r_j}|²`
   on the Ewald sphere). Crystals give spots, polycrystals give ring-like patterns,
   amorphous material gives diffuse halos — all from the *same* code path. The old
   FFT-of-image fallback for non-FCC/BCC/HCP samples is gone.
2. **Four new samples** exercising sample classes the reviewers asked for:
   `polycrystal_grains` (multiple randomly-oriented grains), `dislocation_crystal`
   (edge-dislocation displacement field), `amorphous_film` (random close packing →
   diffuse diffraction), and `shape_assembly` (synthetic convex features; a testbed
   for feature-finding workflows).
3. **Specimen-degradation effects:** beam damage (cumulative dose removes signal in
   the exposed region) and contamination (carbon-like overlayer darkens where the
   beam dwells), alongside the existing mechanical drift.
4. **Simulation environments:** named scenarios (`pristine`, `beam_sensitive`,
   `contaminating`, `thick_drifting`, `low_dose`) bundle realism settings so you can
   run the same code under different conditions.
5. **Failure-aware autofocus:** autofocus now returns a `converged` flag and can
   genuinely diverge (e.g. on beam-sensitive specimens where damage corrupts the
   focus sweep). Convergence is judged from the sharpness-curve shape.
6. **Control / simulation split.** The client is now two objects: a
   `MicroscopeControlClient` whose every method has a real-instrument counterpart
   (the portable surface you'd replace with a vendor SDK), and a `SimulationHarness`
   for twin-only configuration (specimen, environment, injected degradation) that a
   real deployment has no equivalent of. Workflows code against control only;
   `sim` is test scaffolding. This makes the "test here, deploy there" boundary
   explicit rather than implied. 

**New in this revision (instrument realism)**
- **Contamination now brightens** the scanned region (HAADF signal scales with
  mass-thickness, and carbon adds mass), while beam **damage** darkens it — the two
  effects have physically correct opposite signs.
- **Stage safety limits**: ±1.5 mm (x/y), ±1 mm (z), ±30° (α/β). Out-of-range moves
  are rejected, like a real soft-limit.
- **Magnification ↔ field of view** (`mag = k/fov`, calibrated to a real instrument),
  exposed on the control client and every backend.
- **Inherent length scales**: each sample declares its finest feature size, so you
  must raise magnification to resolve detail (and then drift/dose dominate).
- The instrument-control surface is **identical across the twin and every vendor
  backend** (ThermoFisher / Nion / JEOL-PyJEM): same methods, signatures, and units.

> ⚠ If you see `Connection refused` or `ReactorAlreadyRunning`: **Runtime → Restart
> session**, then run top to bottom.
>
> Note: diffraction-from-atoms is slower than a closed-form lookup (~1–5 s/frame).
> The atom budget is capped at 100k, so polycrystal powder rings show correct
> *radial* structure but are not fully continuous. See the closing limitations.


In [ ]:
!pip -q install twisted ase scipy


## 1. Build the `samples/` package

In [ ]:
import os
os.makedirs('samples', exist_ok=True)
print('samples/ ready')


In [ ]:
%%writefile samples/base.py
"""
samples/base.py

Sample base class. Now carries an OPTIONAL crystallographic `lattice`
descriptor so the server can compute real kinematical diffraction for
crystalline samples. Non-crystalline samples leave `lattice=None` and the
server falls back to an FFT-based diffraction proxy.
"""
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional


@dataclass
class SampleMetadata:
    name: str
    display_name: str
    description: str
    default_params: Dict[str, Any] = field(default_factory=dict)
    param_schema: Dict[str, Any] = field(default_factory=dict)


@dataclass
class CrystalLattice:
    """
    Minimal crystallographic description sufficient for kinematical
    diffraction-spot generation.

    - real_vectors: 3x3 array, rows are the real-space lattice vectors a1,a2,a3
      in Angstroms. The crystal is assumed aligned so a3 is roughly along the
      beam (z) at zero tilt.
    - basis: list of (fractional_xyz, atomic_number) for atoms in the unit cell.
    - name: label for display.
    """
    real_vectors: np.ndarray
    basis: List[Any]
    name: str = "crystal"

    def reciprocal_vectors(self) -> np.ndarray:
        """Return 3x3 reciprocal lattice vectors (rows b1,b2,b3), 2*pi convention."""
        a1, a2, a3 = self.real_vectors[0], self.real_vectors[1], self.real_vectors[2]
        vol = np.dot(a1, np.cross(a2, a3))
        b1 = 2 * np.pi * np.cross(a2, a3) / vol
        b2 = 2 * np.pi * np.cross(a3, a1) / vol
        b3 = 2 * np.pi * np.cross(a1, a2) / vol
        return np.array([b1, b2, b3], dtype=np.float64)

    def structure_factor(self, h, k, l) -> complex:
        """Kinematical structure factor F_hkl = sum_j Z_j exp(2pi i (h x + k y + l z))."""
        F = 0.0 + 0.0j
        for frac, Z in self.basis:
            fx, fy, fz = frac
            phase = 2 * np.pi * (h * fx + k * fy + l * fz)
            F += Z * np.exp(1j * phase)
        return F


class Sample:
    meta: SampleMetadata = None
    sample_fov_um: float = 200.0
    # Inherent length scale of the sample's finest meaningful detail, in
    # nanometres. Used by the server as a resolution limit: when the current
    # pixel size (FOV / magnification) is coarser than this, the fine detail is
    # blurred out, so the user must raise magnification to resolve it (and then
    # drift / dose become the limiting factors, as on a real instrument).
    # 0 = no inherent scale (feature always resolvable). Subclasses override.
    feature_scale_nm: float = 0.0
    tilt_strength_px_per_slice: float = 0.35
    # Optional crystallographic descriptor. If set (a CrystalLattice), the
    # server will generate real kinematical diffraction spots. If None, the
    # server uses the FFT proxy.
    lattice: Optional[CrystalLattice] = None

    def __init__(self, **params):
        defaults = self.meta.default_params if self.meta else {}
        self.params = {**defaults, **params}

    def generate_volume(self, D, H, W):
        raise NotImplementedError

    def get_lattice(self):
        """Return the crystal lattice for diffraction, or None if amorphous."""
        return self.lattice

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        """
        Default implementation for particle-based samples. If the sample has
        recorded `self._particles` (list of {'center_vox','radii_vox'}) and
        `self._vol_shape` during generate_volume, fill the in-region particles
        with atoms. Crystalline particles -> spots/rings; amorphous -> halos.
        Samples with their own lattice tiling override this.

        Returns (None, None) if the sample has no atomic structure.
        """
        if not hasattr(self, "_particles") or not hasattr(self, "_vol_shape"):
            return None, None
        D, H, W = self._vol_shape
        px_per_um = W / self.sample_fov_um
        shifted = []
        for part in self._particles:
            pz, py, px = part["center_vox"]
            shifted.append({"center_vox": (pz, py - H/2.0, px - W/2.0),
                            "radii_vox": part["radii_vox"]})
        amorphous = not getattr(self, "crystalline_particles", True)
        return atoms_in_particles(
            shifted, cx_um, cy_um, half_width_um, depth_nm, px_per_um,
            amorphous=amorphous,
            random_orientation=getattr(self, "particles_random_orientation", True))


def tile_lattice_in_region(lattice, half_width_A, depth_A):
    """
    Generate atom positions for a lattice tiled into a box centered at the
    origin of extent [-half_width_A, +half_width_A] in x,y and [-depth_A/2,
    +depth_A/2] in z. Returns (positions, atomic_numbers) where positions are
    in Angstroms relative to the box center.

    Vectorized: builds the integer cell index grid in numpy and filters in one
    shot rather than looping in Python.
    """
    a1, a2, a3 = lattice.real_vectors
    # Cell-index bracket that conservatively covers the box
    max_cell_xy = int(np.ceil(2.0 * half_width_A / min(np.linalg.norm(a1[:2]),
                                                       np.linalg.norm(a2[:2]),
                                                       1.0))) + 2
    max_cell_z = int(np.ceil(depth_A / max(1e-6, abs(a3[2])))) + 2

    # Build cell-index grid
    ii = np.arange(-max_cell_xy, max_cell_xy + 1)
    jj = np.arange(-max_cell_xy, max_cell_xy + 1)
    kk = np.arange(-max_cell_z, max_cell_z + 1)
    I, J, K = np.meshgrid(ii, jj, kk, indexing='ij')
    # Cell origins (N, 3)
    cell_origins = (I[..., None] * a1 + J[..., None] * a2 + K[..., None] * a3).reshape(-1, 3)

    all_pos = []; all_Z = []
    for frac, Z in lattice.basis:
        fx, fy, fz = frac
        offset = fx * a1 + fy * a2 + fz * a3
        positions = cell_origins + offset
        mask = ((np.abs(positions[:, 0]) <= half_width_A) &
                (np.abs(positions[:, 1]) <= half_width_A) &
                (np.abs(positions[:, 2]) <= depth_A / 2))
        kept = positions[mask]
        all_pos.append(kept)
        all_Z.append(np.full(len(kept), Z, dtype=np.int32))

    if not all_pos or sum(len(p) for p in all_pos) == 0:
        return np.zeros((0, 3), dtype=np.float64), np.zeros(0, dtype=np.int32)
    return np.concatenate(all_pos).astype(np.float64), np.concatenate(all_Z)


# ============================================================================
# Particle-based sample support (unified diffraction)
# ============================================================================
# Default Au FCC lattice for filling crystalline nanoparticles with atoms.
_AU_FCC = None

def _get_au_fcc():
    global _AU_FCC
    if _AU_FCC is None:
        a = 4.05
        _AU_FCC = CrystalLattice(
            real_vectors=np.array([[a,0,0],[0,a,0],[0,0,a]], dtype=np.float64),
            basis=[((0,0,0),79),((0,0.5,0.5),79),((0.5,0,0.5),79),((0.5,0.5,0),79)],
            name="Au-FCC")
    return _AU_FCC


def atoms_in_particles(particles, cx_um, cy_um, half_width_um, depth_nm,
                       px_per_um, atom_cap=100000, lattice=None,
                       amorphous=False, amorphous_number_density=0.06,
                       particle_nm_per_vox=0.5, random_orientation=True,
                       orientation_seed=12345, rng=None):
    """
    Fill the particles that fall within the requested region with atoms.

    In-plane PLACEMENT of particles uses the display voxel scale (so particles
    sit where they appear in the image). Each particle's atomic FILLING uses a
    physical scale (particle_nm_per_vox) decoupled from the often-coarse display
    scale, so particle sizes and interatomic spacings come out realistic.

    random_orientation: if True, each crystalline particle is given a random,
        but deterministic (seeded by particle index), 3D orientation. An
        ensemble of many randomly-oriented small crystallites produces
        powder-ring-like diffraction -- the physically correct result for
        dispersed nanoparticles -- rather than a single-crystal spot pattern.

    particles: list of {'center_vox': (z,y,x), 'radii_vox': (rz,ry,rx)} with the
               volume CENTER as origin (in display voxels).
    Returns (positions_A, atomic_numbers) in Angstroms relative to region center.
    """
    if rng is None:
        rng = np.random.default_rng(0)
    if lattice is None:
        lattice = _get_au_fcc()

    half_vox = half_width_um * px_per_um
    cx_vox_off = cx_um * px_per_um
    cy_vox_off = cy_um * px_per_um
    disp_A_per_vox = (1.0 / px_per_um) * 1e4
    fill_A_per_vox = particle_nm_per_vox * 10.0

    def _random_rot(seed):
        r = np.random.default_rng(seed)
        ax = r.normal(size=3); ax /= (np.linalg.norm(ax) + 1e-12)
        ang = r.uniform(0, 2*np.pi)
        K = np.array([[0,-ax[2],ax[1]],[ax[2],0,-ax[0]],[-ax[1],ax[0],0]])
        return np.eye(3) + np.sin(ang)*K + (1-np.cos(ang))*(K@K)

    all_pos = []
    all_Z = []
    total = 0
    for p_idx, part in enumerate(particles):
        cz, cy, cx = part['center_vox']
        rz, ry, rx = part['radii_vox']
        rel_x = cx - cx_vox_off
        rel_y = cy - cy_vox_off
        if (abs(rel_x) - rx > half_vox) or (abs(rel_y) - ry > half_vox):
            continue

        Rx_A = rx * fill_A_per_vox
        Ry_A = ry * fill_A_per_vox
        Rz_A = rz * fill_A_per_vox
        ox_A = rel_x * disp_A_per_vox
        oy_A = rel_y * disp_A_per_vox

        if amorphous:
            vol_A3 = (4.0/3.0) * np.pi * Rx_A * Ry_A * Rz_A
            n_at = int(min(20000, max(8, vol_A3 * amorphous_number_density)))
            u = rng.normal(size=(n_at*2, 3))
            u /= (np.linalg.norm(u, axis=1, keepdims=True) + 1e-9)
            radii_frac = rng.uniform(0, 1, size=(n_at*2,)) ** (1.0/3.0)
            pts = (u * radii_frac[:, None])[:n_at]
            pts_A = pts * np.array([Rx_A, Ry_A, Rz_A])
            pos = np.stack([pts_A[:,0] + ox_A, pts_A[:,1] + oy_A, pts_A[:,2]], axis=1)
            Z = np.full(len(pos), 79, dtype=np.int32)
        else:
            bp, bZ = tile_lattice_in_region(lattice,
                                            half_width_A=max(Rx_A, Ry_A),
                                            depth_A=2*max(Rx_A, Ry_A, Rz_A))
            if len(bp) == 0:
                continue
            # rotate the crystal lattice for this particle (random but deterministic)
            if random_orientation:
                R = _random_rot(orientation_seed + p_idx)
                bp = bp @ R.T
            ex = bp[:,0] / (Rx_A + 1e-6)
            ey = bp[:,1] / (Ry_A + 1e-6)
            ez = bp[:,2] / (Rz_A + 1e-6)
            inside = (ex*ex + ey*ey + ez*ez) <= 1.0
            bp = bp[inside]; bZ = bZ[inside]
            bp = bp + np.array([ox_A, oy_A, 0.0])
            pos = bp; Z = bZ

        all_pos.append(pos)
        all_Z.append(Z)
        total += len(pos)
        if total > atom_cap:
            break

    if not all_pos or total == 0:
        return np.zeros((0,3), dtype=np.float64), np.zeros(0, dtype=np.int32)
    pos = np.concatenate(all_pos)
    Z = np.concatenate(all_Z)
    if len(pos) > atom_cap:
        idx = rng.choice(len(pos), atom_cap, replace=False)
        pos = pos[idx]; Z = Z[idx]
    return pos.astype(np.float64), Z.astype(np.int32)


In [ ]:
%%writefile samples/__init__.py
import importlib
import pkgutil
from typing import Dict, Type
from .base import Sample, SampleMetadata

_REGISTRY: Dict[str, Type[Sample]] = {}


def register(cls: Type[Sample]) -> Type[Sample]:
    """Decorator: registers a Sample subclass by its meta.name."""
    if cls.meta is None:
        raise ValueError(f"{cls.__name__} must define a `meta` SampleMetadata")
    _REGISTRY[cls.meta.name] = cls
    return cls


def discover(force_reload: bool = False):
    """
    Rescan the samples/ directory and import any sample modules not yet loaded.

    Safe to call repeatedly. This is essential in environments (like Colab)
    where sample files may be written AFTER the package was first imported —
    a stale cached `samples` module would otherwise have an empty registry.

    If `force_reload=True`, already-imported sample modules are reloaded so
    their @register decorators run again. Useful after the registry has been
    reset (e.g. by importlib.reload on the package itself).
    """
    import sys as _sys
    import samples as pkg
    importlib.invalidate_caches()
    for _, modname, _ in pkgutil.iter_modules(pkg.__path__):
        if modname in ("base", "__init__"):
            continue
        full = f"samples.{modname}"
        if force_reload and full in _sys.modules:
            importlib.reload(_sys.modules[full])
        else:
            importlib.import_module(full)


def list_samples():
    discover()  # ensure fresh scan before every call
    return [
        {
            "name": cls.meta.name,
            "display_name": cls.meta.display_name,
            "description": cls.meta.description,
            "default_params": cls.meta.default_params,
            "param_schema": cls.meta.param_schema,
        }
        for cls in _REGISTRY.values()
    ]


def get_sample(name: str, **params) -> Sample:
    discover()  # ensure fresh scan before every call
    if name not in _REGISTRY:
        raise KeyError(
            f"Unknown sample '{name}'. Available: {list(_REGISTRY.keys())}"
        )
    return _REGISTRY[name](**params)


# Initial discovery at import time
discover()


### Crystalline samples (atomic lattice → spots; tilt navigates zone axes)

In [ ]:
%%writefile samples/fcc_single_crystal.py
import numpy as np
from .base import Sample, SampleMetadata, CrystalLattice
from . import register


@register
class FCCSingleCrystal(Sample):
    feature_scale_nm = 0.2   # atomic-column spacing (~0.2 nm)
    meta = SampleMetadata(
        name="fcc_single_crystal",
        display_name="FCC Single Crystal",
        description="Face-centered cubic single-crystal volume aligned with axes.",
        default_params={
            "a_px": 24,
            "sigma_px": 1.1,
            "base_level": 80.0,
            "atom_intensity": 9000.0,
            "a_angstrom": 4.05,   # physical lattice parameter (Au-like), for diffraction
            "atomic_number": 79,  # Au
        },
        param_schema={
            "a_px":           {"type": "int",   "min": 8,   "max": 64},
            "sigma_px":       {"type": "float", "min": 0.5, "max": 4.0},
            "base_level":     {"type": "float", "min": 0,   "max": 1000},
            "atom_intensity": {"type": "float", "min": 100, "max": 60000},
            "a_angstrom":     {"type": "float", "min": 1.0, "max": 20.0},
            "atomic_number":  {"type": "int",   "min": 1,   "max": 100},
        },
    )

    def __init__(self, **params):
        super().__init__(**params)
        a = float(self.params["a_angstrom"])
        Z = int(self.params["atomic_number"])
        # FCC conventional cell: cubic, 4-atom basis
        self.lattice = CrystalLattice(
            real_vectors=np.array([[a, 0, 0], [0, a, 0], [0, 0, a]], dtype=np.float64),
            basis=[
                ((0.0, 0.0, 0.0), Z),
                ((0.0, 0.5, 0.5), Z),
                ((0.5, 0.0, 0.5), Z),
                ((0.5, 0.5, 0.0), Z),
            ],
            name="FCC",
        )

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        """For a uniform crystal, the diffraction pattern is the same anywhere.
        We tile an approximately CUBIC region (equal extent in x, y, z) sized to
        the atom budget. A cube is important: a region much deeper than it is wide
        has an anisotropic shape transform (a broad pancake in qx/qy) that smears
        into a cloud at intermediate tilts. Keeping it cubic makes the shape
        transform isotropic so off-axis zone-axis patterns stay sparse and clean.
        (Thickness/relrod effects are exercised separately via the diffraction
        `thickness_nm` control, not by deforming this region.)"""
        from .base import tile_lattice_in_region
        a1, a2, a3 = self.lattice.real_vectors
        cell_vol = abs(np.dot(a1, np.cross(a2, a3)))
        density = len(self.lattice.basis) / cell_vol     # atoms / A^3
        target = 90000.0  # stay UNDER the 100k diffraction atom-cap so no random subsampling (which would break lattice interference and fill the background with a diffuse cloud)
        side_A = float((target / max(1e-9, density)) ** (1.0 / 3.0))
        half_A = side_A / 2.0
        return tile_lattice_in_region(self.lattice, half_A, side_A)

    def generate_volume(self, D, H, W):
        p = self.params
        a_px = int(p["a_px"])
        V = np.zeros((D, H, W), dtype=np.float32) + float(p["base_level"])

        basis = np.array([
            [0.0, 0.0, 0.0],
            [0.0, 0.5, 0.5],
            [0.5, 0.0, 0.5],
            [0.5, 0.5, 0.0],
        ], dtype=np.float32)

        nz = int(np.ceil(D / a_px)) + 2
        ny = int(np.ceil(H / a_px)) + 2
        nx = int(np.ceil(W / a_px)) + 2

        for iz in range(nz):
            for iy in range(ny):
                for ix in range(nx):
                    cell = np.array([iz, iy, ix], dtype=np.float32) * a_px
                    for b in basis:
                        pos = cell + b * a_px
                        z = int(round(pos[0]))
                        y = int(round(pos[1]))
                        x = int(round(pos[2]))
                        if 0 <= z < D and 0 <= y < H and 0 <= x < W:
                            V[z, y, x] += float(p["atom_intensity"])

        def gfreq(n, s):
            f = np.fft.fftfreq(n).astype(np.float32)
            return np.exp(-2.0 * (np.pi ** 2) * (s ** 2) * (f ** 2)).astype(np.float32)

        s = float(p["sigma_px"])
        gz, gy, gx = gfreq(D, s), gfreq(H, s), gfreq(W, s)
        F = np.fft.fftn(V)
        F *= gz[:, None, None]
        F *= gy[None, :, None]
        F *= gx[None, None, :]
        Vb = np.fft.ifftn(F).real.astype(np.float32)
        return np.clip(Vb, 0, 65535).astype(np.float32)


In [ ]:
%%writefile samples/bcc_single_crystal.py
"""
samples/bcc_single_crystal.py
Body-centered cubic single crystal (e.g. alpha-Fe, W, Mo). Two-atom basis.
Carries a CrystalLattice so the server can render real kinematical diffraction.
"""
import numpy as np
from .base import Sample, SampleMetadata, CrystalLattice
from . import register


@register
class BCCSingleCrystal(Sample):
    feature_scale_nm = 0.2   # atomic-column spacing (~0.2 nm)
    meta = SampleMetadata(
        name="bcc_single_crystal",
        display_name="BCC Single Crystal",
        description="Body-centered cubic single-crystal volume (Fe/W-like).",
        default_params={
            "a_px": 24,
            "sigma_px": 1.1,
            "base_level": 80.0,
            "atom_intensity": 9000.0,
            "a_angstrom": 2.87,    # alpha-Fe lattice parameter
            "atomic_number": 26,   # Fe
        },
        param_schema={
            "a_px":           {"type": "int",   "min": 8,   "max": 64},
            "sigma_px":       {"type": "float", "min": 0.5, "max": 4.0},
            "base_level":     {"type": "float", "min": 0,   "max": 1000},
            "atom_intensity": {"type": "float", "min": 100, "max": 60000},
            "a_angstrom":     {"type": "float", "min": 1.0, "max": 20.0},
            "atomic_number":  {"type": "int",   "min": 1,   "max": 100},
        },
    )

    def __init__(self, **params):
        super().__init__(**params)
        a = float(self.params["a_angstrom"])
        Z = int(self.params["atomic_number"])
        self.lattice = CrystalLattice(
            real_vectors=np.array([[a, 0, 0], [0, a, 0], [0, 0, a]], dtype=np.float64),
            basis=[
                ((0.0, 0.0, 0.0), Z),
                ((0.5, 0.5, 0.5), Z),
            ],
            name="BCC",
        )

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        """For a uniform crystal, the diffraction pattern is the same anywhere.
        We tile an approximately CUBIC region (equal extent in x, y, z) sized to
        the atom budget. A cube is important: a region much deeper than it is wide
        has an anisotropic shape transform (a broad pancake in qx/qy) that smears
        into a cloud at intermediate tilts. Keeping it cubic makes the shape
        transform isotropic so off-axis zone-axis patterns stay sparse and clean.
        (Thickness/relrod effects are exercised separately via the diffraction
        `thickness_nm` control, not by deforming this region.)"""
        from .base import tile_lattice_in_region
        a1, a2, a3 = self.lattice.real_vectors
        cell_vol = abs(np.dot(a1, np.cross(a2, a3)))
        density = len(self.lattice.basis) / cell_vol     # atoms / A^3
        target = 90000.0  # stay UNDER the 100k diffraction atom-cap so no random subsampling (which would break lattice interference and fill the background with a diffuse cloud)
        side_A = float((target / max(1e-9, density)) ** (1.0 / 3.0))
        half_A = side_A / 2.0
        return tile_lattice_in_region(self.lattice, half_A, side_A)

    def generate_volume(self, D, H, W):
        p = self.params
        a_px = int(p["a_px"])
        V = np.zeros((D, H, W), dtype=np.float32) + float(p["base_level"])

        basis = np.array([
            [0.0, 0.0, 0.0],
            [0.5, 0.5, 0.5],
        ], dtype=np.float32)

        nz = int(np.ceil(D / a_px)) + 2
        ny = int(np.ceil(H / a_px)) + 2
        nx = int(np.ceil(W / a_px)) + 2

        for iz in range(nz):
            for iy in range(ny):
                for ix in range(nx):
                    cell = np.array([iz, iy, ix], dtype=np.float32) * a_px
                    for b in basis:
                        pos = cell + b * a_px
                        z = int(round(pos[0]))
                        y = int(round(pos[1]))
                        x = int(round(pos[2]))
                        if 0 <= z < D and 0 <= y < H and 0 <= x < W:
                            V[z, y, x] += float(p["atom_intensity"])

        def gfreq(n, s):
            f = np.fft.fftfreq(n).astype(np.float32)
            return np.exp(-2.0 * (np.pi ** 2) * (s ** 2) * (f ** 2)).astype(np.float32)

        s = float(p["sigma_px"])
        gz, gy, gx = gfreq(D, s), gfreq(H, s), gfreq(W, s)
        F = np.fft.fftn(V)
        F *= gz[:, None, None]
        F *= gy[None, :, None]
        F *= gx[None, None, :]
        Vb = np.fft.ifftn(F).real.astype(np.float32)
        return np.clip(Vb, 0, 65535).astype(np.float32)


In [ ]:
%%writefile samples/hcp_single_crystal.py
"""
samples/hcp_single_crystal.py
Hexagonal close-packed single crystal (e.g. Ti, Mg, Zn, Co). Two-atom basis
in a hexagonal cell with c/a ratio. Carries a CrystalLattice so the server can
render real kinematical diffraction (note: hexagonal reciprocal lattice gives
the characteristic 6-fold diffraction symmetry down the c-axis).
"""
import numpy as np
from .base import Sample, SampleMetadata, CrystalLattice
from . import register


@register
class HCPSingleCrystal(Sample):
    feature_scale_nm = 0.2   # atomic-column spacing (~0.2 nm)
    meta = SampleMetadata(
        name="hcp_single_crystal",
        display_name="HCP Single Crystal",
        description="Hexagonal close-packed single-crystal volume (Ti/Mg-like).",
        default_params={
            "a_px": 22,            # in-plane lattice spacing in pixels
            "sigma_px": 1.1,
            "base_level": 80.0,
            "atom_intensity": 9000.0,
            "a_angstrom": 2.95,    # Ti basal lattice parameter
            "c_over_a": 1.587,     # ideal ~1.633; Ti ~1.587
            "atomic_number": 22,   # Ti
        },
        param_schema={
            "a_px":           {"type": "int",   "min": 8,   "max": 64},
            "sigma_px":       {"type": "float", "min": 0.5, "max": 4.0},
            "base_level":     {"type": "float", "min": 0,   "max": 1000},
            "atom_intensity": {"type": "float", "min": 100, "max": 60000},
            "a_angstrom":     {"type": "float", "min": 1.0, "max": 20.0},
            "c_over_a":       {"type": "float", "min": 1.0, "max": 2.5},
            "atomic_number":  {"type": "int",   "min": 1,   "max": 100},
        },
    )

    def __init__(self, **params):
        super().__init__(**params)
        a = float(self.params["a_angstrom"])
        c = a * float(self.params["c_over_a"])
        Z = int(self.params["atomic_number"])
        # Hexagonal lattice vectors:
        # a1 = a (1,0,0); a2 = a (-1/2, sqrt(3)/2, 0); a3 = c (0,0,1)
        a1 = np.array([a, 0.0, 0.0])
        a2 = np.array([-0.5 * a, np.sqrt(3) / 2 * a, 0.0])
        a3 = np.array([0.0, 0.0, c])
        self.lattice = CrystalLattice(
            real_vectors=np.array([a1, a2, a3], dtype=np.float64),
            basis=[
                ((0.0, 0.0, 0.0), Z),
                ((1.0/3.0, 2.0/3.0, 0.5), Z),
            ],
            name="HCP",
        )

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        """For a uniform crystal, the diffraction pattern is the same anywhere.
        We tile an approximately CUBIC region (equal extent in x, y, z) sized to
        the atom budget. A cube is important: a region much deeper than it is wide
        has an anisotropic shape transform (a broad pancake in qx/qy) that smears
        into a cloud at intermediate tilts. Keeping it cubic makes the shape
        transform isotropic so off-axis zone-axis patterns stay sparse and clean.
        (Thickness/relrod effects are exercised separately via the diffraction
        `thickness_nm` control, not by deforming this region.)"""
        from .base import tile_lattice_in_region
        a1, a2, a3 = self.lattice.real_vectors
        cell_vol = abs(np.dot(a1, np.cross(a2, a3)))
        density = len(self.lattice.basis) / cell_vol     # atoms / A^3
        target = 90000.0  # stay UNDER the 100k diffraction atom-cap so no random subsampling (which would break lattice interference and fill the background with a diffuse cloud)
        side_A = float((target / max(1e-9, density)) ** (1.0 / 3.0))
        half_A = side_A / 2.0
        return tile_lattice_in_region(self.lattice, half_A, side_A)

    def generate_volume(self, D, H, W):
        p = self.params
        a_px = float(p["a_px"])
        c_px = a_px * float(p["c_over_a"])
        V = np.zeros((D, H, W), dtype=np.float32) + float(p["base_level"])

        # In-plane (x,y) hexagonal lattice vectors in pixels
        v1 = np.array([a_px, 0.0])
        v2 = np.array([-0.5 * a_px, np.sqrt(3) / 2 * a_px])
        # Basis atoms: (frac along v1, frac along v2, z-fraction of c)
        basis_2d = [
            (0.0, 0.0, 0.0),
            (1.0/3.0, 2.0/3.0, 0.5),
        ]

        # Range of integer cell indices needed to tile the volume
        n_i = int(np.ceil(W / a_px)) + 3
        n_j = int(np.ceil(H / (np.sqrt(3) / 2 * a_px))) + 3
        n_layers = int(np.ceil(D / c_px)) + 3

        for layer in range(n_layers):
            z_base = layer * c_px
            for i in range(-2, n_i):
                for j in range(-2, n_j):
                    origin_xy = i * v1 + j * v2
                    for (f1, f2, fz) in basis_2d:
                        xy = origin_xy + f1 * v1 + f2 * v2
                        x = int(round(xy[0]))
                        y = int(round(xy[1]))
                        z = int(round(z_base + fz * c_px))
                        if 0 <= z < D and 0 <= y < H and 0 <= x < W:
                            V[z, y, x] += float(p["atom_intensity"])

        def gfreq(n, s):
            f = np.fft.fftfreq(n).astype(np.float32)
            return np.exp(-2.0 * (np.pi ** 2) * (s ** 2) * (f ** 2)).astype(np.float32)

        s = float(p["sigma_px"])
        gz, gy, gx = gfreq(D, s), gfreq(H, s), gfreq(W, s)
        F = np.fft.fftn(V)
        F *= gz[:, None, None]
        F *= gy[None, :, None]
        F *= gx[None, None, :]
        Vb = np.fft.ifftn(F).real.astype(np.float32)
        return np.clip(Vb, 0, 65535).astype(np.float32)


### New structural samples (polycrystal, dislocation, amorphous, shapes)

In [ ]:
%%writefile samples/polycrystal_grains.py
"""
samples/polycrystal_grains.py
Procedural FCC polycrystal with a small number (default 4) of contiguous grains,
each a Voronoi region with its own crystallographic orientation. Atoms are placed
in real space according to which grain owns each location, so the IMG view and the
diffraction pattern come from the SAME atomic model:
  - an aperture inside one grain  -> a clean single-crystal spot pattern
  - an aperture spanning a boundary -> two overlapping single-crystal patterns
  - a wide aperture over many grains -> ring-like (powder) tendency
No external file needed.
"""
import numpy as np
from .base import Sample, SampleMetadata, CrystalLattice, tile_lattice_in_region
from . import register


def _rand_rot(seed):
    r = np.random.default_rng(seed)
    ax = r.normal(size=3); ax /= (np.linalg.norm(ax) + 1e-12)
    ang = r.uniform(0, 2*np.pi)
    K = np.array([[0,-ax[2],ax[1]],[ax[2],0,-ax[0]],[-ax[1],ax[0],0]])
    return np.eye(3) + np.sin(ang)*K + (1-np.cos(ang))*(K@K)


@register
class PolycrystalGrains(Sample):
    feature_scale_nm = 0.25   # lattice fringe spacing (~0.25 nm)
    sample_fov_um = 2.0
    meta = SampleMetadata(
        name="polycrystal_grains",
        display_name="Polycrystal (FCC, few grains)",
        description="A few contiguous, differently-oriented FCC grains (Voronoi).",
        default_params={
            "n_grains": 4,
            "seed": 7,
            "a_angstrom": 4.05,
            "atomic_number": 79,
            "base_level": 90.0,
            "grain_intensity": 9000.0,
            "sigma_px": 1.1,
        },
        param_schema={
            "n_grains":        {"type": "int",   "min": 2,    "max": 12},
            "seed":            {"type": "int",   "min": 0,    "max": 2**31-1},
            "a_angstrom":      {"type": "float", "min": 1.0,  "max": 20.0},
            "atomic_number":   {"type": "int",   "min": 1,    "max": 100},
            "base_level":      {"type": "float", "min": 0,    "max": 1000},
            "grain_intensity": {"type": "float", "min": 100,  "max": 60000},
            "sigma_px":        {"type": "float", "min": 0.5,  "max": 4.0},
        },
    )
    crystalline_particles = True

    def __init__(self, **params):
        super().__init__(**params)
        a = float(self.params["a_angstrom"])
        Z = int(self.params["atomic_number"])
        self.lattice = CrystalLattice(
            real_vectors=np.array([[a,0,0],[0,a,0],[0,0,a]], dtype=np.float64),
            basis=[((0,0,0),Z),((0,0.5,0.5),Z),((0.5,0,0.5),Z),((0.5,0.5,0),Z)],
            name="FCC-poly")

    def _grain_setup(self, H, W):
        """Grain seed points (in voxels) and a deterministic orientation each.
        Orientations include a sizeable in-plane rotation so adjacent grains look
        visibly different in the IMG view (rotated lattice rows) AND give distinct
        diffraction patterns."""
        rng = np.random.default_rng(int(self.params["seed"]))
        ng = int(self.params["n_grains"])
        seeds_xy = np.column_stack([rng.uniform(0.1*W, 0.9*W, ng),
                                    rng.uniform(0.1*H, 0.9*H, ng)])
        rots = []
        # spread in-plane angles roughly evenly so grains are clearly different
        base_angles = np.linspace(0, np.pi/2, ng, endpoint=False) + rng.uniform(0, 0.3, ng)
        for g in range(ng):
            # in-plane rotation (about beam z) -- dominates the visual + pattern
            th = base_angles[g]
            Rz = np.array([[np.cos(th), -np.sin(th), 0],
                           [np.sin(th),  np.cos(th), 0],
                           [0, 0, 1.0]])
            # plus a small out-of-plane tilt (proper rotation, small angle)
            r2 = np.random.default_rng(int(self.params["seed"]) * 100 + g)
            ax = r2.normal(size=3); ax[2] = 0  # tilt axis in-plane -> tips z out
            ax /= (np.linalg.norm(ax) + 1e-12)
            phi = r2.uniform(0.0, 0.30)        # up to ~17 deg out-of-plane
            K = np.array([[0,-ax[2],ax[1]],[ax[2],0,-ax[0]],[-ax[1],ax[0],0]])
            Rtilt = np.eye(3) + np.sin(phi)*K + (1-np.cos(phi))*(K@K)
            rots.append(Rz @ Rtilt)
        return seeds_xy, rots

    def _owner_at(self, x_vox, y_vox):
        """Index of the grain owning a voxel location (nearest seed = Voronoi)."""
        d2 = (self._seeds_xy[:, 0] - x_vox)**2 + (self._seeds_xy[:, 1] - y_vox)**2
        return int(np.argmin(d2))

    def generate_volume(self, D, H, W):
        p = self.params
        self._seeds_xy, self._rots = self._grain_setup(H, W)
        self._vol_shape = (D, H, W)

        V = np.zeros((D, H, W), dtype=np.float32) + float(p["base_level"])
        # Render each grain's Voronoi region with a lattice-dot texture whose
        # in-plane orientation matches that grain's rotation (so the IMG view
        # visibly shows differently-oriented grains meeting at boundaries).
        a_px = 20
        gy, gx = np.mgrid[0:H, 0:W]
        # Voronoi ownership per pixel
        d2 = ((gx[..., None] - self._seeds_xy[None, None, :, 0])**2 +
              (gy[..., None] - self._seeds_xy[None, None, :, 1])**2)
        owner = np.argmin(d2, axis=2)
        self._owner_map = owner.astype(np.int16)

        for g in range(len(self._seeds_xy)):
            mask = (owner == g)
            if not mask.any():
                continue
            ang = np.arctan2(self._rots[g][1, 0], self._rots[g][0, 0])
            ca, sa = np.cos(ang), np.sin(ang)
            xr = gx * ca - gy * sa
            yr = gx * sa + gy * ca
            on = (((np.round(xr / a_px) - xr / a_px)**2 +
                   (np.round(yr / a_px) - yr / a_px)**2) < 0.02) & mask
            ys, xs = np.where(on)
            for zz in range(D//2 - 6, D//2 + 6):
                V[zz, ys, xs] += float(p["grain_intensity"])

        def gfreq(n, s):
            f = np.fft.fftfreq(n).astype(np.float32)
            return np.exp(-2.0*(np.pi**2)*(s**2)*(f**2)).astype(np.float32)
        s = float(p["sigma_px"])
        F = np.fft.fftn(V)
        F *= gfreq(D, s)[:,None,None]; F *= gfreq(H, s)[None,:,None]; F *= gfreq(W, s)[None,None,:]
        V = np.clip(np.fft.ifftn(F).real, 0, 65535).astype(np.float32)
        return V

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        """Place atoms in the aperture according to which grain owns each sub-cell.
        We sample the aperture on a fine sub-grid, assign each sub-cell to its
        Voronoi owner, and fill it with that grain's (rotated) FCC lattice. This
        makes a within-grain aperture give a single-crystal pattern and a
        boundary-spanning aperture give two overlapping patterns -- from the SAME
        model as the image. Atoms are kept UNDER the diffraction cap (no random
        subsampling, which would smear the lattice)."""
        if not hasattr(self, "_seeds_xy"):
            return None, None
        D, H, W = self._vol_shape
        px_per_um = W / self.sample_fov_um
        rc_x = W/2.0 + cx_um * px_per_um   # region center in voxels
        rc_y = H/2.0 + cy_um * px_per_um
        half_vox = half_width_um * px_per_um
        depth_A = depth_nm * 10.0

        # Physical aperture size in Angstrom for the atom fill (compressed so the
        # per-grain block stays well under the cap to avoid subsampling).
        target_total = 90000
        # how many distinct grains does the aperture overlap? sample a 3x3 probe
        probe = np.linspace(-half_vox, half_vox, 3)
        owners = set()
        for dx in probe:
            for dy in probe:
                xv = np.clip(rc_x + dx, 0, W-1); yv = np.clip(rc_y + dy, 0, H-1)
                owners.add(self._owner_at(xv, yv))
        owners = sorted(owners)
        n_present = max(1, len(owners))

        # Side of the cubic atom block per grain, sized so total ~ target_total.
        a1, a2, a3 = self.lattice.real_vectors
        density = len(self.lattice.basis) / abs(np.dot(a1, np.cross(a2, a3)))
        side_A = float((target_total / max(1, n_present) / max(1e-9, density)) ** (1.0/3.0))
        half_A = side_A / 2.0

        # Partition the in-plane aperture among the present grains by area fraction.
        # Simple approach: give each present grain a lateral sub-offset so their
        # atom blocks occupy DIFFERENT space (adjacent, not overlapping), then
        # rotate each block by its grain orientation.
        all_pos = []; all_Z = []
        n = len(owners)
        for i, g in enumerate(owners):
            bp, bZ = tile_lattice_in_region(self.lattice, half_A, min(depth_A, side_A))
            if len(bp) == 0:
                continue
            bp = bp @ self._rots[g].T          # this grain's orientation
            # lateral offset so grains tile side-by-side (no physical overlap)
            if n > 1:
                off = (i - (n-1)/2.0) * side_A
                bp = bp + np.array([off, 0.0, 0.0])
            all_pos.append(bp); all_Z.append(bZ)
        if not all_pos:
            return np.zeros((0,3)), np.zeros(0, dtype=np.int32)
        pos = np.concatenate(all_pos); Z = np.concatenate(all_Z)
        return pos.astype(np.float64), Z.astype(np.int32)


In [ ]:
%%writefile samples/dislocation_crystal.py
"""
samples/dislocation_crystal.py
FCC single crystal containing an edge dislocation. The dislocation displacement
field is applied to the atom positions, so diffraction shows the local lattice
distortion (streaking / spot broadening near the core) -- a defect signature.
"""
import numpy as np
from .base import Sample, SampleMetadata, CrystalLattice, tile_lattice_in_region
from . import register


@register
class DislocationCrystal(Sample):
    feature_scale_nm = 0.25   # lattice fringe spacing (~0.25 nm)
    sample_fov_um = 0.5
    meta = SampleMetadata(
        name="dislocation_crystal",
        display_name="FCC Crystal with Edge Dislocation",
        description="FCC single crystal with an edge dislocation displacement field.",
        default_params={
            "a_angstrom": 4.05,
            "atomic_number": 79,
            "burgers_A": 4.05,          # Burgers vector magnitude (~ a)
            "poisson_ratio": 0.34,
            "base_level": 90.0,
            "atom_intensity": 9000.0,
            "sigma_px": 1.1,
            "a_px": 24,
        },
        param_schema={
            "a_angstrom":     {"type": "float", "min": 1.0,  "max": 20.0},
            "atomic_number":  {"type": "int",   "min": 1,    "max": 100},
            "burgers_A":      {"type": "float", "min": 0.5,  "max": 10.0},
            "poisson_ratio":  {"type": "float", "min": 0.0,  "max": 0.5},
            "base_level":     {"type": "float", "min": 0,    "max": 1000},
            "atom_intensity": {"type": "float", "min": 100,  "max": 60000},
            "sigma_px":       {"type": "float", "min": 0.5,  "max": 4.0},
            "a_px":           {"type": "int",   "min": 8,    "max": 64},
        },
    )
    crystalline_particles = True

    def __init__(self, **params):
        super().__init__(**params)
        a = float(self.params["a_angstrom"])
        Z = int(self.params["atomic_number"])
        self.lattice = CrystalLattice(
            real_vectors=np.array([[a,0,0],[0,a,0],[0,0,a]], dtype=np.float64),
            basis=[((0,0,0),Z),((0,0.5,0.5),Z),((0.5,0,0.5),Z),((0.5,0.5,0),Z)],
            name="FCC-dislocation")

    def _apply_edge_dislocation(self, pos):
        """Apply the displacement field of an edge dislocation lying along z with
        Burgers vector b along x. Classic isotropic elasticity solution."""
        b = float(self.params["burgers_A"])
        nu = float(self.params["poisson_ratio"])
        x = pos[:,0].copy(); y = pos[:,1].copy()
        r2 = x*x + y*y + 1.0  # +1 to soften the core singularity
        # ux, uy from Hirth & Lothe (edge dislocation)
        ux = (b/(2*np.pi)) * (np.arctan2(y, x) + (x*y)/(2*(1-nu)*r2))
        uy = -(b/(2*np.pi)) * ((1-2*nu)/(4*(1-nu))*np.log(r2) + (x*x - y*y)/(4*(1-nu)*r2))
        out = pos.copy()
        out[:,0] += ux
        out[:,1] += uy
        return out

    def generate_volume(self, D, H, W):
        p = self.params
        self._vol_shape = (D, H, W)
        a_px = int(p["a_px"])
        V = np.zeros((D, H, W), dtype=np.float32) + float(p["base_level"])
        # Visualize: a dot lattice with an extra half-plane inserted above center
        cy, cx = H/2.0, W/2.0
        for iy in range(0, H, a_px):
            for ix in range(0, W, a_px):
                # insert extra half-plane: shift columns above the glide plane
                shift = 0.5 * a_px if (iy < cy) else 0.0
                xx = int(ix + shift) % W
                for zz in range(D//2-6, D//2+6):
                    V[zz, iy, xx] += float(p["atom_intensity"])
        def gfreq(n, s):
            f = np.fft.fftfreq(n).astype(np.float32)
            return np.exp(-2.0*(np.pi**2)*(s**2)*(f**2)).astype(np.float32)
        s = float(p["sigma_px"])
        F = np.fft.fftn(V)
        F *= gfreq(D,s)[:,None,None]; F *= gfreq(H,s)[None,:,None]; F *= gfreq(W,s)[None,None,:]
        V = np.clip(np.fft.ifftn(F).real, 0, 65535).astype(np.float32)
        return V

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        if not hasattr(self, "_vol_shape"):
            return None, None
        depth_A = depth_nm * 10.0
        half_A = max(30.0, half_width_um * 1e4 * 0.02)  # compress to ~100k atoms
        bp, bZ = tile_lattice_in_region(self.lattice, half_A, depth_A)
        if len(bp) == 0:
            return np.zeros((0,3)), np.zeros(0, dtype=np.int32)
        # apply the dislocation displacement field (core at region center)
        bp = self._apply_edge_dislocation(bp)
        if len(bp) > 100000:
            ii = np.random.default_rng(0).choice(len(bp), 100000, replace=False)
            bp = bp[ii]; bZ = bZ[ii]
        return bp.astype(np.float64), bZ.astype(np.int32)


In [ ]:
%%writefile samples/amorphous_film.py
"""
samples/amorphous_film.py
Amorphous film: atoms placed with short-range order but no long-range lattice
(random close packing with a minimum-distance constraint). Diffraction shows
broad diffuse halos (amorphous rings), not sharp spots -- the correct signature
of an amorphous material.
"""
import numpy as np
from .base import Sample, SampleMetadata
from . import register


@register
class AmorphousFilm(Sample):
    feature_scale_nm = 0.3   # nearest-neighbour distance (~0.3 nm)
    sample_fov_um = 0.5
    meta = SampleMetadata(
        name="amorphous_film",
        display_name="Amorphous Film",
        description="Amorphous (non-crystalline) film -> diffuse diffraction halos.",
        default_params={
            "seed": 11,
            "atomic_number": 14,      # Si-like (light amorphous former)
            "nn_distance_A": 2.5,     # typical nearest-neighbour distance (Angstrom)
            "base_level": 120.0,
            "atom_intensity": 1200.0,
            "film_intensity_sigma_px": 1.3,
        },
        param_schema={
            "seed":           {"type": "int",   "min": 0,    "max": 2**31-1},
            "atomic_number":  {"type": "int",   "min": 1,    "max": 100},
            "nn_distance_A":  {"type": "float", "min": 1.5,  "max": 5.0},
            "base_level":     {"type": "float", "min": 0,    "max": 1000},
            "atom_intensity": {"type": "float", "min": 100,  "max": 60000},
        },
    )
    crystalline_particles = False  # amorphous

    def generate_volume(self, D, H, W):
        p = self.params
        self._vol_shape = (D, H, W)
        rng = np.random.default_rng(int(p["seed"]))
        # Simple amorphous texture in the image: filtered noise blobs
        V = rng.normal(float(p["base_level"]), 30.0, size=(D, H, W)).astype(np.float32)
        # add a band of denser material in the central slab
        zc = D // 2
        for z in range(max(0, zc-8), min(D, zc+8)):
            V[z] += float(p["atom_intensity"]) * 0.4 * np.exp(-((z-zc)/5.0)**2)
        def gfreq(n, s):
            f = np.fft.fftfreq(n).astype(np.float32)
            return np.exp(-2.0*(np.pi**2)*(s**2)*(f**2)).astype(np.float32)
        s = float(p["film_intensity_sigma_px"])
        F = np.fft.fftn(V)
        F *= gfreq(D,s)[:,None,None]; F *= gfreq(H,s)[None,:,None]; F *= gfreq(W,s)[None,None,:]
        V = np.clip(np.fft.ifftn(F).real, 0, 65535).astype(np.float32)
        return V

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        """Generate amorphous atoms (random close packing) within the region.
        Position-independent statistics (amorphous is uniform), so stage motion
        doesn't change the halo -- which is physically correct for a uniform
        amorphous film."""
        p = self.params
        rng = np.random.default_rng(int(p["seed"]))
        depth_A = depth_nm * 10.0
        # Use a fixed-size box that yields ~target atoms at the given nn distance
        nn = float(p["nn_distance_A"])
        number_density = 0.7 / (nn**3)   # ~RCP packing fraction proxy
        target = 60000
        # box volume to hit target: V = target/density; box is square in-plane * depth
        box_xy_A = float(np.sqrt(target / max(1e-9, number_density * depth_A)))
        half_A = box_xy_A / 2.0
        # Poisson-disk-ish: oversample random points then thin by min distance via grid
        n_try = int(target * 2.5)
        pts = np.column_stack([
            rng.uniform(-half_A, half_A, n_try),
            rng.uniform(-half_A, half_A, n_try),
            rng.uniform(-depth_A/2, depth_A/2, n_try),
        ])
        # grid-based thinning to enforce a minimum separation ~ nn
        cell = nn
        keys = np.floor(pts / cell).astype(np.int64)
        # keep first point per occupied cell (gives roughly nn spacing)
        seen = {}
        keep = np.zeros(len(pts), dtype=bool)
        for i in range(len(pts)):
            k = (keys[i,0], keys[i,1], keys[i,2])
            if k not in seen:
                seen[k] = True
                keep[i] = True
            if keep.sum() >= target:
                break
        pos = pts[keep]
        Z = np.full(len(pos), int(p["atomic_number"]), dtype=np.int32)
        return pos.astype(np.float64), Z


In [ ]:
%%writefile samples/shape_assembly.py
"""
samples/shape_assembly.py
Assembly of random convex shapes (circles/ellipses, rectangles, hexagons) with
random rotations and aspect ratios, extruded through depth. Built on the user's
Shape Generator. Each shape is filled with crystalline Au atoms (so it diffracts)
OR amorphous atoms, selectable. Designed as a testbed for "find and characterize
isolated features" workflows.
"""
import numpy as np
from .base import Sample, SampleMetadata, atoms_in_particles
from . import register


def generate_shapes_array(height=600, width=800, num_shapes=200, min_size=8,
                          max_size=45, aspect_min=0.6, aspect_max=1.8, seed=None,
                          background_value=0.0, shape_intensity=1.0,
                          enable_rotation=True, non_overlapping=False,
                          shape_types=None, _return_placed=False):
    """Adapted from the user's Shape Generator. Returns the 2D array, and
    optionally the list of placed shapes (cy, cx, size, type, aspect, angle)."""
    if seed is not None:
        np.random.seed(seed)
    if shape_types is None:
        shape_types = ['circle', 'rect', 'hex']
    array = np.full((height, width), background_value, dtype=float)
    placed = []
    num_to_attempt = int(num_shapes * (2.0 if non_overlapping else 1.0)) + 100
    successful = 0
    for _ in range(num_to_attempt):
        if successful >= num_shapes:
            break
        size = np.random.randint(min_size, max_size + 1)
        aspect = np.random.uniform(aspect_min, aspect_max)
        shape_type = np.random.choice(shape_types)
        angle = np.random.uniform(0, 2*np.pi) if enable_rotation else 0.0
        cy = np.random.randint(0, height)
        cx = np.random.randint(0, width)
        if non_overlapping:
            overlaps = False
            for py, px, psize, _, _, _ in placed:
                if np.hypot(cy-py, cx-px) < (psize+size)*0.95:
                    overlaps = True; break
            if overlaps:
                continue
        y, x = np.ogrid[:height, :width]
        dy = y - cy; dx = x - cx
        ca, sa = np.cos(angle), np.sin(angle)
        dx_rot = dx*ca - dy*sa; dy_rot = dx*sa + dy*ca
        if shape_type == 'circle':
            mask = ((dx_rot/aspect)**2 + dy_rot**2) <= size**2
        elif shape_type == 'rect':
            mask = (np.abs(dx_rot) <= size*aspect) & (np.abs(dy_rot) <= size)
        else:  # hex
            dx_h = dx_rot/aspect; dy_h = dy_rot; s = size
            mask = ((np.abs(dx_h) <= s) & (np.abs(dy_h) <= s*0.866) &
                    (np.abs(dx_h*0.5 + dy_h*0.866) <= s) &
                    (np.abs(dx_h*0.5 - dy_h*0.866) <= s))
        array[mask] += shape_intensity
        placed.append((cy, cx, size, shape_type, aspect, angle))
        successful += 1
    if _return_placed:
        return array, placed
    return array


@register
class ShapeAssembly(Sample):
    feature_scale_nm = 30.0   # smallest shape feature (~30 nm)
    sample_fov_um = 5.0
    meta = SampleMetadata(
        name="shape_assembly",
        display_name="Shape Assembly (synthetic features)",
        description="Random rotated convex shapes extruded to 3D; testbed for feature-finding workflows.",
        default_params={
            "num_shapes": 40,
            "min_size": 8,
            "max_size": 28,
            "aspect_min": 0.6,
            "aspect_max": 1.8,
            "seed": 42,
            "non_overlapping": True,
            "crystalline": True,        # True -> shapes diffract as crystals; False -> amorphous
            "depth_fraction": 0.5,      # fraction of D each shape spans
            "base_level": 100.0,
            "shape_intensity": 4000.0,
            "sigma_px": 1.2,
        },
        param_schema={
            "num_shapes":     {"type": "int",   "min": 1,    "max": 400},
            "min_size":       {"type": "int",   "min": 2,    "max": 100},
            "max_size":       {"type": "int",   "min": 3,    "max": 200},
            "aspect_min":     {"type": "float", "min": 0.2,  "max": 2.0},
            "aspect_max":     {"type": "float", "min": 0.5,  "max": 5.0},
            "seed":           {"type": "int",   "min": 0,    "max": 2**31-1},
            "crystalline":    {"type": "int",   "min": 0,    "max": 1},
            "base_level":     {"type": "float", "min": 0,    "max": 1000},
            "shape_intensity":{"type": "float", "min": 100,  "max": 60000},
            "sigma_px":       {"type": "float", "min": 0.5,  "max": 4.0},
        },
    )

    @property
    def crystalline_particles(self):
        return bool(self.params.get("crystalline", True))

    particles_random_orientation = True

    def generate_volume(self, D, H, W):
        p = self.params
        self._vol_shape = (D, H, W)
        arr2d, placed = generate_shapes_array(
            height=H, width=W, num_shapes=int(p["num_shapes"]),
            min_size=int(p["min_size"]), max_size=int(p["max_size"]),
            aspect_min=float(p["aspect_min"]), aspect_max=float(p["aspect_max"]),
            seed=int(p["seed"]), shape_intensity=1.0, enable_rotation=True,
            non_overlapping=bool(p["non_overlapping"]),
            shape_types=['circle','rect','hex'], _return_placed=True)

        # Record each shape as a particle for the unified atom diffraction path.
        # Treat shape 'size' as the in-plane radius; z-extent = depth_fraction*D/2.
        rz = max(1, int(round(float(p["depth_fraction"]) * D / 2)))
        self._particles = []
        for (cy, cx, size, stype, aspect, angle) in placed:
            ry = int(round(size))
            rx = int(round(size * aspect))
            self._particles.append({"center_vox": (D//2, int(cy), int(cx)),
                                    "radii_vox": (rz, ry, rx)})

        # Build the 3D volume by extruding the 2D shape map through the central slab
        V = np.zeros((D, H, W), dtype=np.float32) + float(p["base_level"])
        zc = D // 2
        prof = np.zeros(D, dtype=np.float32)
        for z in range(D):
            prof[z] = np.exp(-((z - zc) / max(1, rz))**2)
        for z in range(D):
            if prof[z] > 0.02:
                V[z] += float(p["shape_intensity"]) * prof[z] * arr2d.astype(np.float32)

        def gfreq(n, s):
            f = np.fft.fftfreq(n).astype(np.float32)
            return np.exp(-2.0*(np.pi**2)*(s**2)*(f**2)).astype(np.float32)
        s = float(p["sigma_px"])
        F = np.fft.fftn(V)
        F *= gfreq(D,s)[:,None,None]; F *= gfreq(H,s)[None,:,None]; F *= gfreq(W,s)[None,None,:]
        V = np.clip(np.fft.ifftn(F).real, 0, 65535).astype(np.float32)
        return V


### Nanoparticle / structure samples (now expose atoms for unified diffraction)

In [ ]:
%%writefile samples/au_dispersed.py
"""
samples/au_dispersed.py
Dispersed Au nanoparticles — port of the original Au notebook generator.
Particles are placed uniformly at random across the volume.
"""
import numpy as np
from .base import Sample, SampleMetadata
from . import register


@register
class AuDispersedNanoparticles(Sample):
    feature_scale_nm = 2.0   # small nanoparticle diameter (~2 nm)
    meta = SampleMetadata(
        name="au_dispersed",
        display_name="Au Nanoparticles (Dispersed)",
        description="Gold nanoparticles distributed uniformly at random — the original Au sample.",
        default_params={
            "n_particles": 1200,
            "seed": 123,
            "base_level": 150.0,
            "base_noise": 6.0,
            # particle radius ranges (voxels)
            "rz_min": 1, "rz_max": 6,
            "ry_min": 3, "ry_max": 18,
            "rx_min": 3, "rx_max": 18,
            "intensity_min": 800.0,
            "intensity_max": 6000.0,
            "profile_sharpness": 2.2,
        },
        param_schema={
            "n_particles":        {"type": "int",   "min": 1,   "max": 50000},
            "seed":               {"type": "int",   "min": 0,   "max": 2**31-1},
            "base_level":         {"type": "float", "min": 0,   "max": 10000},
            "base_noise":         {"type": "float", "min": 0,   "max": 1000},
            "intensity_min":      {"type": "float", "min": 0,   "max": 60000},
            "intensity_max":      {"type": "float", "min": 0,   "max": 60000},
            "profile_sharpness":  {"type": "float", "min": 0.1, "max": 20.0},
        },
    )

    def generate_volume(self, D, H, W):
        p = self.params
        rng = np.random.default_rng(int(p["seed"]))
        V = rng.normal(loc=float(p["base_level"]),
                       scale=float(p["base_noise"]),
                       size=(D, H, W)).astype(np.float32)

        n = int(p["n_particles"])
        sharp = float(p["profile_sharpness"])
        self._particles = []   # record for atom-based diffraction
        self._vol_shape = (D, H, W)

        for _ in range(n):
            cz = int(rng.integers(0, D))
            cy = int(rng.integers(0, H))
            cx = int(rng.integers(0, W))

            rz = int(rng.integers(int(p["rz_min"]), int(p["rz_max"]) + 1))
            ry = int(rng.integers(int(p["ry_min"]), int(p["ry_max"]) + 1))
            rx = int(rng.integers(int(p["rx_min"]), int(p["rx_max"]) + 1))

            self._particles.append({"center_vox": (cz, cy, cx),
                                    "radii_vox": (rz, ry, rx)})

            intensity = float(rng.uniform(float(p["intensity_min"]),
                                          float(p["intensity_max"])))

            z0 = max(0, cz - rz*2); z1 = min(D, cz + rz*2 + 1)
            y0 = max(0, cy - ry*2); y1 = min(H, cy + ry*2 + 1)
            x0 = max(0, cx - rx*2); x1 = min(W, cx + rx*2 + 1)

            if z1 <= z0 or y1 <= y0 or x1 <= x0:
                continue

            zz, yy, xx = np.mgrid[z0:z1, y0:y1, x0:x1]
            dz = (zz - cz).astype(np.float32)
            dy = (yy - cy).astype(np.float32)
            dx = (xx - cx).astype(np.float32)

            d = ((dx*dx)/(rx*rx + 1e-6)
                 + (dy*dy)/(ry*ry + 1e-6)
                 + (dz*dz)/(rz*rz + 1e-6))
            profile = np.exp(-sharp * d).astype(np.float32)

            V[z0:z1, y0:y1, x0:x1] += intensity * profile

        return np.clip(V, 0, 65535).astype(np.float32)

    # Crystalline Au nanoparticles -> real spots/rings via the unified atom path.
    crystalline_particles = True
    particles_random_orientation = True


In [ ]:
%%writefile samples/au_clustered.py
"""
samples/au_clustered.py
Clustered Au nanoparticles — particles are grouped into N clusters scattered
across the volume, with a small fraction of isolated background particles for
realism. Useful for testing shape/morphology detection, cluster counting, and
dispersion-metric routines.
"""
import numpy as np
from .base import Sample, SampleMetadata
from . import register


@register
class AuClusteredNanoparticles(Sample):
    feature_scale_nm = 2.0   # small nanoparticle diameter (~2 nm)
    meta = SampleMetadata(
        name="au_clustered",
        display_name="Au Nanoparticles (Clustered)",
        description="Gold nanoparticles grouped into clusters, plus a few isolated particles.",
        default_params={
            "n_clusters": 25,                # number of cluster centers (in the XY plane)
            "particles_per_cluster_mean": 40,
            "particles_per_cluster_std": 12,
            "cluster_radius_px_mean": 60.0,  # spatial spread of a cluster (voxels)
            "cluster_radius_px_std": 20.0,
            "isolated_fraction": 0.05,       # fraction of total particles placed uniformly
            "seed": 7,
            "base_level": 150.0,
            "base_noise": 6.0,
            # particle radius ranges (voxels) — slightly tighter than dispersed
            "rz_min": 1, "rz_max": 5,
            "ry_min": 3, "ry_max": 14,
            "rx_min": 3, "rx_max": 14,
            "intensity_min": 800.0,
            "intensity_max": 6000.0,
            "profile_sharpness": 2.2,
        },
        param_schema={
            "n_clusters":                 {"type": "int",   "min": 1,    "max": 1000},
            "particles_per_cluster_mean": {"type": "int",   "min": 1,    "max": 2000},
            "particles_per_cluster_std":  {"type": "int",   "min": 0,    "max": 500},
            "cluster_radius_px_mean":     {"type": "float", "min": 1.0,  "max": 1000.0},
            "cluster_radius_px_std":      {"type": "float", "min": 0.0,  "max": 500.0},
            "isolated_fraction":          {"type": "float", "min": 0.0,  "max": 1.0},
            "seed":                       {"type": "int",   "min": 0,    "max": 2**31-1},
        },
    )

    crystalline_particles = True
    particles_random_orientation = True

    def generate_volume(self, D, H, W):
        self._particles = []; self._vol_shape = (D, H, W)
        p = self.params
        rng = np.random.default_rng(int(p["seed"]))
        V = rng.normal(loc=float(p["base_level"]),
                       scale=float(p["base_noise"]),
                       size=(D, H, W)).astype(np.float32)

        sharp = float(p["profile_sharpness"])

        # 1. Generate cluster centers (in XY, with z spanning the slab)
        n_clusters = int(p["n_clusters"])
        cluster_cx = rng.uniform(0, W, size=n_clusters)
        cluster_cy = rng.uniform(0, H, size=n_clusters)
        cluster_radii = np.maximum(
            1.0,
            rng.normal(float(p["cluster_radius_px_mean"]),
                       float(p["cluster_radius_px_std"]),
                       size=n_clusters),
        )

        # 2. For each cluster, generate a list of particle centers
        all_centers = []  # list of (cz, cy, cx)
        total_clustered = 0
        for k in range(n_clusters):
            n_in = max(
                1,
                int(rng.normal(float(p["particles_per_cluster_mean"]),
                               float(p["particles_per_cluster_std"]))),
            )
            # Gaussian spread around the cluster center in XY,
            # uniform across Z (thin slab anyway).
            dx = rng.normal(0.0, cluster_radii[k], size=n_in)
            dy = rng.normal(0.0, cluster_radii[k], size=n_in)
            cx = np.clip(cluster_cx[k] + dx, 0, W - 1)
            cy = np.clip(cluster_cy[k] + dy, 0, H - 1)
            cz = rng.integers(0, D, size=n_in)
            for i in range(n_in):
                all_centers.append((int(cz[i]), int(cy[i]), int(cx[i])))
            total_clustered += n_in

        # 3. Add isolated background particles
        iso_n = int(total_clustered * float(p["isolated_fraction"]) /
                    max(1e-6, (1.0 - float(p["isolated_fraction"]))))
        for _ in range(iso_n):
            all_centers.append((
                int(rng.integers(0, D)),
                int(rng.integers(0, H)),
                int(rng.integers(0, W)),
            ))

        # 4. Splat each particle as a 3D Gaussian ellipsoid
        for cz, cy, cx in all_centers:
            rz = int(rng.integers(int(p["rz_min"]), int(p["rz_max"]) + 1))
            ry = int(rng.integers(int(p["ry_min"]), int(p["ry_max"]) + 1))
            rx = int(rng.integers(int(p["rx_min"]), int(p["rx_max"]) + 1))
            self._particles.append({"center_vox": (cz, cy, cx), "radii_vox": (rz, ry, rx)})
            intensity = float(rng.uniform(float(p["intensity_min"]),
                                          float(p["intensity_max"])))

            z0 = max(0, cz - rz*2); z1 = min(D, cz + rz*2 + 1)
            y0 = max(0, cy - ry*2); y1 = min(H, cy + ry*2 + 1)
            x0 = max(0, cx - rx*2); x1 = min(W, cx + rx*2 + 1)
            if z1 <= z0 or y1 <= y0 or x1 <= x0:
                continue

            zz, yy, xx = np.mgrid[z0:z1, y0:y1, x0:x1]
            d = (((xx - cx).astype(np.float32)**2)/(rx*rx + 1e-6)
                 + ((yy - cy).astype(np.float32)**2)/(ry*ry + 1e-6)
                 + ((zz - cz).astype(np.float32)**2)/(rz*rz + 1e-6))
            profile = np.exp(-sharp * d).astype(np.float32)
            V[z0:z1, y0:y1, x0:x1] += intensity * profile

        return np.clip(V, 0, 65535).astype(np.float32)


In [ ]:
%%writefile samples/au_bimodal.py
"""
samples/au_bimodal.py
Bimodal Au nanoparticles — two size populations (small + large) dispersed
uniformly. Useful for ML-based segmentation / size-classification tests where
the model must distinguish two distinct particle classes.
"""
import numpy as np
from .base import Sample, SampleMetadata
from . import register


@register
class AuBimodalNanoparticles(Sample):
    feature_scale_nm = 1.5   # smallest particle mode (~1.5 nm)
    meta = SampleMetadata(
        name="au_bimodal",
        display_name="Au Nanoparticles (Bimodal Size)",
        description="Two distinct size populations of Au nanoparticles, dispersed uniformly.",
        default_params={
            "n_small": 900,
            "n_large": 150,
            "seed": 31,
            "base_level": 150.0,
            "base_noise": 6.0,
            # small population radius ranges (voxels)
            "small_rz_min": 1, "small_rz_max": 3,
            "small_ry_min": 2, "small_ry_max": 5,
            "small_rx_min": 2, "small_rx_max": 5,
            # large population radius ranges (voxels)
            "large_rz_min": 2, "large_rz_max": 6,
            "large_ry_min": 12, "large_ry_max": 26,
            "large_rx_min": 12, "large_rx_max": 26,
            # intensity ranges per population
            "small_intensity_min": 600.0, "small_intensity_max": 2500.0,
            "large_intensity_min": 2500.0, "large_intensity_max": 7000.0,
            "profile_sharpness": 2.2,
        },
        param_schema={
            "n_small":        {"type": "int", "min": 0, "max": 50000},
            "n_large":        {"type": "int", "min": 0, "max": 50000},
            "seed":           {"type": "int", "min": 0, "max": 2**31-1},
        },
    )

    def _splat(self, V, rng, n, r_ranges, i_range, sharp, D, H, W):
        rz_min, rz_max, ry_min, ry_max, rx_min, rx_max = r_ranges
        i_min, i_max = i_range
        for _ in range(int(n)):
            cz = int(rng.integers(0, D))
            cy = int(rng.integers(0, H))
            cx = int(rng.integers(0, W))
            rz = int(rng.integers(rz_min, rz_max + 1))
            ry = int(rng.integers(ry_min, ry_max + 1))
            rx = int(rng.integers(rx_min, rx_max + 1))
            self._particles.append({"center_vox": (cz, cy, cx), "radii_vox": (rz, ry, rx)})
            intensity = float(rng.uniform(i_min, i_max))

            z0 = max(0, cz - rz*2); z1 = min(D, cz + rz*2 + 1)
            y0 = max(0, cy - ry*2); y1 = min(H, cy + ry*2 + 1)
            x0 = max(0, cx - rx*2); x1 = min(W, cx + rx*2 + 1)
            if z1 <= z0 or y1 <= y0 or x1 <= x0:
                continue

            zz, yy, xx = np.mgrid[z0:z1, y0:y1, x0:x1]
            d = (((xx - cx).astype(np.float32)**2)/(rx*rx + 1e-6)
                 + ((yy - cy).astype(np.float32)**2)/(ry*ry + 1e-6)
                 + ((zz - cz).astype(np.float32)**2)/(rz*rz + 1e-6))
            profile = np.exp(-sharp * d).astype(np.float32)
            V[z0:z1, y0:y1, x0:x1] += intensity * profile

    crystalline_particles = True
    particles_random_orientation = True

    def generate_volume(self, D, H, W):
        self._particles = []; self._vol_shape = (D, H, W)
        p = self.params
        rng = np.random.default_rng(int(p["seed"]))
        V = rng.normal(loc=float(p["base_level"]),
                       scale=float(p["base_noise"]),
                       size=(D, H, W)).astype(np.float32)
        sharp = float(p["profile_sharpness"])

        # small population
        self._splat(
            V, rng, p["n_small"],
            (int(p["small_rz_min"]), int(p["small_rz_max"]),
             int(p["small_ry_min"]), int(p["small_ry_max"]),
             int(p["small_rx_min"]), int(p["small_rx_max"])),
            (float(p["small_intensity_min"]), float(p["small_intensity_max"])),
            sharp, D, H, W,
        )
        # large population
        self._splat(
            V, rng, p["n_large"],
            (int(p["large_rz_min"]), int(p["large_rz_max"]),
             int(p["large_ry_min"]), int(p["large_ry_max"]),
             int(p["large_rx_min"]), int(p["large_rx_max"])),
            (float(p["large_intensity_min"]), float(p["large_intensity_max"])),
            sharp, D, H, W,
        )

        return np.clip(V, 0, 65535).astype(np.float32)


In [ ]:
%%writefile samples/au_on_substrate.py
"""
samples/au_on_substrate.py
Supported-catalyst style sample: a uniform low-Z substrate layer occupying the
lower part of the slab, with Au nanoparticles sitting on top of it. Simulates
nanoparticles supported on a carbon / oxide film, as in heterogeneous catalysis.
"""
import numpy as np
from .base import Sample, SampleMetadata
from . import register


@register
class AuOnSubstrate(Sample):
    feature_scale_nm = 2.0   # small nanoparticle diameter (~2 nm)
    meta = SampleMetadata(
        name="au_on_substrate",
        display_name="Au on Substrate (Supported Catalyst)",
        description="Uniform low-Z substrate layer with Au nanoparticles resting on top.",
        default_params={
            "n_particles": 700,
            "seed": 17,
            "base_level": 60.0,
            "base_noise": 5.0,
            # substrate occupies z in [0, substrate_thickness_frac * D)
            "substrate_thickness_frac": 0.55,
            "substrate_intensity": 380.0,     # low-Z support contrast
            "substrate_texture": 40.0,         # random texture amplitude in substrate
            # particle radii (voxels)
            "rz_min": 2, "rz_max": 6,
            "ry_min": 4, "ry_max": 16,
            "rx_min": 4, "rx_max": 16,
            "intensity_min": 2000.0,
            "intensity_max": 7000.0,
            "profile_sharpness": 2.2,
            # how far above the substrate top particles can sit (voxels)
            "particle_z_jitter": 4,
        },
        param_schema={
            "n_particles":               {"type": "int",   "min": 0,   "max": 50000},
            "seed":                      {"type": "int",   "min": 0,   "max": 2**31-1},
            "substrate_thickness_frac":  {"type": "float", "min": 0.0, "max": 1.0},
            "substrate_intensity":       {"type": "float", "min": 0,   "max": 60000},
        },
    )

    crystalline_particles = True
    particles_random_orientation = True

    def generate_volume(self, D, H, W):
        self._particles = []; self._vol_shape = (D, H, W)
        p = self.params
        rng = np.random.default_rng(int(p["seed"]))
        V = rng.normal(loc=float(p["base_level"]),
                       scale=float(p["base_noise"]),
                       size=(D, H, W)).astype(np.float32)
        sharp = float(p["profile_sharpness"])

        # 1. Build the substrate slab in the lower portion of z
        sub_top = int(round(float(p["substrate_thickness_frac"]) * D))
        sub_top = max(1, min(D, sub_top))
        texture = rng.normal(0.0, float(p["substrate_texture"]),
                             size=(sub_top, H, W)).astype(np.float32)
        V[0:sub_top] += float(p["substrate_intensity"]) + texture

        # 2. Place particles resting on top of the substrate surface
        n = int(p["n_particles"])
        jitter = int(p["particle_z_jitter"])
        for _ in range(n):
            cy = int(rng.integers(0, H))
            cx = int(rng.integers(0, W))
            # particle center near the substrate top surface
            cz = int(np.clip(sub_top + rng.integers(-jitter, jitter + 1), 0, D - 1))

            rz = int(rng.integers(int(p["rz_min"]), int(p["rz_max"]) + 1))
            ry = int(rng.integers(int(p["ry_min"]), int(p["ry_max"]) + 1))
            rx = int(rng.integers(int(p["rx_min"]), int(p["rx_max"]) + 1))
            self._particles.append({"center_vox": (cz, cy, cx), "radii_vox": (rz, ry, rx)})
            intensity = float(rng.uniform(float(p["intensity_min"]),
                                          float(p["intensity_max"])))

            z0 = max(0, cz - rz*2); z1 = min(D, cz + rz*2 + 1)
            y0 = max(0, cy - ry*2); y1 = min(H, cy + ry*2 + 1)
            x0 = max(0, cx - rx*2); x1 = min(W, cx + rx*2 + 1)
            if z1 <= z0 or y1 <= y0 or x1 <= x0:
                continue

            zz, yy, xx = np.mgrid[z0:z1, y0:y1, x0:x1]
            d = (((xx - cx).astype(np.float32)**2)/(rx*rx + 1e-6)
                 + ((yy - cy).astype(np.float32)**2)/(ry*ry + 1e-6)
                 + ((zz - cz).astype(np.float32)**2)/(rz*rz + 1e-6))
            profile = np.exp(-sharp * d).astype(np.float32)
            V[z0:z1, y0:y1, x0:x1] += intensity * profile

        return np.clip(V, 0, 65535).astype(np.float32)


In [ ]:
%%writefile samples/core_shell.py
"""
samples/core_shell.py
Core-shell nanoparticles: each particle is two concentric ellipsoids — a bright
core (e.g. Au) surrounded by a lower-intensity shell (e.g. Ag or oxide). Useful
for testing Z-contrast analysis, radial intensity profiling, and segmentation
of multi-component particles.
"""
import numpy as np
from .base import Sample, SampleMetadata
from . import register


@register
class CoreShellNanoparticles(Sample):
    feature_scale_nm = 1.0   # shell thickness (~1 nm)
    meta = SampleMetadata(
        name="core_shell",
        display_name="Core-Shell Nanoparticles",
        description="Concentric bright-core / dim-shell particles for Z-contrast tests.",
        default_params={
            "n_particles": 250,
            "seed": 5,
            "base_level": 120.0,
            "base_noise": 5.0,
            # core radius range (voxels)
            "core_r_min": 4, "core_r_max": 10,
            # shell thickness added on top of core radius (voxels)
            "shell_thickness_min": 3, "shell_thickness_max": 8,
            # z-radius scaling relative to in-plane radius (thin-slab anisotropy)
            "z_scale": 0.5,
            "core_intensity_min": 4000.0,
            "core_intensity_max": 8000.0,
            "shell_intensity_min": 800.0,
            "shell_intensity_max": 2500.0,
            "core_sharpness": 3.0,
            "shell_sharpness": 2.0,
        },
        param_schema={
            "n_particles":          {"type": "int",   "min": 0,   "max": 20000},
            "seed":                 {"type": "int",   "min": 0,   "max": 2**31-1},
            "core_r_min":           {"type": "int",   "min": 1,   "max": 50},
            "core_r_max":           {"type": "int",   "min": 1,   "max": 50},
            "shell_thickness_min":  {"type": "int",   "min": 0,   "max": 50},
            "shell_thickness_max":  {"type": "int",   "min": 0,   "max": 50},
            "z_scale":              {"type": "float", "min": 0.05, "max": 2.0},
        },
    )

    crystalline_particles = True
    particles_random_orientation = True

    def generate_volume(self, D, H, W):
        self._particles = []; self._vol_shape = (D, H, W)
        p = self.params
        rng = np.random.default_rng(int(p["seed"]))
        V = rng.normal(loc=float(p["base_level"]),
                       scale=float(p["base_noise"]),
                       size=(D, H, W)).astype(np.float32)

        z_scale = float(p["z_scale"])
        core_sharp = float(p["core_sharpness"])
        shell_sharp = float(p["shell_sharpness"])

        for _ in range(int(p["n_particles"])):
            cz = int(rng.integers(0, D))
            cy = int(rng.integers(0, H))
            cx = int(rng.integers(0, W))

            core_r = int(rng.integers(int(p["core_r_min"]), int(p["core_r_max"]) + 1))
            shell_t = int(rng.integers(int(p["shell_thickness_min"]),
                                       int(p["shell_thickness_max"]) + 1))
            shell_r = core_r + shell_t

            core_rz = max(1, int(round(core_r * z_scale)))
            shell_rz = max(1, int(round(shell_r * z_scale)))
            self._particles.append({"center_vox": (cz, cy, cx),
                                    "radii_vox": (shell_rz, shell_r, shell_r)})

            core_I = float(rng.uniform(float(p["core_intensity_min"]),
                                       float(p["core_intensity_max"])))
            shell_I = float(rng.uniform(float(p["shell_intensity_min"]),
                                        float(p["shell_intensity_max"])))

            # bounding box uses the outer (shell) radius
            z0 = max(0, cz - shell_rz*2); z1 = min(D, cz + shell_rz*2 + 1)
            y0 = max(0, cy - shell_r*2);  y1 = min(H, cy + shell_r*2 + 1)
            x0 = max(0, cx - shell_r*2);  x1 = min(W, cx + shell_r*2 + 1)
            if z1 <= z0 or y1 <= y0 or x1 <= x0:
                continue

            zz, yy, xx = np.mgrid[z0:z1, y0:y1, x0:x1]
            dzf = (zz - cz).astype(np.float32)
            dyf = (yy - cy).astype(np.float32)
            dxf = (xx - cx).astype(np.float32)

            # shell ellipsoid (normalized distance, 1.0 at shell surface)
            d_shell = ((dxf*dxf)/(shell_r*shell_r + 1e-6)
                       + (dyf*dyf)/(shell_r*shell_r + 1e-6)
                       + (dzf*dzf)/(shell_rz*shell_rz + 1e-6))
            # core ellipsoid
            d_core = ((dxf*dxf)/(core_r*core_r + 1e-6)
                      + (dyf*dyf)/(core_r*core_r + 1e-6)
                      + (dzf*dzf)/(core_rz*core_rz + 1e-6))

            shell_profile = np.exp(-shell_sharp * d_shell).astype(np.float32)
            core_profile = np.exp(-core_sharp * d_core).astype(np.float32)

            # Make the shell *hollow*: suppress the shell contribution inside the
            # core radius so the two components are distinguishable. d_core < 1.0
            # means "inside the core ellipsoid".
            shell_only = shell_profile * np.clip(d_core, 0.0, 1.0)

            # shell forms a ring/annulus; core forms a bright center.
            # Together: bright center, dim surrounding shell, vacuum outside.
            V[z0:z1, y0:y1, x0:x1] += shell_I * shell_only
            V[z0:z1, y0:y1, x0:x1] += core_I * core_profile

        return np.clip(V, 0, 65535).astype(np.float32)


In [ ]:
%%writefile samples/atomsk_polycrystal.py
"""
samples/atomsk_polycrystal.py

Loads atomistic structure files (typically from Atomsk's `--polycrystal` mode)
and rasterizes them into the digital-twin's voxel volume.

Supported input formats (via ASE if installed; falls back to a built-in XYZ
reader otherwise):
  - XYZ                (.xyz)         — universal, no dependencies needed
  - LAMMPS data        (.lmp, .data)  — Atomsk's default polycrystal output
  - CFG                (.cfg)         — common Atomsk output
  - CIF                (.cif)         — crystallographic info file
  - VASP POSCAR        (POSCAR/.vasp) — VASP structure file
  - XSF                (.xsf)         — XCrySDen

Atomsk example to produce a usable file:
    atomsk --create fcc 4.05 Au seed.cfg
    atomsk --polycrystal seed.cfg poly.txt poly.lmp xyz cfg
                                              ^      ^
                                              also writes poly.xyz, poly.cfg

Default file path: 'sample_data/polycrystal.xyz' (override with file_path param).

A note on units:
  Atomsk works in Angstroms. The voxel grid is whatever physical scale you set
  via `sample_fov_um`. The `scale_factor` parameter controls how atomistic
  coordinates map to voxels; the default tries to fit the whole structure
  into the volume. Use `auto_fit=True` to ignore scale_factor and just stretch
  the bounding box to fill the volume.
"""
import os
import numpy as np
from .base import Sample, SampleMetadata
from . import register


# ---------------------------------------------------------------------------
# File reading: prefer ASE for breadth, fall back to a built-in XYZ reader.
# ---------------------------------------------------------------------------
def _read_xyz_fallback(path):
    """Minimal extended-XYZ reader. Returns (positions_A, symbols)."""
    with open(path, "r") as f:
        lines = [ln.rstrip("\n") for ln in f]
    if not lines:
        raise ValueError(f"Empty file: {path}")
    try:
        n_atoms = int(lines[0].strip())
    except ValueError:
        raise ValueError(
            f"Could not parse atom count from first line of {path!r} "
            "(not an XYZ file?)"
        )
    if len(lines) < n_atoms + 2:
        raise ValueError(
            f"File claims {n_atoms} atoms but only has {len(lines) - 2} data lines"
        )
    positions = np.zeros((n_atoms, 3), dtype=np.float32)
    symbols = []
    for i in range(n_atoms):
        parts = lines[2 + i].split()
        if len(parts) < 4:
            raise ValueError(
                f"Line {3+i} of {path!r} has fewer than 4 columns: {parts!r}"
            )
        symbols.append(parts[0])
        positions[i] = [float(parts[1]), float(parts[2]), float(parts[3])]
    return positions, symbols


def _read_structure_file(path):
    """
    Returns (positions_A: (N,3) float32, symbols: list[str]).
    Tries ASE first; falls back to XYZ-only if ASE isn't available.
    """
    ext = os.path.splitext(path)[1].lower()
    try:
        from ase.io import read as ase_read
    except ImportError:
        if ext != ".xyz":
            raise ImportError(
                f"Reading '{ext}' files requires the `ase` package. "
                "Install with: pip install ase. "
                "Alternatively, convert your file to .xyz format (e.g. with "
                "Atomsk: atomsk input.lmp output.xyz)."
            )
        positions, symbols = _read_xyz_fallback(path)
        return positions, symbols

    # ASE format hints for files Atomsk commonly emits
    fmt_hint = None
    if ext in (".lmp", ".data"):
        fmt_hint = "lammps-data"
    elif ext == ".cfg":
        fmt_hint = "cfg"
    elif ext == ".cif":
        fmt_hint = "cif"
    elif ext == ".xsf":
        fmt_hint = "xsf"

    try:
        if fmt_hint:
            atoms = ase_read(path, format=fmt_hint)
        else:
            atoms = ase_read(path)
    except Exception as e:
        raise ValueError(
            f"ASE could not parse {path!r}: {e}. "
            "If it's an Atomsk file, try converting to XYZ first: "
            "`atomsk infile outfile.xyz`."
        )

    positions = np.asarray(atoms.get_positions(), dtype=np.float32)
    symbols = list(atoms.get_chemical_symbols())
    return positions, symbols


# ---------------------------------------------------------------------------
# Atomic-number proxy → Z-contrast intensity. Real STEM HAADF roughly ∝ Z^1.7,
# but we use a small lookup so unknown elements still get a sane value.
# ---------------------------------------------------------------------------
_Z_TABLE = {
    "H": 1,  "C": 6,  "N": 7,  "O": 8,
    "Na": 11, "Mg": 12, "Al": 13, "Si": 14,
    "Ti": 22, "V": 23, "Cr": 24, "Mn": 25, "Fe": 26, "Co": 27, "Ni": 28,
    "Cu": 29, "Zn": 30,
    "Mo": 42, "Ag": 47, "W": 74, "Pt": 78, "Au": 79, "Pb": 82,
}


def _intensity_from_symbol(sym):
    z = _Z_TABLE.get(sym, 14)  # default ~ Si
    return float(z) ** 1.7


# ---------------------------------------------------------------------------
@register
class AtomskPolycrystal(Sample):
    meta = SampleMetadata(
        name="atomsk_polycrystal",
        display_name="Atomsk Polycrystal (file-driven)",
        description=("Loads an atomistic structure file (xyz, lmp, cfg, cif, xsf) "
                     "and rasterizes it. Designed for Atomsk --polycrystal output."),
        default_params={
            # Path to the structure file. Relative paths are resolved against cwd.
            "file_path": "sample_data/polycrystal.xyz",
            # If True, auto-scale the structure's bounding box to fill the volume
            # in XY. Z gets centered with no stretching beyond what fits.
            "auto_fit": True,
            # Manual scaling: voxels per Angstrom (only used if auto_fit=False).
            "scale_factor": 0.1,
            # Gaussian sigma used to splat each atom (voxels). Larger = blurrier.
            "atom_sigma_px": 0.9,
            # Cap how many atoms to render (random subsample if exceeded).
            # Keeps generation time bounded for huge polycrystals.
            "max_atoms": 200000,
            "seed": 42,
            "base_level": 60.0,
            "intensity_scale": 30.0,   # multiplies the Z^1.7 contribution
        },
        param_schema={
            "file_path":        {"type": "str"},
            "auto_fit":         {"type": "bool"},
            "scale_factor":     {"type": "float", "min": 1e-4, "max": 100.0},
            "atom_sigma_px":    {"type": "float", "min": 0.2,  "max": 8.0},
            "max_atoms":        {"type": "int",   "min": 100,  "max": 5_000_000},
            "seed":             {"type": "int",   "min": 0,    "max": 2**31-1},
            "base_level":       {"type": "float", "min": 0,    "max": 10000},
            "intensity_scale":  {"type": "float", "min": 0,    "max": 10000},
        },
    )

    # Polycrystals are atomistic — much smaller real extent than µm samples.
    # Override the physical mapping so a single Atomsk structure spans the
    # active FOV nicely. Users can still rescale via load_sample(...) params.
    sample_fov_um = 50.0
    tilt_strength_px_per_slice = 0.35

    def generate_volume(self, D, H, W):
        p = self.params
        path = str(p["file_path"])
        if not os.path.isfile(path):
            raise FileNotFoundError(
                f"Atomsk file not found: {path!r}. "
                "Upload it to the Colab session or set `file_path` to its actual location. "
                "See the notebook's 'How to provide a polycrystal file' section."
            )

        positions, symbols = _read_structure_file(path)
        n_total = len(positions)
        if n_total == 0:
            raise ValueError(f"File {path!r} contained zero atoms.")

        # Subsample if too big
        rng = np.random.default_rng(int(p["seed"]))
        max_n = int(p["max_atoms"])
        if n_total > max_n:
            idx = rng.choice(n_total, size=max_n, replace=False)
            positions = positions[idx]
            symbols = [symbols[i] for i in idx]
            n_used = max_n
        else:
            n_used = n_total

        # Compute element intensities
        intensities = np.array([_intensity_from_symbol(s) for s in symbols],
                               dtype=np.float32)
        intensities *= float(p["intensity_scale"])

        # ---- Map atom coordinates (Angstroms) → voxel coordinates ----
        pmin = positions.min(axis=0)
        pmax = positions.max(axis=0)
        extent_A = np.maximum(pmax - pmin, 1e-6)

        if bool(p["auto_fit"]):
            # Stretch XY to fill (with a small margin), preserve XY aspect ratio,
            # center Z (don't stretch Z beyond available slices).
            margin = 0.92
            sxy = min((W * margin) / extent_A[0], (H * margin) / extent_A[1])
            sz = min(sxy, (D * margin) / max(extent_A[2], 1e-6))
            scales = np.array([sxy, sxy, sz], dtype=np.float32)
        else:
            s = float(p["scale_factor"])
            scales = np.array([s, s, s], dtype=np.float32)

        # Center the mapped structure inside the volume
        centered = (positions - (pmin + pmax) * 0.5) * scales
        # voxel coords: (x, y, z) -> indices in (z, y, x)
        xv = centered[:, 0] + 0.5 * W
        yv = centered[:, 1] + 0.5 * H
        zv = centered[:, 2] + 0.5 * D

        # ---- Splat each atom as a small 3D Gaussian ----
        V = np.full((D, H, W), float(p["base_level"]), dtype=np.float32)
        sigma = float(p["atom_sigma_px"])
        rad = max(1, int(np.ceil(3.0 * sigma)))  # truncate Gaussian at 3σ
        two_sig_sq = 2.0 * sigma * sigma

        # Precompute the 1D Gaussian kernel for axis-separable splatting
        offsets = np.arange(-rad, rad + 1, dtype=np.float32)
        gauss_1d = np.exp(-(offsets * offsets) / two_sig_sq).astype(np.float32)
        # Outer-product 3D kernel (small, since rad is small)
        k3d = (gauss_1d[:, None, None]
               * gauss_1d[None, :, None]
               * gauss_1d[None, None, :])

        # Iterate (works fine up to a few hundred thousand atoms)
        skipped = 0
        for i in range(n_used):
            ix = int(round(xv[i])); iy = int(round(yv[i])); iz = int(round(zv[i]))
            if not (0 <= ix < W and 0 <= iy < H and 0 <= iz < D):
                skipped += 1
                continue

            z0 = max(0, iz - rad); z1 = min(D, iz + rad + 1)
            y0 = max(0, iy - rad); y1 = min(H, iy + rad + 1)
            x0 = max(0, ix - rad); x1 = min(W, ix + rad + 1)

            kz0 = z0 - (iz - rad); kz1 = k3d.shape[0] - ((iz + rad + 1) - z1)
            ky0 = y0 - (iy - rad); ky1 = k3d.shape[1] - ((iy + rad + 1) - y1)
            kx0 = x0 - (ix - rad); kx1 = k3d.shape[2] - ((ix + rad + 1) - x1)

            V[z0:z1, y0:y1, x0:x1] += (
                intensities[i] * k3d[kz0:kz1, ky0:ky1, kx0:kx1]
            )

        if skipped > 0:
            print(f"[atomsk] {skipped}/{n_used} atoms fell outside the volume "
                  "(consider lowering scale_factor or enabling auto_fit).")

        # Store atom positions in microns relative to the centered sample, plus
        # atomic numbers, for local-region diffraction. centered[] is in
        # Angstroms; convert to microns for stage-frame coordinate matching.
        # Map symbol -> atomic number for diffraction form factors.
        _Z_TABLE = {"H":1,"He":2,"Li":3,"Be":4,"B":5,"C":6,"N":7,"O":8,"F":9,"Ne":10,
                    "Na":11,"Mg":12,"Al":13,"Si":14,"P":15,"S":16,"Cl":17,"Ar":18,
                    "K":19,"Ca":20,"Ti":22,"V":23,"Cr":24,"Mn":25,"Fe":26,"Co":27,
                    "Ni":28,"Cu":29,"Zn":30,"Ga":31,"Ge":32,"As":33,"Se":34,
                    "Mo":42,"Pd":46,"Ag":47,"Cd":48,"In":49,"Sn":50,"Sb":51,
                    "W":74,"Pt":78,"Au":79,"Pb":82,"U":92}
        Zs = np.array([_Z_TABLE.get(s, 6) for s in symbols], dtype=np.int32)
        # Centered positions are still in Angstroms (before voxel scaling). Use
        # those directly so diffraction sees real interatomic spacings.
        self._atoms_A = (positions - (pmin + pmax) * 0.5).astype(np.float64)
        self._atoms_Z = Zs
        # Sample extent (sample_fov_um) for stage-frame conversion
        # 1 voxel = (extent / volume_size); the sample occupies the volume center
        self._extent_A = extent_A
        self._n_atoms = n_used
        print(f"[atomsk] retained {n_used} atom positions for local diffraction")

        return np.clip(V, 0, 65535).astype(np.float32)

    def get_atoms_in_region(self, cx_um, cy_um, half_width_um, depth_nm):
        """Return atoms inside an aperture-sized region centered at the stage
        position. Atoms are returned in Angstroms relative to the region center."""
        if not hasattr(self, "_atoms_A") or self._atoms_A is None:
            return None, None
        # Convert stage-frame microns to atom-frame Angstroms
        cx_A = cx_um * 1e4
        cy_A = cy_um * 1e4
        half_A = half_width_um * 1e4
        depth_A = depth_nm * 10.0
        # The atom positions are centered on the sample, so the box in atom
        # coords is centered at (cx_A, cy_A, 0) with half-widths (half_A, half_A, depth_A/2)
        pos = self._atoms_A
        mask = ((np.abs(pos[:, 0] - cx_A) <= half_A) &
                (np.abs(pos[:, 1] - cy_A) <= half_A) &
                (np.abs(pos[:, 2]) <= depth_A / 2))
        kept = pos[mask].copy()
        kept[:, 0] -= cx_A
        kept[:, 1] -= cy_A
        # If too many, subsample
        if len(kept) > 100000:
            rng = np.random.default_rng(0)
            idx = rng.choice(len(kept), 100000, replace=False)
            kept = kept[idx]
            Zs = self._atoms_Z[mask][idx]
        else:
            Zs = self._atoms_Z[mask]
        return kept, Zs


Verify the registry sees all thirteen samples (forces a fresh rescan for Colab):

In [ ]:
import importlib, samples
importlib.reload(samples)
samples.discover(force_reload=True)
for s in samples.list_samples():
    print(f"  {s['name']}")


## 2. Write the digital-twin server (unified diffraction + specimen realism + environments)

In [ ]:
%%writefile stem_server_twisted_colab.py
from twisted.internet import reactor, protocol, threads
from twisted.internet.protocol import Factory
import json, time, base64, traceback
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, Any, List, Tuple

import samples

# Magnification <-> field-of-view calibration.  mag = MAG_K / fov_metres
# (equivalently fov_metres = MAG_K / mag).  MAG_K was calibrated from a real
# instrument: 57 kx corresponds to a 1.6564523008 um field of view, i.e.
# MAG_K = 57000 * 1.6564523008e-6 m = 0.0944177811456 (SI). We store MAG_K in SI
# (metres) and convert to microns at the boundary.
MAG_K = 57000.0 * 1.6564523008e-6   # = 0.0944177811456 (mag * metres)

def mag_to_fov_um(mag):
    """Magnification -> field of view in microns."""
    return (MAG_K / float(mag)) * 1e6

def fov_um_to_mag(fov_um):
    """Field of view in microns -> magnification."""
    return MAG_K / (float(fov_um) * 1e-6)


# ============================================================================
# PHYSICS (items 1-5): dose noise, PSF, drift, thickness law, kinematical diff
# ============================================================================
def make_psf(defocus_nm, cs_mm=1.0, aperture_probe_px=1.4, kv=200.0, pixel_nm=1.0,
             max_radius=24):
    sigma0 = float(aperture_probe_px)
    cs_term = 0.15 * cs_mm * (200.0 / max(50.0, kv))
    defocus_px = abs(defocus_nm) / max(1e-3, pixel_nm)
    sigma = np.sqrt(sigma0**2 + (0.18 * defocus_px)**2 + cs_term**2)
    r = int(min(max_radius, max(2, np.ceil(3 * sigma))))
    y, x = np.mgrid[-r:r+1, -r:r+1].astype(np.float32)
    rr = np.sqrt(x*x + y*y)
    psf = np.exp(-(rr*rr) / (2 * sigma * sigma)).astype(np.float32)
    if defocus_px > 6.0:
        ring_r = 0.6 * defocus_px
        ring_w = max(1.0, 0.15 * defocus_px)
        ring = np.exp(-((rr - ring_r)**2) / (2 * ring_w * ring_w)).astype(np.float32)
        psf = psf + 0.12 * ring
    s = psf.sum()
    if s > 0:
        psf /= s
    return psf


def convolve2d_fft(img, psf):
    H, W = img.shape
    kh, kw = psf.shape
    pad = max(kh, kw)
    ap = np.pad(img, ((pad, pad), (pad, pad)), mode="edge").astype(np.float32)
    fh, fw = ap.shape
    F = np.fft.rfft2(ap)
    K = np.fft.rfft2(psf, s=(fh, fw))
    out = np.fft.irfft2(F * K, s=(fh, fw))
    out = np.roll(out, (-(kh // 2), -(kw // 2)), axis=(0, 1))
    return out[pad:pad+H, pad:pad+W].astype(np.float32)


def apply_dose_noise(signal_norm, dose_e_per_px, dqe=0.8, readout_e=1.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    lam = np.clip(signal_norm, 0, None) * float(dose_e_per_px) * float(dqe)
    counts = rng.poisson(lam).astype(np.float32)
    if readout_e > 0:
        counts = counts + rng.normal(0.0, readout_e, counts.shape).astype(np.float32)
    return np.clip(counts, 0, None)


def apply_scan_distortion(img, drift_px_xy=(0.0, 0.0), line_jitter_px=0.0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    H, W = img.shape
    dx_total, dy_total = drift_px_xy
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    frac = (yy / max(1.0, H - 1.0))
    src_x = xx - dx_total * frac
    src_y = yy - dy_total * frac
    if line_jitter_px > 0:
        jit = rng.normal(0.0, line_jitter_px, size=H).astype(np.float32)
        src_x = src_x + jit[:, None]
    x0 = np.floor(src_x).astype(np.int32); y0 = np.floor(src_y).astype(np.int32)
    x1 = x0 + 1; y1 = y0 + 1
    x0 = np.clip(x0, 0, W-1); x1 = np.clip(x1, 0, W-1)
    y0 = np.clip(y0, 0, H-1); y1 = np.clip(y1, 0, H-1)
    Ia = img[y0, x0]; Ib = img[y1, x0]; Ic = img[y0, x1]; Id = img[y1, x1]
    wa = (x1 - src_x)*(y1 - src_y); wb = (x1 - src_x)*(src_y - y0)
    wc = (src_x - x0)*(y1 - src_y); wd = (src_x - x0)*(src_y - y0)
    return (Ia*wa + Ib*wb + Ic*wc + Id*wd).astype(np.float32)


def thickness_contrast(projected_sum, mfp_scale):
    return 1.0 - np.exp(-np.clip(projected_sum, 0, None) / max(1e-6, mfp_scale))


def kinematical_diffraction(lattice, out_size, tilt_a_deg, tilt_b_deg,
                            kv=200.0, camera_length_mm=800.0,
                            beamstop_radius_px=6.0, hkl_max=5,
                            thickness_nm=20.0, sigma_override=None,
                            spot_sigma_px=2.5, rng=None):
    """
    Kinematical diffraction with a thickness-driven relrod (v6 physics).

    The reflection intensity is structure_factor^2 * form_factor(g) * relrod(s),
    where the relrod is a sinc^2 in the excitation error s whose width is set by
    sample thickness (thin foil -> long relrod -> spots survive more tilt). This
    makes zone-axis patterns sharp and correct (square for cubic [001]), keeps the
    low-order net clean via the form-factor envelope, and lets tilting actually
    move between zone axes (the in-plane reflections fade as they leave the Bragg
    condition). `sigma_override` (1/A) bypasses the thickness->width mapping.
    """
    if rng is None:
        rng = np.random.default_rng()
    V = kv * 1e3
    lam = 12.2639 / np.sqrt(V * (1.0 + 0.97845e-6 * V))   # Angstrom
    k_mag = 2 * np.pi / lam                                # 1/A (2pi convention)
    recip = lattice.reciprocal_vectors()

    a = np.deg2rad(tilt_a_deg); b = np.deg2rad(tilt_b_deg)
    Rx = np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry = np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    R = Ry @ Rx
    beam = R @ np.array([0.0, 0.0, 1.0])
    k0 = k_mag * beam

    # Relrod half-width (1/A): thin foil -> long relrod -> more tolerant of tilt.
    t_A = max(10.0, thickness_nm * 10.0)
    relrod_hw = 6.0 / t_A
    if sigma_override is not None:
        relrod_hw = float(sigma_override)

    raw = []
    for h in range(-hkl_max, hkl_max + 1):
        for k in range(-hkl_max, hkl_max + 1):
            for l in range(-hkl_max, hkl_max + 1):
                if h == 0 and k == 0 and l == 0:
                    continue
                G = h * recip[0] + k * recip[1] + l * recip[2]
                Gmag = np.linalg.norm(G)
                if Gmag < 1e-6:
                    continue
                F = lattice.structure_factor(h, k, l)
                I_F = (abs(F)) ** 2
                if I_F < 1e-3:
                    continue
                # excitation error (deviation from Ewald sphere)
                s = -(2 * np.dot(k0, G) + Gmag**2) / (2 * k_mag)
                # relrod: sinc^2 with thickness-set width
                relrod = (np.sinc(s / relrod_hw)) ** 2
                if relrod < 1e-3:
                    continue
                # atomic form-factor falloff with scattering angle (suppresses
                # high orders so the clean low-order net shows)
                g = Gmag / (2 * np.pi)
                form = np.exp(-1.5 * g * g)
                G_perp = G - np.dot(G, beam) * beam
                raw.append((G_perp[0], G_perp[1], I_F * form * relrod, Gmag))

    spots = []
    if raw:
        gperp = [np.hypot(r[0], r[1]) for r in raw if np.hypot(r[0], r[1]) > 1e-6]
        gmin = min(gperp) if gperp else 1.0
        base_radius = 0.18 * out_size
        cl_zoom = camera_length_mm / 800.0
        scale = (base_radius / gmin) * cl_zoom
        for gx, gy, intensity, _ in raw:
            spots.append((gx * scale, gy * scale, intensity))

    img = np.zeros((out_size, out_size), dtype=np.float32)
    cx = cy = out_size / 2.0
    yy, xx = np.mgrid[0:out_size, 0:out_size].astype(np.float32)
    if spots:
        max_I = max(s[2] for s in spots)
        for det_x, det_y, intensity in spots:
            px = cx + det_x; py = cy + det_y
            if not (0 <= px < out_size and 0 <= py < out_size):
                continue
            d2 = (xx - px)**2 + (yy - py)**2
            img += (intensity / max_I) * np.exp(-d2 / (2 * spot_sigma_px**2))
    d2c = (xx - cx)**2 + (yy - cy)**2
    img += 1.2 * np.exp(-d2c / (2 * (spot_sigma_px*1.3)**2))
    if beamstop_radius_px and beamstop_radius_px > 0:
        img[np.sqrt(d2c) <= beamstop_radius_px] = 0.0
    img = img - img.min()
    mx = img.max()
    if mx > 1e-6:
        img = img / mx
    return np.clip(img * 65535.0, 0, 65535).astype(np.float32)


# ============================================================================
# Local-region diffraction from atomic positions (item 6)
# ============================================================================
def diffraction_from_atoms(positions_A, atomic_numbers, out_size,
                           tilt_a_deg, tilt_b_deg,
                           kv=200.0, q_max_invA=4.0,
                           thickness_nm=20.0, camera_length_mm=800.0,
                           beamstop_radius_px=6.0, atom_cap=100000, rng=None):
    """
    Diffraction pattern computed directly from atomic positions. The pattern
    is |sum_j f_j(q) exp(2pi i q.r_j)|^2 evaluated on the Ewald sphere across
    a 2D detector grid. Locality lives upstream -- the caller passes only the
    atoms in the diffracting region, so stage motion changes the pattern.
    """
    if rng is None:
        rng = np.random.default_rng()
    N = positions_A.shape[0]
    if N == 0:
        # No atoms in region: return empty pattern
        return np.zeros((out_size, out_size), dtype=np.float32)
    if N > atom_cap:
        idx = rng.choice(N, atom_cap, replace=False)
        positions_A = positions_A[idx]
        atomic_numbers = atomic_numbers[idx]
        N = atom_cap

    V_acc = kv * 1e3
    lam = 12.2639 / np.sqrt(V_acc * (1.0 + 0.97845e-6 * V_acc))
    k_mag = 2 * np.pi / lam

    a = np.deg2rad(tilt_a_deg); b = np.deg2rad(tilt_b_deg)
    Rx = np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry = np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    R = Ry @ Rx
    beam = R @ np.array([0.0, 0.0, 1.0])
    tmp = np.array([1.0, 0, 0]) if abs(beam[0]) < 0.9 else np.array([0, 1.0, 0])
    e_x = np.cross(tmp, beam); e_x /= np.linalg.norm(e_x)
    e_y = np.cross(beam, e_x)

    # Compute q-grid resolution. Note the diffracting region is kept ~cubic by
    # the sample's get_atoms_in_region (isotropic shape transform); with that,
    # 64 samples resolve the reciprocal lattice cleanly without aliasing. A
    # non-cubic (column) region was what previously produced an off-axis cloud.
    grid = min(out_size, 64)
    qs = np.linspace(-q_max_invA, q_max_invA, grid, dtype=np.float32)
    qx, qy = np.meshgrid(qs, qs, indexing='xy')
    q_perp_sq = qx**2 + qy**2
    q_z = -q_perp_sq / (2 * k_mag)
    Nq = grid * grid
    q_3d = (qx[:, :, None].astype(np.float32) * e_x[None, None, :].astype(np.float32) +
            qy[:, :, None].astype(np.float32) * e_y[None, None, :].astype(np.float32) +
            q_z[:, :, None].astype(np.float32) * beam[None, None, :].astype(np.float32)
            ).reshape(Nq, 3)
    positions_f32 = positions_A.astype(np.float32)
    chunk = max(64, min(Nq, int(2.5e7 / max(1, N))))
    A = np.zeros(Nq, dtype=np.complex64)
    q_mag = np.sqrt(q_perp_sq).reshape(Nq).astype(np.float32)
    unique_Z = np.unique(atomic_numbers)
    for start in range(0, Nq, chunk):
        end = min(Nq, start + chunk)
        qch = q_3d[start:end]
        for Z in unique_Z:
            pos_Z = positions_f32[atomic_numbers == Z]
            if len(pos_Z) == 0:
                continue
            phases = (2 * np.pi) * (qch @ pos_Z.T)
            amp = (np.cos(phases).sum(axis=1).astype(np.float32) +
                   1j * np.sin(phases).sum(axis=1).astype(np.float32))
            f = (Z * np.exp(-0.5 * q_mag[start:end] ** 2)).astype(np.float32)
            A[start:end] += f * amp.astype(np.complex64)

    I = (A.real**2 + A.imag**2).reshape(grid, grid)
    # Coherence damping from thickness (gentle high-q falloff)
    coh = np.exp(-q_perp_sq * (thickness_nm / 200.0))
    I = I * coh

    # Camera-length zoom: crop a centered region then upsample
    cl_zoom = camera_length_mm / 800.0
    if cl_zoom != 1.0:
        crop = max(8, int(grid / cl_zoom))
        if crop < grid:
            s0 = (grid - crop) // 2
            I = I[s0:s0+crop, s0:s0+crop]
            grid_eff = crop
        else:
            grid_eff = grid
    else:
        grid_eff = grid

    # Upsample to out_size
    if grid_eff != out_size:
        try:
            from scipy.ndimage import zoom
            I = zoom(I, out_size / grid_eff, order=1)
        except ImportError:
            rep = out_size / grid_eff
            yi = (np.arange(out_size) / rep).astype(int).clip(0, grid_eff - 1)
            xi = (np.arange(out_size) / rep).astype(int).clip(0, grid_eff - 1)
            I = I[yi][:, xi]
    # Beamstop
    yy, xx = np.mgrid[0:out_size, 0:out_size].astype(np.float32)
    cx = cy = out_size / 2.0
    rr = np.sqrt((yy-cy)**2 + (xx-cx)**2)
    if beamstop_radius_px > 0:
        I[rr <= beamstop_radius_px] = 0.0

    I = I - I.min()
    mx = float(I.max())
    if mx > 1e-6:
        I = I / mx
    return np.clip(I * 65535.0, 0, 65535).astype(np.float32)



# ============================================================================
# Simulated microscope state
# ============================================================================
@dataclass
class SimMicroscope:
    stage: Dict[str, float] = field(default_factory=lambda: {"x":0.0,"y":0.0,"z":0.0,"a":0.0,"b":0.0})
    beam: Dict[str, float]  = field(default_factory=lambda: {"x":0.0,"y":0.0,"current_pA":50.0,"voltage_kV":200.0})
    vacuum: float = 1e-6
    status: str = "Idle"
    holder_type: str = "DoubleTilt"
    mode: str = "IMG"
    diff: Dict[str, float] = field(default_factory=lambda: {
        "camera_length_mm": 800.0, "beamstop_radius_px": 6.0, "thickness_nm": 20.0,
        "aperture_um": 0.0, "depth_nm": 0.0,  # 0 = auto (use FOV / sample depth)
        "use_local_atoms": 1.0,  # 1=local-region from-atoms, 0=analytical kinematical
    })
    # NEW: aberrations & optics
    optics: Dict[str, float] = field(default_factory=lambda: {"cs_mm": 1.0, "aperture_probe_px": 1.4})
    # NEW: drift state (accumulates over time)
    drift: Dict[str, float] = field(default_factory=lambda: {
        "vx_px_per_s": 0.0, "vy_px_per_s": 0.0,   # constant drift velocity
        "accum_x_px": 0.0, "accum_y_px": 0.0,      # accumulated offset
        "line_jitter_px": 0.0,
        "enabled": 0.0,
    })
    # NEW: beam damage + contamination (specimen-degradation effects)
    specimen: Dict[str, float] = field(default_factory=lambda: {
        # beam damage: cumulative dose above 'damage_dose_threshold' (e-/A^2)
        # progressively removes signal in the exposed region.
        "beam_damage_enabled": 0.0,
        "damage_dose_threshold": 5.0e3,    # e-/A^2 before damage onset
        "damage_rate": 1.0,                # how fast contrast is lost past threshold
        # contamination: carbon builds up where the beam dwells, darkening the
        # image and adding diffuse background to diffraction over time.
        "contamination_enabled": 0.0,
        "contamination_rate": 1.0,         # growth per unit exposure
    })


def default_haadf(detector_dict):
    detector_dict["haadf"] = {
        "size": 256, "exposure": 0.1, "binning": 1,
        "field_of_view_um": 20.0, "noise_sigma": 12.0,
        # magnification is kept in sync with field_of_view_um (mag = MAG_K/fov_m)
        "magnification": MAG_K / (20.0 * 1e-6),
        # NEW: dose model parameters
        "dwell_us": 10.0,        # per-pixel dwell time (microseconds)
        "dqe": 0.8,
        "readout_e": 1.5,
        "use_dose_model": 1.0,   # 1 = Poisson-dose, 0 = legacy gaussian
    }


def serialize_ndarray_b64(arr):
    raw = arr.tobytes(order="C")
    b64 = base64.b64encode(raw).decode("ascii")
    return {"__ndarray_b64__": b64, "shape": arr.shape, "dtype": str(arr.dtype)}


def sharpness_metric(img_u16):
    img = img_u16.astype(np.float32)
    gx = np.abs(img[:, 1:] - img[:, :-1]).mean()
    gy = np.abs(img[1:, :] - img[:-1, :]).mean()
    return float(gx + gy)


def bilinear_sample(img, y, x):
    H, W = img.shape
    x0 = np.floor(x).astype(np.int32); y0 = np.floor(y).astype(np.int32)
    x1 = x0 + 1; y1 = y0 + 1
    x0 = np.clip(x0, 0, W - 1); x1 = np.clip(x1, 0, W - 1)
    y0 = np.clip(y0, 0, H - 1); y1 = np.clip(y1, 0, H - 1)
    Ia = img[y0, x0].astype(np.float32); Ib = img[y1, x0].astype(np.float32)
    Ic = img[y0, x1].astype(np.float32); Id = img[y1, x1].astype(np.float32)
    wa = (x1 - x) * (y1 - y); wb = (x1 - x) * (y - y0)
    wc = (x - x0) * (y1 - y); wd = (x - x0) * (y - y0)
    return Ia*wa + Ib*wb + Ic*wc + Id*wd


# ============================================================================
# STEMServer
# ============================================================================
class STEMServer(object):
    def __init__(self, default_sample="fcc_single_crystal", D=64, H=768, W=768):
        print("STEMServer (DT v5) init (fast stage)")
        self.detectors = {}
        default_haadf(self.detectors)
        self.sim = SimMicroscope()
        self.command_log = []
        self.focus_plane_z_m = 0.0
        self._default_DHW = (D, H, W)
        self._default_sample = default_sample
        self._ready = False
        self._init_error = None
        self.vol = None
        self.vol_D = None
        self.current_sample_name = None
        self.current_sample = None
        self.sample_fov_um = 200.0
        self.sample_px_per_um = 1.0
        self.tilt_strength_px_per_slice = 0.35
        # thickness law scale (raw projected-sum units that give ~63% signal)
        self.mfp_scale = 2000.0
        self._last_acquire_t = time.time()
        # Specimen-degradation maps (sample-frame, low-res grids that accumulate
        # exposure). Allocated lazily per-sample in _ensure_specimen_maps().
        self._dose_map = None          # cumulative dose proxy (e-/A^2-ish)
        self._contam_map = None        # cumulative contamination thickness proxy
        self._specimen_grid = 128      # resolution of the maps

    def _ensure_specimen_maps(self):
        g = self._specimen_grid
        if self._dose_map is None or self._dose_map.shape != (g, g):
            self._dose_map = np.zeros((g, g), dtype=np.float32)
            self._contam_map = np.zeros((g, g), dtype=np.float32)

    def reset_specimen(self):
        """Clear accumulated beam damage and contamination (fresh specimen)."""
        self._dose_map = None
        self._contam_map = None
        self._ensure_specimen_maps()
        self._log("reset_specimen", {}, "cleared")
        return {"reset": True}

    def finish_init(self):
        try:
            import importlib
            import samples as _sm
            importlib.reload(_sm)
            _sm.discover(force_reload=True)
            D, H, W = self._default_DHW
            name = self._default_sample
            avail = [s["name"] for s in _sm.list_samples()]
            print(f"[DT] Registered samples: {avail}")
            if name not in avail:
                raise RuntimeError(f"Default sample '{name}' not registered. Available: {avail}")
            print(f"[DT] Generating sample '{name}' {D}x{H}x{W} ...")
            self.current_sample_name = name
            self.current_sample = _sm.get_sample(name)
            self.vol = self.current_sample.generate_volume(D, H, W)
            self.vol_D = D
            self.sample_fov_um = self.current_sample.sample_fov_um
            self.sample_px_per_um = W / self.sample_fov_um
            self.tilt_strength_px_per_slice = self.current_sample.tilt_strength_px_per_slice
            self._calibrate_contrast()
            self._ready = True
            print("[DT] Volume ready.")
        except Exception:
            self._init_error = traceback.format_exc()
            print("[DT] INIT FAILED:\n" + self._init_error)

    def is_ready(self):
        return {"ready": bool(self._ready),
                "error": (self._init_error if not self._ready else None),
                "sample": self.current_sample_name}

    def _calibrate_contrast(self):
        """
        Set mfp_scale so a typical projected column gives a mid-range signal,
        keeping the thickness-saturation curve in its responsive region rather
        than saturating everything to ~1.
        """
        # sample the projected sum on a coarse grid for speed
        sub = self.vol[:, ::8, ::8].sum(axis=0)
        med = float(np.median(sub))
        # choose scale so the median maps to ~0.4 signal: 1-exp(-med/scale)=0.4
        # => scale = med / -ln(0.6) = med / 0.5108
        self.mfp_scale = max(1.0, med / 0.5108)

    def _require_ready(self):
        if not self._ready:
            if self._init_error:
                raise RuntimeError(f"Sample init failed:\n{self._init_error}")
            raise RuntimeError("Sample is still loading. Call is_ready() to check.")

    def _log(self, method, params, result=None):
        self.command_log.append({"t": time.time(), "method": method,
                                 "params": params, "result_preview": str(result)[:200]})

    def get_command_log(self, last_n=50):
        return self.command_log[-int(last_n):]

    def clear_command_log(self):
        self.command_log = []
        return 1

    # ---- sample registry ----
    def list_samples(self):
        r = samples.list_samples()
        self._log("list_samples", {}, f"{len(r)} samples")
        return r

    def get_current_sample(self):
        r = {"name": self.current_sample_name,
             "params": (self.current_sample.params if self.current_sample else None),
             "crystalline": bool(self.current_sample and self.current_sample.get_lattice() is not None)}
        self._log("get_current_sample", {}, r)
        return r

    def load_sample(self, name, params=None, D=None, H=None, W=None):
        params = params or {}
        D0, H0, W0 = self._default_DHW
        D = int(D) if D is not None else D0
        H = int(H) if H is not None else H0
        W = int(W) if W is not None else W0
        sample = samples.get_sample(name, **params)
        print(f"[DT] Loading sample '{name}' ({D}x{H}x{W}) ...")
        prev_ready = self._ready
        self._ready = False
        try:
            self.vol = sample.generate_volume(D, H, W)
            self.vol_D = D
            self.current_sample_name = name
            self.current_sample = sample
            self.sample_fov_um = sample.sample_fov_um
            self.sample_px_per_um = W / self.sample_fov_um
            self.tilt_strength_px_per_slice = sample.tilt_strength_px_per_slice
            self._calibrate_contrast()
            self._ready = True
            self._init_error = None
        except Exception:
            self._init_error = traceback.format_exc()
            self._ready = prev_ready
            raise
        r = {"loaded": name, "shape": [D, H, W], "params": sample.params}
        self._log("load_sample", {"name": name, "params": params}, r)
        print(f"[DT] Sample '{name}' loaded.")
        return r

    # ---- beam ----
    def get_beam(self):
        b = self.sim.beam
        r = {k: float(b.get(k, 0.0)) for k in ["x","y","current_pA","voltage_kV"]}
        self._log("get_beam", {}, r)
        return r

    def set_beam(self, beam_settings, relative=False):
        keys = ["x","y","current_pA","voltage_kV"]
        if not isinstance(beam_settings, dict):
            raise ValueError("beam_settings must be a dict")
        for k in keys:
            if k in beam_settings and beam_settings[k] is not None:
                if relative:
                    self.sim.beam[k] = float(self.sim.beam.get(k, 0.0)) + float(beam_settings[k])
                else:
                    self.sim.beam[k] = float(beam_settings[k])
        r = {"new_beam": {k: float(self.sim.beam.get(k, 0.0)) for k in keys}, "relative": bool(relative)}
        self._log("set_beam", {"beam_settings": beam_settings, "relative": relative}, r)
        return r

    # ---- optics (NEW: aberrations) ----
    def get_optics(self):
        o = self.sim.optics
        r = {"cs_mm": float(o["cs_mm"]), "aperture_probe_px": float(o["aperture_probe_px"])}
        self._log("get_optics", {}, r)
        return r

    def set_optics(self, **kwargs):
        if "cs_mm" in kwargs and kwargs["cs_mm"] is not None:
            self.sim.optics["cs_mm"] = float(kwargs["cs_mm"])
        if "aperture_probe_px" in kwargs and kwargs["aperture_probe_px"] is not None:
            self.sim.optics["aperture_probe_px"] = float(kwargs["aperture_probe_px"])
        r = self.get_optics()
        self._log("set_optics", kwargs, r)
        return r

    # ---- drift (NEW) ----
    def get_drift(self):
        d = self.sim.drift
        r = {k: float(d[k]) for k in d}
        self._log("get_drift", {}, r)
        return r

    def set_drift(self, vx_px_per_s=None, vy_px_per_s=None, line_jitter_px=None,
                  enabled=None, reset_accum=False):
        d = self.sim.drift
        if vx_px_per_s is not None: d["vx_px_per_s"] = float(vx_px_per_s)
        if vy_px_per_s is not None: d["vy_px_per_s"] = float(vy_px_per_s)
        if line_jitter_px is not None: d["line_jitter_px"] = float(line_jitter_px)
        if enabled is not None: d["enabled"] = 1.0 if enabled else 0.0
        if reset_accum:
            d["accum_x_px"] = 0.0; d["accum_y_px"] = 0.0
            self._last_acquire_t = time.time()
        r = {k: float(d[k]) for k in d}
        self._log("set_drift", {"vx": vx_px_per_s, "vy": vy_px_per_s}, r)
        return r

    # ---- specimen degradation (NEW: beam damage + contamination) ----
    def get_specimen(self):
        s = self.sim.specimen
        r = {k: float(s[k]) for k in s}
        # report a summary of accumulated maps
        if self._dose_map is not None:
            r["max_accumulated_dose"] = float(self._dose_map.max())
            r["max_contamination"] = float(self._contam_map.max())
        self._log("get_specimen", {}, r)
        return r

    def set_specimen(self, **kwargs):
        s = self.sim.specimen
        for key in ["damage_dose_threshold", "damage_rate", "contamination_rate"]:
            if kwargs.get(key) is not None:
                s[key] = float(kwargs[key])
        if kwargs.get("beam_damage_enabled") is not None:
            s["beam_damage_enabled"] = 1.0 if kwargs["beam_damage_enabled"] else 0.0
        if kwargs.get("contamination_enabled") is not None:
            s["contamination_enabled"] = 1.0 if kwargs["contamination_enabled"] else 0.0
        r = self.get_specimen()
        self._log("set_specimen", kwargs, r)
        return r

    # ---- mode / diffraction ----
    def get_mode(self):
        r = {"mode": str(self.sim.mode)}
        self._log("get_mode", {}, r)
        return r

    def set_mode(self, mode="IMG"):
        m = str(mode).upper().strip()
        if m not in ("IMG", "DIFF"):
            raise ValueError("mode must be 'IMG' or 'DIFF'")
        self.sim.mode = m
        r = {"mode": m}
        self._log("set_mode", {"mode": mode}, r)
        return r

    def get_diffraction_settings(self):
        d = self.sim.diff
        r = {"camera_length_mm": float(d["camera_length_mm"]),
             "beamstop_radius_px": float(d["beamstop_radius_px"]),
             "thickness_nm": float(d.get("thickness_nm", 20.0)),
             "aperture_um": float(d.get("aperture_um", 0.0)),
             "depth_nm": float(d.get("depth_nm", 0.0)),
             "use_local_atoms": float(d.get("use_local_atoms", 1.0))}
        self._log("get_diffraction_settings", {}, r)
        return r

    def set_diffraction_settings(self, **kwargs):
        if kwargs.get("camera_length_mm") is not None:
            self.sim.diff["camera_length_mm"] = float(kwargs["camera_length_mm"])
        if kwargs.get("beamstop_radius_px") is not None:
            self.sim.diff["beamstop_radius_px"] = float(kwargs["beamstop_radius_px"])
        if kwargs.get("thickness_nm") is not None:
            self.sim.diff["thickness_nm"] = float(kwargs["thickness_nm"])
        if kwargs.get("aperture_um") is not None:
            self.sim.diff["aperture_um"] = float(kwargs["aperture_um"])
        if kwargs.get("depth_nm") is not None:
            self.sim.diff["depth_nm"] = float(kwargs["depth_nm"])
        if kwargs.get("use_local_atoms") is not None:
            self.sim.diff["use_local_atoms"] = 1.0 if kwargs["use_local_atoms"] else 0.0
        r = self.get_diffraction_settings()
        self._log("set_diffraction_settings", kwargs, r)
        return r

    # ---- detectors / stage ----
    def get_detectors(self):
        r = list(self.detectors.keys())
        self._log("get_detectors", {}, r)
        return r

    def device_settings(self, device, **args):
        if device not in self.detectors:
            return 0
        # Magnification and field-of-view are two views of the same quantity
        # (mag = MAG_K / fov_metres). Accept either; keep both consistent.
        if "magnification" in args and args["magnification"] is not None:
            fov_um = mag_to_fov_um(float(args["magnification"]))
            args["field_of_view_um"] = fov_um
            self.detectors[device]["magnification"] = float(args["magnification"])
        for k, v in args.items():
            if k in self.detectors[device]:
                self.detectors[device][k] = v
        # If FOV was set directly, refresh the derived magnification too.
        if "field_of_view_um" in args and args["field_of_view_um"] is not None:
            self.detectors[device]["magnification"] = fov_um_to_mag(
                float(args["field_of_view_um"]))
        self._log("device_settings", {"device": device, **args}, 1)
        return 1

    def get_magnification(self, device="haadf"):
        if device not in self.detectors:
            return 0
        fov_um = float(self.detectors[device].get("field_of_view_um", 20.0))
        mag = fov_um_to_mag(fov_um)
        r = {"magnification": mag, "field_of_view_um": fov_um}
        self._log("get_magnification", {"device": device}, r)
        return r

    def set_magnification(self, magnification, device="haadf"):
        if device not in self.detectors:
            return 0
        fov_um = mag_to_fov_um(float(magnification))
        self.detectors[device]["field_of_view_um"] = fov_um
        self.detectors[device]["magnification"] = float(magnification)
        r = {"magnification": float(magnification), "field_of_view_um": fov_um}
        self._log("set_magnification", {"magnification": magnification, "device": device}, r)
        return r

    def get_stage(self):
        st = self.sim.stage
        r = [st["x"], st["y"], st["z"], st["a"], st["b"]]
        self._log("get_stage", {}, r)
        return r

    # Stage travel limits (safety interlock). x/y/z in METRES, a/b in DEGREES.
    # A move whose TARGET exceeds any limit is rejected outright (nothing moves),
    # mimicking a hardware soft-limit that protects the stage/specimen/pole-piece.
    STAGE_LIMITS = {
        "x": 1.5e-3,   # +/- 1.5 mm
        "y": 1.5e-3,   # +/- 1.5 mm
        "z": 1.0e-3,   # +/- 1.0 mm
        "a": 30.0,     # +/- 30 degrees
        "b": 30.0,     # +/- 30 degrees
    }

    def set_stage(self, stage_positions, relative=True):
        keys = ["x","y","z","a","b"]
        move = {k: 0.0 for k in keys}
        if isinstance(stage_positions, dict):
            for k in keys:
                if k in stage_positions and stage_positions[k] is not None:
                    move[k] = float(stage_positions[k])
        elif isinstance(stage_positions, (list, tuple)):
            for i, k in enumerate(keys):
                if i < len(stage_positions) and stage_positions[i] is not None:
                    move[k] = float(stage_positions[i])
        else:
            raise ValueError("stage_positions must be dict or list/tuple")

        # Compute the intended target and check it against the soft limits BEFORE
        # applying anything. Reject the whole move if any axis is out of range.
        target = {}
        for k in keys:
            target[k] = (self.sim.stage[k] + move[k]) if relative else move[k]
        violations = []
        for k in keys:
            lim = self.STAGE_LIMITS[k]
            if abs(target[k]) > lim + 1e-12:
                if k in ("x", "y", "z"):
                    violations.append(
                        f"{k}={target[k]*1e3:+.3f} mm exceeds +/-{lim*1e3:.3f} mm")
                else:
                    violations.append(
                        f"{k}={target[k]:+.2f} deg exceeds +/-{lim:.1f} deg")
        if violations:
            msg = ("Stage move rejected by safety limits: " + "; ".join(violations)
                   + ". Stage did not move.")
            self._log("set_stage", {"stage_positions": stage_positions,
                                     "relative": relative}, {"rejected": msg})
            raise ValueError(msg)

        for k in keys:
            self.sim.stage[k] = target[k]
        r = {"new_stage": [self.sim.stage[k] for k in keys], "relative": bool(relative)}
        self._log("set_stage", {"stage_positions": stage_positions, "relative": relative}, r)
        return r

    # ---- imaging (items 1-4) ----
    def _render_camera_image_u16(self, device, for_autofocus=False):
        self._require_ready()
        det = self.detectors[device]
        out_size = int(det["size"])
        fov_um = float(det["field_of_view_um"])

        b = self.sim.beam
        current_pA = float(b.get("current_pA", 50.0))
        voltage_kV = float(b.get("voltage_kV", 200.0))

        sx_um = self.sim.stage["x"] * 1e6
        sy_um = self.sim.stage["y"] * 1e6
        W = self.vol.shape[2]; H = self.vol.shape[1]
        cx = (0.5 * W + (sx_um * self.sample_px_per_um)) % W
        cy = (0.5 * H + (sy_um * self.sample_px_per_um)) % H

        # Item 3: update accumulated drift BEFORE choosing the sampling center, so
        # successive frames are translated relative to one another. (Skip during
        # autofocus so scoring is stable.)
        intra_dx = intra_dy = 0.0
        if not for_autofocus and self.sim.drift.get("enabled", 0.0) >= 0.5:
            now = time.time()
            dt = max(0.0, now - self._last_acquire_t)
            self._last_acquire_t = now
            self.sim.drift["accum_x_px"] += self.sim.drift["vx_px_per_s"] * dt
            self.sim.drift["accum_y_px"] += self.sim.drift["vy_px_per_s"] * dt
            # accumulated offset shifts the whole frame (between-frame drift)
            cx = (cx + self.sim.drift["accum_x_px"]) % W
            cy = (cy + self.sim.drift["accum_y_px"]) % H
            # intra-frame drift (shear within one frame) computed from frame time
            frame_t = float(det["dwell_us"]) * 1e-6 * out_size * out_size
            intra_dx = self.sim.drift["vx_px_per_s"] * frame_t
            intra_dy = self.sim.drift["vy_px_per_s"] * frame_t

        half = 0.5 * fov_um * self.sample_px_per_um
        xs = np.linspace(cx - half, cx + half, out_size, dtype=np.float32)
        ys = np.linspace(cy - half, cy + half, out_size, dtype=np.float32)
        Y0, X0 = np.meshgrid(ys, xs, indexing="ij")

        a_deg = float(self.sim.stage.get("a", 0.0))
        b_deg = float(self.sim.stage.get("b", 0.0))
        sa = np.tan(np.deg2rad(a_deg)) * self.tilt_strength_px_per_slice
        sb = np.tan(np.deg2rad(b_deg)) * self.tilt_strength_px_per_slice

        D = self.vol_D
        z0 = (D - 1) * 0.5
        proj = np.zeros((out_size, out_size), dtype=np.float32)
        for z in range(D):
            dz = (z - z0)
            Xq = X0 - sb * dz
            Yq = Y0 - sa * dz
            proj += bilinear_sample(self.vol[z], Yq, Xq)

        # Item 4: thickness saturation + Z-contrast-ish nonlinearity already
        # baked into per-sample intensities. Convert raw sum -> scattering frac.
        signal = thickness_contrast(proj, self.mfp_scale)  # in [0,1)

        # Item 2: PSF convolution (defocus + aberrations)
        dz_um = (self.sim.stage["z"] - self.focus_plane_z_m) * 1e6
        defocus_nm = dz_um * 1000.0
        # Physical pixel size so defocus blur scales correctly with FOV. A wide
        # FOV means each pixel covers more sample, so a given defocus blurs fewer
        # pixels; a narrow FOV resolves the blur. Without this the defocus_px term
        # saturated instantly and the autofocus curve was flat.
        nm_per_px = (fov_um * 1000.0) / max(1, out_size)
        psf = make_psf(defocus_nm,
                       cs_mm=float(self.sim.optics["cs_mm"]),
                       aperture_probe_px=float(self.sim.optics["aperture_probe_px"]),
                       kv=voltage_kV, pixel_nm=nm_per_px)
        signal = convolve2d_fft(signal, psf)
        signal = np.clip(signal, 0, None)

        # Resolution limit tied to the sample's INHERENT length scale. Each sample
        # declares `feature_scale_nm` (the size of its finest meaningful detail:
        # atomic-column spacing for crystals, particle size for nanoparticles). If
        # the current pixel size (set by FOV / magnification) is coarser than that
        # scale, the fine detail cannot be resolved -- so we blur it away in
        # proportion to how far under-resolved it is. Consequence: you must raise
        # magnification (shrink the FOV) to see the structure, exactly as on a real
        # instrument, and at high mag drift/dose then dominate.
        feat_nm = float(getattr(self.current_sample, "feature_scale_nm", 0.0) or 0.0)
        if feat_nm > 0.0:
            # need ~2 px across a feature to resolve it (Nyquist-ish)
            needed_nm_per_px = feat_nm / 2.0
            under = nm_per_px / max(1e-9, needed_nm_per_px)
            if under > 1.0:
                # blur sigma (in px) grows as we go further below the resolution
                # needed for this sample's features; capped so it stays finite.
                sigma_px = min(6.0, 0.5 * (under - 1.0))
                if sigma_px > 0.15:
                    from scipy.ndimage import gaussian_filter
                    signal = gaussian_filter(signal, sigma=sigma_px).astype(np.float32)

        # voltage affects contrast slightly
        voltage_scale = max(0.1, min(3.0, voltage_kV / 200.0))
        signal = signal * (1.0 / (0.85 + 0.15 * voltage_scale))

        # --- Specimen degradation: beam damage + contamination (review 2) ---
        # Accumulate exposure in the region currently under the beam, then apply
        # the cumulative effect. Skipped during autofocus so it doesn't corrupt
        # focus scoring (though damage during AF is modeled separately by passing
        # for_autofocus and still reading the current maps).
        sp = self.sim.specimen
        damage_on = float(sp.get("beam_damage_enabled", 0.0)) >= 0.5
        contam_on = float(sp.get("contamination_enabled", 0.0)) >= 0.5
        if damage_on or contam_on:
            self._ensure_specimen_maps()
            g = self._specimen_grid
            # Map the current FOV window to a sub-rectangle of the specimen grid.
            # Stage position determines the window center in normalized [0,1].
            fov_frac = min(1.0, fov_um / max(1e-6, self.sample_fov_um))
            cxn = 0.5 + (sx_um / max(1e-6, self.sample_fov_um))
            cyn = 0.5 + (sy_um / max(1e-6, self.sample_fov_um))
            half_f = 0.5 * fov_frac
            gx0 = int(np.clip((cxn - half_f) * g, 0, g-1))
            gx1 = int(np.clip((cxn + half_f) * g, 0, g-1)) + 1
            gy0 = int(np.clip((cyn - half_f) * g, 0, g-1))
            gy1 = int(np.clip((cyn + half_f) * g, 0, g-1)) + 1
            # Exposure increment per acquisition. Tuned so a typical frame adds a
            # modest fraction of the damage threshold -> gradual damage over
            # several frames rather than instantaneous saturation.
            inc = (current_pA / 100.0) * (float(det.get("dwell_us", 10.0)) / 20.0) * 300.0
            if not for_autofocus and (gx1 > gx0) and (gy1 > gy0):
                if damage_on:
                    self._dose_map[gy0:gy1, gx0:gx1] += inc
                if contam_on:
                    self._contam_map[gy0:gy1, gx0:gx1] += (inc / 300.0) * float(sp.get("contamination_rate",1.0))
            from scipy.ndimage import zoom as _zoom
            def _patch_to_out(maparr):
                patch = maparr[gy0:gy1, gx0:gx1]
                if patch.size == 0:
                    return np.zeros((out_size, out_size), dtype=np.float32)
                zy = out_size / patch.shape[0]; zx = out_size / patch.shape[1]
                return _zoom(patch, (zy, zx), order=1)[:out_size, :out_size]
            # Two DISTINCT effects, applied after dose-model renormalization
            # (per-frame max-normalization would otherwise cancel a uniform change):
            #   - Beam DAMAGE: mass loss / sputtering removes scatterers, so the
            #     HAADF signal DROPS -> multiplicative attenuation (->0), darker.
            #   - CONTAMINATION: carbon builds up where the beam dwells, ADDING
            #     projected mass-thickness. HAADF signal scales with mass-thickness,
            #     so contaminated regions get BRIGHTER -> additive brightening.
            # 1.0 = pristine for the damage factor; 0.0 = none for contam brighten.
            self._degradation_factor = np.ones((out_size, out_size), dtype=np.float32)
            self._contam_brighten = None
            if damage_on:
                dose_patch = _patch_to_out(self._dose_map)
                thr = float(sp.get("damage_dose_threshold", 5e3))
                rate = float(sp.get("damage_rate", 1.0))
                excess = np.clip(dose_patch - thr, 0, None) / max(1.0, thr)
                self._degradation_factor *= np.exp(-rate * excess).astype(np.float32)
            if contam_on:
                contam_patch = _patch_to_out(self._contam_map)
                # fraction of "full" contamination in [0,1); brightening grows with
                # accumulated carbon. Applied as an additive HAADF signal increase.
                contam_frac = 1.0 - np.exp(-0.004 * contam_patch)
                self._contam_brighten = (contam_frac * 22000.0).astype(np.float32)
            # Damage reduces electrons pre-noise (correct noise statistics); the
            # contamination brightening is applied post-normalization below.
            signal = signal * self._degradation_factor
        else:
            self._degradation_factor = None
            self._contam_brighten = None

        rng = np.random.default_rng(int(time.time() * 1e6) % (2**32))

        # Item 3: intra-frame drift shear + line jitter (offset already applied
        # to the sampling center above).
        if not for_autofocus and self.sim.drift.get("enabled", 0.0) >= 0.5:
            if abs(intra_dx) > 1e-6 or abs(intra_dy) > 1e-6 or self.sim.drift["line_jitter_px"] > 0:
                signal = apply_scan_distortion(
                    signal,
                    drift_px_xy=(intra_dx, intra_dy),
                    line_jitter_px=float(self.sim.drift["line_jitter_px"]),
                    rng=rng,
                )

        # Item 1: Poisson-dose noise
        if not for_autofocus and float(det.get("use_dose_model", 1.0)) >= 0.5:
            dwell_s = float(det["dwell_us"]) * 1e-6
            # electrons per pixel = current(A)/e * dwell ; current_pA in pA
            dose_e = (current_pA * 1e-12 / 1.602e-19) * dwell_s
            counts = apply_dose_noise(signal, dose_e,
                                      dqe=float(det["dqe"]),
                                      readout_e=float(det["readout_e"]),
                                      rng=rng)
            # Per-frame normalization for display dynamic range. Damage/contam
            # would be cancelled by this (a uniform attenuation rescales away), so
            # we re-apply the degradation factor AFTER normalization: a degraded
            # region/frame ends up genuinely darker than a pristine one.
            mx = counts.max()
            out = (counts / mx * 60000.0) if mx > 1e-6 else counts
            if getattr(self, "_degradation_factor", None) is not None:
                out = out * self._degradation_factor            # damage: darker
            if getattr(self, "_contam_brighten", None) is not None:
                out = out + self._contam_brighten                # contamination: brighter
            return np.clip(out, 0, 65535).astype(np.uint16)
        else:
            # legacy / autofocus path: scale signal, light gaussian noise
            current_scale = max(0.05, current_pA / 50.0)
            img_f = signal * 60000.0 * current_scale
            if not for_autofocus:
                noise_sigma = float(det.get("noise_sigma", 12.0))
                img_f = img_f + rng.normal(0.0, noise_sigma, img_f.shape).astype(np.float32)
            return np.clip(img_f, 0, 65535).astype(np.uint16)

    # ---- diffraction (item 5) ----
    def _normalize_to_u16(self, img_f):
        x = img_f.astype(np.float32); x -= x.min()
        mx = float(x.max())
        if mx > 1e-6: x = x / mx
        return np.clip(x * 65535.0, 0, 65535).astype(np.uint16)

    def _render_diffraction_fft_proxy(self, img_u16, beamstop_radius_px=0):
        x = img_u16.astype(np.float32); x = x - float(x.mean())
        F = np.fft.fftshift(np.fft.fft2(x))
        P = np.log1p(np.abs(F)).astype(np.float32)
        if beamstop_radius_px and beamstop_radius_px > 0:
            H, W = P.shape
            yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
            cy, cx = (H - 1) * 0.5, (W - 1) * 0.5
            r = np.sqrt((yy - cy)**2 + (xx - cx)**2)
            P[r <= float(beamstop_radius_px)] = 0.0
        return self._normalize_to_u16(P)

    def acquire_image(self, device, **args):
        self._require_ready()
        if device not in self.detectors:
            return None
        det = self.detectors[device]
        out_size = int(det["size"])

        if str(self.sim.mode).upper() == "DIFF":
            beamstop = float(self.sim.diff.get("beamstop_radius_px", 6.0))
            rng = np.random.default_rng(int(time.time()*1e6) % (2**32))
            method = None
            img_f = None

            # Try the local-region atom path first (item 6: stage-aware diffraction).
            # Falls back automatically if the sample doesn't expose atoms.
            use_local = float(self.sim.diff.get("use_local_atoms", 1.0)) >= 0.5
            if use_local and self.current_sample is not None:
                # Determine diffracting region:
                #   aperture_um: 0 (auto) -> current detector FOV
                #   depth_nm:    0 (auto) -> full sample depth
                ap_um = float(self.sim.diff.get("aperture_um", 0.0))
                if ap_um <= 0:
                    ap_um = float(det.get("field_of_view_um", 20.0))
                depth_nm = float(self.sim.diff.get("depth_nm", 0.0))
                if depth_nm <= 0:
                    # full sample depth = D voxels * (sample_fov_um / W voxels) approximated
                    # Use volume depth in microns -> nm
                    depth_um = self.vol_D * (self.sample_fov_um / max(1, self.vol.shape[2]))
                    depth_nm = max(1.0, depth_um * 1000.0)
                # Stage position in microns (for locality on non-uniform samples)
                cx_um = float(self.sim.stage.get("x", 0.0)) * 1e6
                cy_um = float(self.sim.stage.get("y", 0.0)) * 1e6
                try:
                    atoms_pos, atoms_Z = self.current_sample.get_atoms_in_region(
                        cx_um, cy_um, ap_um / 2.0, depth_nm)
                except Exception:
                    atoms_pos, atoms_Z = None, None
                if atoms_pos is not None and len(atoms_pos) > 0:
                    img_f = diffraction_from_atoms(
                        atoms_pos, atoms_Z, out_size,
                        tilt_a_deg=float(self.sim.stage.get("a", 0.0)),
                        tilt_b_deg=float(self.sim.stage.get("b", 0.0)),
                        kv=float(self.sim.beam.get("voltage_kV", 200.0)),
                        camera_length_mm=float(self.sim.diff["camera_length_mm"]),
                        beamstop_radius_px=beamstop,
                        thickness_nm=float(self.sim.diff.get("thickness_nm", 20.0)),
                        rng=rng)
                    method = f"from_atoms (N={len(atoms_pos)})"

            # Fallback: analytical kinematical from a declared lattice. This is
            # only reached if a sample exposes a lattice but no atoms -- with the
            # unified atom path, all crystalline/particle samples expose atoms, so
            # this is a rarely-used compatibility branch.
            if img_f is None:
                lat = self.current_sample.get_lattice() if self.current_sample else None
                if lat is not None:
                    img_f = kinematical_diffraction(
                        lat, out_size,
                        tilt_a_deg=float(self.sim.stage.get("a", 0.0)),
                        tilt_b_deg=float(self.sim.stage.get("b", 0.0)),
                        kv=float(self.sim.beam.get("voltage_kV", 200.0)),
                        camera_length_mm=float(self.sim.diff["camera_length_mm"]),
                        beamstop_radius_px=beamstop,
                        thickness_nm=float(self.sim.diff.get("thickness_nm", 20.0)),
                        rng=rng)
                    method = "kinematical_lattice"

            # Single diffraction channel: no FFT proxy. If a sample exposes
            # neither atoms nor a lattice, return an empty detector (just the
            # central beam + noise) rather than a physically meaningless FFT.
            if img_f is None:
                img_f = np.zeros((out_size, out_size), dtype=np.float32)
                cyx = out_size / 2.0
                yy, xx = np.mgrid[0:out_size, 0:out_size].astype(np.float32)
                d2c = (xx - cyx)**2 + (yy - cyx)**2
                img_f += 65535.0 * 1.0 * np.exp(-d2c / (2 * (3.0)**2))
                if beamstop and beamstop > 0:
                    img_f[np.sqrt(d2c) <= beamstop] = 0.0
                method = "empty (no atomic structure)"
            noise = rng.normal(0.0, 250.0, img_f.shape).astype(np.float32)
            img = np.clip(img_f + noise, 0, 65535).astype(np.uint16)

            r = serialize_ndarray_b64(img)
            self._log("acquire_image", {"device": device, "mode": "DIFF", "diff_method": method},
                      f"image {img.shape}")
            return r
        else:
            img = self._render_camera_image_u16(device)
            r = serialize_ndarray_b64(img)
            self._log("acquire_image", {"device": device, "mode": "IMG"}, f"image {img.shape}")
            return r

    def autofocus(self, device="haadf", z_range_um=2.0, z_steps=9):
        self._require_ready()
        if device not in self.detectors:
            raise ValueError(f"Unknown device {device}")
        z0 = self.sim.stage["z"]
        zs_um = np.linspace(-z_range_um, z_range_um, int(z_steps))
        zs_m = z0 + (zs_um * 1e-6)
        scores = []
        best_score = -1e18; best_z = z0
        for z_m, z_um in zip(zs_m, zs_um):
            self.sim.stage["z"] = float(z_m)
            # On beam-sensitive specimens, let damage accumulate DURING the sweep
            # so later Z-steps see a degraded specimen -> the sharpness curve is
            # corrupted and AF can legitimately diverge (review 3). On robust
            # specimens (damage disabled) the sweep is non-destructive.
            damaging = float(self.sim.specimen.get("beam_damage_enabled", 0.0)) >= 0.5
            img = self._render_camera_image_u16(device, for_autofocus=not damaging)
            sc = sharpness_metric(img)
            scores.append((float(z_um), float(sc)))
            if sc > best_score:
                best_score = sc; best_z = float(z_m)

        # --- Convergence assessment (review 3) ---
        # Autofocus should NOT always succeed. Judge the sharpness curve:
        #  - weak peak prominence above the baseline -> low-contrast/flat, AF unreliable
        #  - multiple comparable maxima              -> multimodal, ambiguous
        sc_arr = np.array([s for _, s in scores], dtype=np.float64)
        floor = float(sc_arr.min())
        peak = float(sc_arr.max())
        # prominence: how far the peak rises above the baseline, relative to baseline
        contrast = (peak - floor) / (abs(floor) + 1e-9)
        # count local maxima within 90% of the (prominence-scaled) peak
        thresh_level = floor + 0.9 * (peak - floor)
        near_peak = 0
        for i in range(1, len(sc_arr)-1):
            if sc_arr[i] >= sc_arr[i-1] and sc_arr[i] >= sc_arr[i+1] and sc_arr[i] >= thresh_level:
                near_peak += 1
        if len(sc_arr) >= 2 and sc_arr[0] >= thresh_level and sc_arr[0] > sc_arr[1]: near_peak += 1
        if len(sc_arr) >= 2 and sc_arr[-1] >= thresh_level and sc_arr[-1] > sc_arr[-2]: near_peak += 1

        converged = True
        reason = "ok"
        # thresholds (could be environment-tuned)
        min_contrast = float(self.sim.specimen.get("af_min_contrast", 0.08))
        if contrast < min_contrast:
            converged = False
            reason = f"low contrast (peak/floor ratio {contrast:.3f} < {min_contrast:.3f}); specimen may be low-contrast or beam-damaged"
        elif near_peak >= 3:
            converged = False
            reason = f"multimodal sharpness curve ({near_peak} comparable maxima); focus ambiguous"

        # If not converged, do NOT commit to best_z (leave stage where it was) to
        # mimic a real AF routine refusing/failing rather than guessing.
        if converged:
            self.sim.stage["z"] = best_z
        else:
            self.sim.stage["z"] = z0

        result = {"converged": bool(converged),
                  "reason": reason,
                  "best_z_m": float(best_z),
                  "best_z_um_relative": float((best_z - z0) * 1e6),
                  "curve_contrast": float(contrast),
                  "n_candidate_peaks": int(near_peak),
                  "scores": scores}
        self._log("autofocus", {"device": device, "z_range_um": z_range_um,
                                "z_steps": z_steps, "converged": converged}, result)
        return result

    # ---- simulation environment (review 3): named realism scenarios ----
    def set_environment(self, name="pristine"):
        """Configure a bundle of realism settings for a named test scenario.
        This is the 'simulation environment' a user selects to stress-test their
        code under specific conditions without changing the specimen itself."""
        name = str(name).lower().strip()
        presets = {
            "pristine": {  # ideal conditions: no drift, no damage, high dose
                "drift": dict(vx_px_per_s=0, vy_px_per_s=0, line_jitter_px=0, enabled=False),
                "specimen": dict(beam_damage_enabled=False, contamination_enabled=False),
                "detector": dict(dwell_us=20.0, dqe=0.9, readout_e=1.0),
                "af_min_contrast": 0.05,
            },
            "beam_sensitive": {  # damage accumulates quickly; AF can diverge
                "drift": dict(vx_px_per_s=2, vy_px_per_s=1, line_jitter_px=0.3, enabled=True),
                "specimen": dict(beam_damage_enabled=True, contamination_enabled=False,
                                 damage_dose_threshold=4.0e3, damage_rate=0.7),
                "detector": dict(dwell_us=10.0, dqe=0.8, readout_e=1.5),
                "af_min_contrast": 0.12,
            },
            "contaminating": {  # carbon builds up where the beam dwells
                "drift": dict(vx_px_per_s=3, vy_px_per_s=2, line_jitter_px=0.4, enabled=True),
                "specimen": dict(beam_damage_enabled=False, contamination_enabled=True,
                                 contamination_rate=3.0),
                "detector": dict(dwell_us=15.0, dqe=0.8, readout_e=1.5),
                "af_min_contrast": 0.10,
            },
            "thick_drifting": {  # thick noisy sample with strong drift
                "drift": dict(vx_px_per_s=12, vy_px_per_s=7, line_jitter_px=1.0, enabled=True),
                "specimen": dict(beam_damage_enabled=False, contamination_enabled=False),
                "detector": dict(dwell_us=6.0, dqe=0.7, readout_e=2.5),
                "af_min_contrast": 0.10,
            },
            "low_dose": {  # dose-limited: very noisy, AF struggles
                "drift": dict(vx_px_per_s=4, vy_px_per_s=2, line_jitter_px=0.5, enabled=True),
                "specimen": dict(beam_damage_enabled=True, contamination_enabled=False,
                                 damage_dose_threshold=2.5e3, damage_rate=0.8),
                "detector": dict(dwell_us=2.0, dqe=0.75, readout_e=2.0),
                "af_min_contrast": 0.15,
            },
        }
        if name not in presets:
            raise ValueError(f"Unknown environment '{name}'. Options: {list(presets.keys())}")
        cfg = presets[name]
        self.set_drift(**cfg["drift"], reset_accum=True)
        self.set_specimen(**cfg["specimen"])
        for k, v in cfg["detector"].items():
            self.detectors["haadf"][k] = v
        self.sim.specimen["af_min_contrast"] = cfg["af_min_contrast"]
        self.reset_specimen()
        self._current_environment = name
        r = {"environment": name, "config": cfg}
        self._log("set_environment", {"name": name}, r)
        return r

    def get_environment(self):
        return {"environment": getattr(self, "_current_environment", "pristine"),
                "available": ["pristine","beam_sensitive","contaminating","thick_drifting","low_dose"]}

    def close(self):
        self._log("close", {}, 1)
        return 1


# ============================================================================
# Netstring JSON-RPC protocol
# ============================================================================
class NetstringJSONProtocol(protocol.Protocol):
    def __init__(self, server_instance):
        self.buffer = b""
        self.server_instance = server_instance

    def dataReceived(self, data):
        self.buffer += data
        while True:
            colon = self.buffer.find(b":")
            if colon < 0: return
            length_str = self.buffer[:colon]
            if not length_str: return
            try:
                length = int(length_str)
            except ValueError:
                comma = self.buffer.find(b",")
                self.buffer = self.buffer[comma+1:] if comma >= 0 else b""
                continue
            if len(self.buffer) < colon + 1 + length + 1: return
            payload = self.buffer[colon+1:colon+1+length]
            trailing = self.buffer[colon+1+length:colon+1+length+1]
            if trailing != b",":
                comma = self.buffer.find(b",")
                self.buffer = self.buffer[comma+1:] if comma >= 0 else b""
                continue
            self.buffer = self.buffer[colon+1+length+1:]
            self._handle_payload(payload)

    def _handle_payload(self, payload_bytes):
        try:
            request = json.loads(payload_bytes.decode("utf-8"))
            method = request.get("method"); params = request.get("params", {})
            req_id = request.get("id", None)
            d = threads.deferToThread(self._dispatch_method, method, params)
            d.addCallback(lambda result: self._send_success(req_id, result))
            d.addErrback(lambda f: self._send_error(req_id, str(f)))
        except Exception as e:
            self._send_error(None, f"Invalid JSON payload: {e}")

    def _dispatch_method(self, method, params):
        if not hasattr(self.server_instance, method):
            raise AttributeError(f"Method {method} not implemented.")
        func = getattr(self.server_instance, method)
        return func(**params) if isinstance(params, dict) else func(params)

    def _send_success(self, req_id, result):
        self._write_netstring({"jsonrpc": "2.0", "id": req_id, "result": result})

    def _send_error(self, req_id, message):
        self._write_netstring({"jsonrpc": "2.0", "id": req_id, "error": str(message)})

    def _write_netstring(self, obj):
        payload = json.dumps(obj, separators=(",", ":")).encode("utf-8")
        self.transport.write(f"{len(payload)}:".encode("ascii") + payload + b",")


class NetstringFactory(Factory):
    def __init__(self, server_instance):
        self.server_instance = server_instance
    def buildProtocol(self, addr):
        return NetstringJSONProtocol(self.server_instance)


_SERVER_INSTANCE = None


def main(host="127.0.0.1", port=9094):
    global _SERVER_INSTANCE
    if _SERVER_INSTANCE is not None:
        print("Server already initialized. Restart runtime for a fresh server.")
        return _SERVER_INSTANCE
    server_inst = STEMServer()
    _SERVER_INSTANCE = server_inst
    factory = NetstringFactory(server_inst)
    reactor.listenTCP(port, factory, interface=host)
    print(f"STEM Twisted DT server listening on {host}:{port} (initializing sample...)")
    reactor.callWhenRunning(lambda: threads.deferToThread(server_inst.finish_init))
    if reactor.running:
        print("Reactor already running; listener installed.")
        return server_inst
    reactor.run(installSignalHandlers=False)
    return server_inst


if __name__ == "__main__":
    main()


In [ ]:
import time
time.sleep(2)


## 3. Start the server

In [ ]:
import threading, time, socket, traceback
import stem_server_twisted_colab

HOST, PORT = "127.0.0.1", 9094
_server_status = {"error": None}

def _run_server():
    try:
        stem_server_twisted_colab.main(host=HOST, port=PORT)
    except Exception:
        _server_status["error"] = traceback.format_exc()
        print(_server_status["error"])

threading.Thread(target=_run_server, daemon=True).start()

deadline = time.time() + 30
while time.time() < deadline:
    if _server_status["error"]:
        raise RuntimeError("Server crashed:\n" + _server_status["error"])
    try:
        with socket.create_connection((HOST, PORT), timeout=1):
            print(f"Port open on {HOST}:{PORT}")
            break
    except (ConnectionRefusedError, OSError):
        time.sleep(0.3)
else:
    raise TimeoutError("Port never opened. Restart the runtime.")
print("Waiting for sample volume to generate ...")


## 4. Write the client

In [ ]:
%%writefile stem_client.py
"""
stem_client.py

Two client classes, deliberately separated so the architecture states plainly
which calls are portable to a real instrument and which exist only for the twin:

  MicroscopeControlClient
      Instrument control. EVERY method here has a real-microscope counterpart
      (stage, beam, mode, detector settings, image acquisition, autofocus). A
      real deployment REPLACES this object with a vendor-SDK client exposing the
      same operations -- this is the surface a portable workflow is written
      against, and the one `TwinBackend` (see microscope_backend.py) implements.

  SimulationHarness
      Test-harness configuration for the digital twin. NONE of these methods has
      a real-instrument counterpart -- they choose the specimen, set the
      simulation environment, and inject drift / beam damage / contamination so a
      workflow can be stress-tested without physical sample preparation. A real
      deployment has NO SimulationHarness: there is nothing for it to configure.

Relationship: SimulationHarness wraps a MicroscopeControlClient and sends its
twin-only commands over that same connection. The control client is the portable
surface; the harness is twin-specific scaffolding layered beside it. A workflow
that only ever touches the control client is, by construction, free of any
simulation-only dependency.
"""
import socket, json, base64
import numpy as np


# ===========================================================================
# Instrument control  (portable; real instruments expose the same operations)
# ===========================================================================
class MicroscopeControlClient:
    """The instrument-control surface. Replace this object with a vendor-SDK
    client to drive a real microscope; the workflow code above it is unchanged."""

    def __init__(self, host="127.0.0.1", port=9094, timeout=30):
        self.host = host; self.port = port; self.timeout = timeout; self._next_id = 1

    # ---- transport (netstring JSON-RPC) ----
    def _to_netstring(self, obj):
        payload = json.dumps(obj, separators=(",", ":")).encode("utf-8")
        return f"{len(payload)}:".encode("ascii") + payload + b","

    def _recv_exact(self, sock, n):
        chunks = []; remaining = n
        while remaining > 0:
            chunk = sock.recv(remaining)
            if not chunk: raise ConnectionError("closed while reading")
            chunks.append(chunk); remaining -= len(chunk)
        return b"".join(chunks)

    def _recv_netstring(self, sock):
        length_bytes = b""
        while True:
            c = sock.recv(1)
            if not c: raise ConnectionError("no response")
            if c == b":": break
            length_bytes += c
        length = int(length_bytes.decode("ascii"))
        payload = self._recv_exact(sock, length)
        if self._recv_exact(sock, 1) != b",":
            raise RuntimeError("malformed netstring")
        return json.loads(payload.decode("utf-8"))

    def _decode(self, obj):
        if isinstance(obj, dict) and "__ndarray_b64__" in obj:
            raw = base64.b64decode(obj["__ndarray_b64__"])
            return np.frombuffer(raw, dtype=np.dtype(obj["dtype"])).reshape(obj["shape"])
        return obj

    def _call(self, method, params=None):
        if params is None: params = {}
        msg = {"jsonrpc": "2.0", "id": self._next_id, "method": method, "params": params}
        self._next_id += 1
        with socket.create_connection((self.host, self.port), timeout=self.timeout) as sock:
            sock.settimeout(self.timeout)
            sock.sendall(self._to_netstring(msg))
            reply = self._recv_netstring(sock)
        if "error" in reply:
            raise RuntimeError(f"Server error: {reply['error']}")
        return reply.get("result", None)

    # ---- readiness (transport-level; on real hardware this is the connection check) ----
    def is_ready(self): return self._call("is_ready")
    def wait_until_ready(self, timeout=300, poll=1.0):
        import time
        deadline = time.time() + timeout
        while time.time() < deadline:
            r = self.is_ready()
            if r.get("error"): raise RuntimeError("Server init failed:\n" + r["error"])
            if r.get("ready"): return r
            time.sleep(poll)
        raise TimeoutError(f"not ready within {timeout}s")

    # ---- detector configuration ----
    # Geometry (size, field_of_view_um, dwell_us, binning) maps to real detector
    # controls. A few keys (noise_sigma, dqe, readout_e, use_dose_model) are
    # simulation-only knobs of the twin's noise model -- documented as such.
    def get_detectors(self): return self._call("get_detectors")
    def device_settings(self, device, **kw): return self._call("device_settings", {"device": device, **kw})

    # Magnification is the same quantity as field of view (mag = MAG_K / fov_metres).
    # Setting one updates the other. These mirror a real instrument's mag control.
    def get_magnification(self, device="haadf"): return self._call("get_magnification", {"device": device})
    def set_magnification(self, magnification, device="haadf"):
        return self._call("set_magnification", {"magnification": magnification, "device": device})

    # ---- stage ----
    def get_stage(self): return self._call("get_stage")
    def set_stage(self, sp, relative=True): return self._call("set_stage", {"stage_positions": sp, "relative": relative})

    # ---- beam ----
    def get_beam(self): return self._call("get_beam")
    def set_beam(self, bs, relative=False): return self._call("set_beam", {"beam_settings": bs, "relative": relative})

    # ---- optics / aberrations ----
    def get_optics(self): return self._call("get_optics")
    def set_optics(self, **kw): return self._call("set_optics", kw)

    # ---- imaging vs diffraction mode ----
    def get_mode(self): return self._call("get_mode")
    def set_mode(self, mode="IMG"): return self._call("set_mode", {"mode": mode})

    # ---- diffraction settings ----
    # camera_length_mm / beamstop_radius_px / aperture_um / depth_nm map to real
    # projection + selected-area controls. `use_local_atoms` is a simulation-only
    # toggle (how the twin COMPUTES diffraction) and has no real counterpart.
    def get_diffraction_settings(self): return self._call("get_diffraction_settings")
    def set_diffraction_settings(self, **kw): return self._call("set_diffraction_settings", kw)

    # ---- acquisition ----
    def acquire_image(self, device, **kw): return self._decode(self._call("acquire_image", {"device": device, **kw}))

    # ---- autofocus ----
    # A real instrument-side routine; its FAILURE behavior on the twin is driven
    # by the simulation environment (set via SimulationHarness), which is exactly
    # what lets a workflow's failure-handling be tested.
    def autofocus(self, device="haadf", z_range_um=2.0, z_steps=9):
        return self._call("autofocus", {"device": device, "z_range_um": z_range_um, "z_steps": z_steps})

    def close(self): return self._call("close")


# ===========================================================================
# Simulation harness  (twin-only; NO real-instrument counterpart)
# ===========================================================================
class SimulationHarness:
    """Twin-only configuration: choose the specimen, set the environment, inject
    drift / beam damage / contamination. A real deployment has no equivalent.

    Wraps a MicroscopeControlClient and reuses its connection for the twin's
    simulation commands. Access the control surface via `.control`."""

    def __init__(self, control_client):
        self.control = control_client

    def _call(self, method, params=None):
        # Simulation RPCs travel over the control client's transport: in the twin,
        # control and simulation commands reach the same server process.
        return self.control._call(method, params)

    # ---- specimen selection (real microscopes have a physical specimen) ----
    def list_samples(self): return self._call("list_samples")
    def get_current_sample(self): return self._call("get_current_sample")
    def load_sample(self, name, params=None, D=None, H=None, W=None):
        p = {"name": name, "params": params or {}}
        if D is not None: p["D"] = D
        if H is not None: p["H"] = H
        if W is not None: p["W"] = W
        return self._call("load_sample", p)

    # ---- simulation environment (named realism scenarios) ----
    def set_environment(self, name="pristine"): return self._call("set_environment", {"name": name})
    def get_environment(self): return self._call("get_environment")

    # ---- specimen degradation (beam damage + contamination) ----
    def get_specimen(self): return self._call("get_specimen")
    def set_specimen(self, **kw): return self._call("set_specimen", kw)
    def reset_specimen(self): return self._call("reset_specimen")

    # ---- mechanical drift injection ----
    def get_drift(self): return self._call("get_drift")
    def set_drift(self, **kw): return self._call("set_drift", kw)

    # ---- twin introspection / debugging ----
    def get_command_log(self, last_n=50): return self._call("get_command_log", {"last_n": last_n})


# ===========================================================================
# Backward-compatibility shim
# ===========================================================================
class STEMClient(MicroscopeControlClient, SimulationHarness):
    """Combined client preserving the original single-object API (control +
    simulation on one object). Provided so existing scripts keep working; new
    code should prefer MicroscopeControlClient + SimulationHarness so the
    portable surface is explicit."""

    def __init__(self, host="127.0.0.1", port=9094, timeout=30):
        MicroscopeControlClient.__init__(self, host=host, port=port, timeout=timeout)
        self.control = self  # SimulationHarness._call resolves through self


## 5. Connect & wait for readiness — control vs. simulation

The client is split into two objects to make the architecture explicit:

- **`control`** (`MicroscopeControlClient`) — the **instrument-control surface**.
  Every call (`set_stage`, `set_beam`, `set_mode`, `device_settings`,
  `acquire_image`, `autofocus`, …) has a real-microscope counterpart. On real
  hardware you **replace this object with a vendor-SDK client**; the workflow code
  above it is unchanged. This is also the surface `TwinBackend` is built on.
- **`sim`** (`SimulationHarness`) — **twin-only configuration**. Choosing the
  specimen (`load_sample`), the simulation environment (`set_environment`), and
  injecting drift / beam damage / contamination (`set_specimen`, `set_drift`) have
  **no real-instrument counterpart** — a real microscope has a physical specimen
  you prepared, not an API call. `sim` wraps `control` and reuses its connection.

A workflow that only ever touches `control.*` is, by construction, portable; the
`sim.*` calls are the scaffolding that lets you stress-test it without a real sample.

In [ ]:
import importlib, stem_client
importlib.reload(stem_client)
control = stem_client.MicroscopeControlClient(host="127.0.0.1", port=9094, timeout=120)
sim = stem_client.SimulationHarness(control)  # twin-only configuration, wraps `control`
print("Status:", control.wait_until_ready(timeout=300))
print("Detectors:", control.get_detectors())
print("Current sample:", sim.get_current_sample())
print("Environment:", sim.get_environment())


## 6. Sample registry

In [ ]:
for s in sim.list_samples():
    print(f"  {s['name']:22s} | {s['display_name']}")


## Demo A — Poisson-dose noise

Dose = beam current × dwell time. Low dose is shot-noise limited; raising current
or dwell cleans it up (noise ∝ 1/√dose). Basis for any dose-limited workflow.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
sim.set_environment("pristine")
sim.load_sample("au_dispersed", D=64, H=512, W=512)
control.set_mode("IMG")
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)

control.set_beam({"current_pA": 5}, relative=False)
control.device_settings("haadf", size=256, field_of_view_um=30.0, dwell_us=1.0, use_dose_model=1.0)
low = control.acquire_image("haadf")
control.set_beam({"current_pA": 400}, relative=False)
control.device_settings("haadf", dwell_us=50.0)
high = control.acquire_image("haadf")

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs[0].imshow(low,  cmap="gray"); axs[0].set_title("Low dose (5 pA, 1 us)");   axs[0].axis("off")
axs[1].imshow(high, cmap="gray"); axs[1].set_title("High dose (400 pA, 50 us)"); axs[1].axis("off")
plt.tight_layout(); plt.show()


## Demo B — Probe PSF, defocus & autofocus (with convergence reporting)

The PSF broadens with defocus (FOV-aware) and spherical aberration. Autofocus now
returns a `converged` flag and the sharpness curve. On a clean specimen it converges;
later we show it diverging on a beam-sensitive one.

In [ ]:
sim.set_environment("pristine")
sim.load_sample("au_dispersed", D=64, H=512, W=512)
control.set_mode("IMG")
control.set_beam({"current_pA": 100}, relative=False)
control.device_settings("haadf", size=256, field_of_view_um=30.0, dwell_us=20.0)

control.set_stage({"z": 3e-6}, relative=False)
before = control.acquire_image("haadf")
af = control.autofocus(device="haadf", z_range_um=5.0, z_steps=15)
after = control.acquire_image("haadf")
print("Autofocus:", {k: af[k] for k in ["converged","reason","best_z_um_relative","curve_contrast"]})

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(before, cmap="gray"); axs[0].set_title("Defocused (z=+3 um)"); axs[0].axis("off")
axs[1].imshow(after,  cmap="gray"); axs[1].set_title(f"After AF (converged={af['converged']})"); axs[1].axis("off")
zs=[z for z,_ in af["scores"]]; ss=[s for _,s in af["scores"]]
axs[2].plot(zs, ss, marker="o"); axs[2].set_xlabel("Z offset (um)"); axs[2].set_ylabel("Sharpness")
axs[2].set_title("Autofocus curve")
plt.tight_layout(); plt.show()


## Demo C — Unified diffraction across sample classes

Every sample's diffraction comes from the *same* atom-based engine. A single crystal
gives sharp spots; a polycrystal gives ring-like maxima; an amorphous film gives a
diffuse halo. No FFT-of-image proxy anywhere.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4.7))
panels = [("fcc_single_crystal", "single crystal → spots", {}),
          ("polycrystal_grains", "polycrystal → rings", {}),
          ("amorphous_film",     "amorphous → halo", {}),
          ("dislocation_crystal","dislocation → distorted", {})]
for ax, (name, title, params) in zip(axs, panels):
    sim.load_sample(name, params=params, D=40, H=256, W=256)
    control.set_mode("DIFF")
    control.set_diffraction_settings(aperture_um=0.4, depth_nm=20, thickness_nm=20)
    control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
    dp = control.acquire_image("haadf")
    ax.imshow(dp, cmap="inferno"); ax.axis("off"); ax.set_title(f"{name}\n{title}")
plt.tight_layout(); plt.show()


## Demo D — FCC zone-axis navigation (tilting toward ⟨110⟩)

Tilting moves between zone axes: the square [001] net evolves toward a
centered-rectangular ⟨110⟩ net as the tilt increases, passing through sparse
off-axis conditions. This is the signal a zone-axis-alignment routine keys on.
Note the stage tilt is capped at ±30° by the safety limits (Section on stage
safety), so we tilt *toward* the ⟨110⟩ zone axis (at 45°) rather than reaching it.


In [ ]:
sim.load_sample("fcc_single_crystal", D=40, H=256, W=256)
control.set_mode("DIFF")
control.set_diffraction_settings(camera_length_mm=800.0, thickness_nm=20, aperture_um=0.4, depth_nm=20)
fig, axs = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, a in zip(axs, [0, 12, 22, 28]):
    control.set_stage({"a": a, "b": 0}, relative=False)
    dp = control.acquire_image("haadf")
    label = "[001] square" if a == 0 else ("approaching zone" if a >= 26 else "off-axis")
    ax.imshow(dp, cmap="inferno"); ax.axis("off"); ax.set_title(f"FCC a={a}deg\n{label}")
plt.tight_layout(); plt.show()


## Demo E — Stage-aware diffraction (locality)

Diffraction is computed from atoms in the illuminated region. On a non-uniform sample
(polycrystal) moving the stage changes which grains are in view, so the pattern changes.
On a uniform crystal it would stay the same — the physically correct behavior.

In [ ]:
sim.load_sample("polycrystal_grains", D=40, H=256, W=256)
control.set_mode("DIFF")
control.set_diffraction_settings(aperture_um=0.3, depth_nm=20, thickness_nm=20)
positions = [(0,0), (0.4e-6, 0.3e-6), (-0.4e-6, -0.3e-6)]
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, (sx, sy) in zip(axs, positions):
    control.set_stage({"x":sx,"y":sy,"z":0,"a":0,"b":0}, relative=False)
    dp = control.acquire_image("haadf")
    ax.imshow(dp, cmap="inferno"); ax.axis("off")
    ax.set_title(f"stage ({sx*1e6:+.1f}, {sy*1e6:+.1f}) um")
plt.suptitle("Polycrystal diffraction changes with stage position (different grains illuminated)")
plt.tight_layout(); plt.show()


## Demo F — Beam damage

On a beam-sensitive specimen, cumulative dose removes signal in the exposed region.
Acquire repeatedly at high dose in one spot and watch the contrast fade — the failure
mode that breaks naive repeated-acquisition workflows.

In [ ]:
sim.set_environment("beam_sensitive")
sim.reset_specimen()
sim.load_sample("fcc_single_crystal", D=40, H=256, W=256)
control.set_mode("IMG")
control.device_settings("haadf", size=128, field_of_view_um=10.0, dwell_us=60.0)
control.set_beam({"current_pA": 500}, relative=False)
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)

frames = [control.acquire_image("haadf")]
for _ in range(6):
    frames.append(control.acquire_image("haadf"))
means = [f.astype(float).mean() for f in frames]

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(frames[0], cmap="gray"); axs[0].set_title("Frame 1 (fresh)"); axs[0].axis("off")
axs[1].imshow(frames[-1], cmap="gray"); axs[1].set_title("Frame 7 (after dose)"); axs[1].axis("off")
axs[2].plot(range(1, len(means)+1), means, marker="o"); axs[2].set_xlabel("frame #")
axs[2].set_ylabel("mean signal"); axs[2].set_title("Signal loss from beam damage")
plt.tight_layout(); plt.show()
print("Specimen state:", sim.get_specimen())


## Demo G — Contamination

Carbon-like contamination builds up where the beam dwells, progressively darkening
the image. Different failure mode from damage: the signal is *attenuated* by an
overlayer rather than destroyed.

In [ ]:
sim.set_environment("contaminating")
sim.reset_specimen()
sim.load_sample("fcc_single_crystal", D=40, H=256, W=256)
control.set_mode("IMG")
control.device_settings("haadf", size=128, field_of_view_um=10.0, dwell_us=40.0)
control.set_beam({"current_pA": 300}, relative=False)
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)

frames = [control.acquire_image("haadf")]
for _ in range(6):
    frames.append(control.acquire_image("haadf"))
means = [f.astype(float).mean() for f in frames]

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(frames[0], cmap="gray"); axs[0].set_title("Frame 1"); axs[0].axis("off")
axs[1].imshow(frames[-1], cmap="gray"); axs[1].set_title("Frame 7 (contaminated)"); axs[1].axis("off")
axs[2].plot(range(1, len(means)+1), means, marker="o"); axs[2].set_xlabel("frame #")
axs[2].set_ylabel("mean signal"); axs[2].set_title("Darkening from contamination")
plt.tight_layout(); plt.show()


## Demo H — Simulation environments & autofocus failure (three outcomes)

The *same* autofocus code is run under three conditions to show it does **not** always
converge. `pristine` converges cleanly. `low_dose` **diverges** because shot noise swamps
the sharpness signal (a flat, noisy curve). A severe beam-damage case **diverges** because
the specimen is destroyed during the Z-sweep. This is what lets you certify that your code
*detects and handles* autofocus failure before trusting it on a real, irreplaceable specimen.

In [ ]:
def try_autofocus(env, current_pA, dwell_us, extra_specimen=None):
    sim.set_environment(env)
    sim.reset_specimen()
    if extra_specimen:
        sim.set_specimen(**extra_specimen)
        sim.reset_specimen()
    sim.load_sample("au_dispersed", D=64, H=512, W=512)
    control.set_mode("IMG")
    control.device_settings("haadf", size=256, field_of_view_um=30.0, dwell_us=dwell_us)
    control.set_beam({"current_pA": current_pA}, relative=False)
    control.set_stage({"z": 3e-6}, relative=False)
    return control.autofocus(device="haadf", z_range_um=5.0, z_steps=15)

cases = [
    ("pristine (clean)",      try_autofocus("pristine", 100, 20.0)),
    ("low dose (noisy)",      try_autofocus("low_dose", 80, 2.0)),
    ("severe beam damage",    try_autofocus("beam_sensitive", 2000, 200.0,
                                            {"damage_dose_threshold": 1.0e3, "damage_rate": 1.5})),
]
for label, af in cases:
    print(f"{label:22s} -> converged={str(af['converged']):5s}  ({af['reason']})")

fig, axs = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (label, af) in zip(axs, cases):
    zs=[z for z,_ in af["scores"]]; ss=[s for _,s in af["scores"]]
    ax.plot(zs, ss, marker="o")
    ax.set_title(f"{label}\nconverged={af['converged']}")
    ax.set_xlabel("Z offset (um)"); ax.set_ylabel("sharpness")
plt.tight_layout(); plt.show()
print("\nA workflow must branch on af['converged'] -- two of three cases here fail.")


## Demo H2 — Accumulating mechanical drift

In the `thick_drifting` environment the stage drifts frame-to-frame. We acquire a short
burst at a fixed nominal position and estimate the image shift by cross-correlation: the
field visibly walks, which is the effect a drift-correction routine must compensate.

In [ ]:
from numpy.fft import fft2, ifft2
def estimate_shift(a, b):
    a = a.astype(float) - a.mean(); b = b.astype(float) - b.mean()
    c = np.abs(ifft2(fft2(a) * np.conj(fft2(b))))
    py, px = np.unravel_index(np.argmax(c), c.shape)
    sy = py - a.shape[0] if py > a.shape[0]//2 else py
    sx = px - a.shape[1] if px > a.shape[1]//2 else px
    return sx, sy

sim.set_environment("thick_drifting"); sim.reset_specimen()
sim.load_sample("au_dispersed", D=64, H=512, W=512)
control.set_mode("IMG"); control.device_settings("haadf", size=256, field_of_view_um=20.0, dwell_us=20.0)
control.set_beam({"current_pA": 150}, relative=False)
sim.set_drift(reset_accum=True)
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)

frames = [control.acquire_image("haadf") for _ in range(4)]
shifts = [estimate_shift(frames[0], f) for f in frames]
print("cumulative shift from frame 0 (px):", shifts)

fig, axs = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, f, (sx, sy) in zip(axs, frames, shifts):
    ax.imshow(f, cmap="gray"); ax.axis("off"); ax.set_title(f"shift=({sx:+d},{sy:+d})px")
plt.suptitle("thick_drifting: the field walks between frames (drift to be corrected)")
plt.tight_layout(); plt.show()


## Demo H3 — Dose series: noise versus dose

Holding the specimen fixed, we raise current × dwell across three acquisitions. Shot noise
falls as roughly 1/√dose, so the signal-to-noise improves visibly. This is the trade a
dose-limited workflow manages (image quality vs. beam damage).

In [ ]:
sim.set_environment("pristine")
sim.load_sample("au_dispersed", D=64, H=512, W=512); control.set_mode("IMG")
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)

settings = [(2, 1.0), (20, 5.0), (200, 30.0)]
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for ax, (pA, dwell) in zip(axs, settings):
    control.set_beam({"current_pA": pA}, relative=False)
    control.device_settings("haadf", size=256, field_of_view_um=30.0, dwell_us=dwell, use_dose_model=1.0)
    im = control.acquire_image("haadf").astype(float)
    snr = im.mean() / (im.std() + 1e-9)
    ax.imshow(im, cmap="gray"); ax.axis("off")
    ax.set_title(f"{pA} pA, {dwell} us\nSNR proxy = {snr:.2f}")
plt.suptitle("Higher dose -> less shot noise (SNR ~ sqrt(dose))")
plt.tight_layout(); plt.show()


## Demo H4 — Contamination leaves a footprint

We park on a small field and acquire repeatedly (concentrated dwell), then zoom out. The
previously-scanned square is darker — carbon-like contamination has built up exactly where the beam dwelled,
adding mass-thickness (HAADF signal scales with mass-thickness, so it brightens). This spatial footprint is the classic bright contamination "scan box" seen on real HAADF-STEM.

In [ ]:
sim.set_environment("contaminating"); sim.reset_specimen()
sim.load_sample("fcc_single_crystal", D=40, H=256, W=256); control.set_mode("IMG")

# 1) concentrate dose on a small central field
control.device_settings("haadf", size=128, field_of_view_um=4.0, dwell_us=60.0)
control.set_beam({"current_pA": 400}, relative=False)
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
for _ in range(6):
    control.acquire_image("haadf")

# 2) zoom out: the contaminated patch is BRIGHTER than its surroundings
control.device_settings("haadf", size=256, field_of_view_um=12.0, dwell_us=20.0)
wide = control.acquire_image("haadf").astype(float)
center = wide[112:144, 112:144].mean(); edge = wide[:32, :32].mean()
print(f"central (contaminated) mean={center:.0f}  vs  edge mean={edge:.0f}")

plt.figure(figsize=(7, 7)); plt.imshow(wide, cmap="gray"); plt.axis("off")
plt.title("Contamination footprint: bright scanned box at center"); plt.show()


## Demo H5 — Tilt rocking & the thickness relrod (diffraction)

Two diffraction effects in one cell. **Left row:** tilting a BCC crystal through several
angles rocks reflections in and out — the pattern evolves continuously. **Right row:** at a
fixed off-axis tilt, increasing the specimen thickness lengthens the relrod, so *more*
reflections satisfy the Ewald condition and additional spots appear.

In [ ]:
# Tilt rocking on BCC
sim.load_sample("bcc_single_crystal", D=40, H=256, W=256); control.set_mode("DIFF")
control.set_diffraction_settings(aperture_um=0.4, depth_nm=20, thickness_nm=20)
fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for ax, a in zip(axs[0], [0, 10, 20, 28]):
    control.set_stage({"a": a, "b": 0}, relative=False)
    ax.imshow(control.acquire_image("haadf"), cmap="inferno"); ax.axis("off")
    ax.set_title(f"BCC tilt a={a}deg")

# Thickness/relrod at a fixed off-axis tilt (a=8deg)
sim.load_sample("fcc_single_crystal", D=40, H=256, W=256)
control.set_stage({"a": 8, "b": 0}, relative=False)
for ax, t in zip(axs[1], [3, 15, 40, 80]):
    control.set_diffraction_settings(aperture_um=0.4, depth_nm=20, thickness_nm=t)
    ax.imshow(control.acquire_image("haadf"), cmap="inferno"); ax.axis("off")
    ax.set_title(f"FCC a=8deg, t={t}nm")
plt.suptitle("Top: tilt rocks reflections.  Bottom: thicker specimen -> longer relrod -> more spots")
plt.tight_layout(); plt.show()


## Demo J — Magnification and field of view

Magnification and field of view are the same quantity: `mag = k / fov`, with the
field of view in metres. The calibration constant `k` here is taken from a real
instrument (57 kx ↔ 1.6564523008 µm). Setting the magnification sets the FOV and
vice-versa; raising the magnification shrinks the field so you see a smaller region
in more detail.

In [ ]:
# The calibration: 57 kx should give a 1.6564523008 um field of view.
control.set_magnification(57000)
print("mag=57 kx ->", control.get_magnification())

# Sweep magnification and show the shrinking field of view.
sim.set_environment("pristine")
sim.load_sample("au_dispersed", D=40, H=256, W=256)
control.set_mode("IMG")
mags = [10000, 40000, 160000]
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, mag in zip(axs, mags):
    control.set_magnification(mag)
    g = control.get_magnification()
    control.device_settings("haadf", dwell_us=15.0)
    im = control.acquire_image("haadf")
    ax.imshow(im, cmap="gray"); ax.axis("off")
    ax.set_title(f"{mag/1000:.0f} kx\nFOV = {g['field_of_view_um']:.3f} um")
plt.suptitle("mag = k / fov  (higher magnification -> smaller field of view)")
plt.tight_layout(); plt.show()


## Demo K — Stage safety limits

The stage enforces travel limits, like a real instrument's soft-limits that protect
the stage, specimen, and pole-piece: **±1.5 mm in x/y, ±1 mm in z, ±30° in α/β**. A
move whose target exceeds any limit is **rejected outright** — nothing moves — and
raises a clear error. A workflow must catch this and stay within range.

In [ ]:
def attempt(desc, sp, relative=False):
    try:
        control.set_stage(sp, relative=relative)
        print(f"  OK       {desc}")
    except Exception as e:
        # the server rejects out-of-range targets with a descriptive error
        print(f"  REJECTED {desc}: {str(e).split('Stage move rejected by safety limits:')[-1].strip()[:60]}")

control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
print("Stage safety limits (x,y +/-1.5mm, z +/-1mm, a,b +/-30deg):")
attempt("x = +1.0 mm (in range)", {"x": 1.0e-3})
attempt("x = +2.0 mm (over)",     {"x": 2.0e-3})
attempt("z = +1.5 mm (over)",     {"z": 1.5e-3})
attempt("a = +25 deg (in range)", {"a": 25})
attempt("a = +45 deg (over)",     {"a": 45})
# a rejected move leaves the stage untouched:
control.set_stage({"x": 1.0e-3}, relative=False)
attempt("relative +0.8 mm from 1.0 mm -> 1.8 mm (over)", {"x": 0.8e-3}, relative=True)
print("x after the rejected relative move:", round(control.get_stage()[0]*1e3, 3), "mm (unchanged)")


## Demo L — Inherent length scale: you must zoom in to resolve features

Every sample declares an inherent feature scale (atomic-column spacing for crystals,
particle size for nanoparticles, the smallest shape for the shape assembly). When the
pixel size set by the magnification is coarser than that scale, the fine detail cannot
be resolved and is blurred away — so you have to raise the magnification to see it,
exactly as on a real instrument. (And at high magnification, drift and dose then become
the limiting factors — see Demos H2 and H3.)

In [ ]:
# shape_assembly has ~30 nm features over a 5 um field: sweep magnification and
# watch fine detail appear only once the pixels are fine enough to resolve it.
sim.set_environment("pristine")
sim.load_sample("shape_assembly", params={"num_shapes": 30}, D=40, H=256, W=256)
control.set_mode("IMG")

def high_freq_fraction(im):
    from numpy.fft import fft2, fftshift
    F = np.abs(fftshift(fft2(im.astype(float) - im.mean())))
    h, w = im.shape; cy, cx = h//2, w//2
    Y, X = np.mgrid[0:h, 0:w]; r = np.hypot(Y-cy, X-cx)
    return F[r > h*0.25].sum() / (F.sum() + 1e-9)

fovs = [5.0, 2.0, 0.6]
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, fov in zip(axs, fovs):
    control.device_settings("haadf", field_of_view_um=fov, dwell_us=15.0)
    im = control.acquire_image("haadf")
    nm_px = fov*1000/256
    ax.imshow(im, cmap="gray"); ax.axis("off")
    ax.set_title(f"FOV={fov} um ({nm_px:.1f} nm/px)\nhigh-freq detail={high_freq_fraction(im):.3f}")
plt.suptitle("Detail is resolved only as the pixel size drops below the feature scale")
plt.tight_layout(); plt.show()


## Demo I — Shape-assembly testbed (feature finding)

`shape_assembly` places random convex features (circles/ellipses, rectangles, hexagons).
It is a substrate for *feature-finding* workflows: an agent has to locate features and
decide which to study — the kind of underspecified task that motivates an LLM.

In [ ]:
sim.set_environment("pristine")
sim.load_sample("shape_assembly", params={"num_shapes": 40, "non_overlapping": True}, D=40, H=256, W=256)
control.set_mode("IMG")
control.device_settings("haadf", size=256, field_of_view_um=5.0, dwell_us=15.0)
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
img = control.acquire_image("haadf")
plt.figure(figsize=(8,8)); plt.imshow(img, cmap="gray"); plt.axis("off")
plt.title("shape_assembly — synthetic features for feature-finding workflows"); plt.show()


## 7. Portability layer — "test here, deploy there"

A workflow is written **once** against an abstract `MicroscopeBackend` interface.
Only the *binding* changes between the twin and a real instrument: the twin binding
(`TwinBackend`) is complete and working; the vendor adapters (`ThermoFisherBackend`
via AutoScript, `NionBackend` via Nion Swift, `JEOLBackend' via pyJEM) are **skeletons** 
showing the command mapping — their exact SDK method names must be verified against 
your microscope'sdocumentation. Specimen-class information (crystallinity, defects, …)
is **not** passed to the workflow; it must be detected from acquired data, exactly as on a real tool.

In [ ]:
%%writefile microscope_backend.py
"""
microscope_backend.py

Backend-abstraction layer for STEM automation workflows.

The point of this module is the reviewers' "test here, deploy there" requirement:
a workflow is written ONCE against the abstract `MicroscopeBackend` interface, and
only the *binding* changes between the digital twin and a real instrument. The twin
(STEMClient) already implements this interface; the real-microscope adapters below
show how each abstract operation maps onto a vendor SDK.

IMPORTANT (honesty note): the vendor adapters are SKELETONS. The method names used
inside them are illustrative and MUST be checked against the actual SDK documentation
for your microscope and software version (AutoScript / TEM Scripting, Nion Swift, pyJEM etc.).
They are provided to show the mapping structure, not as drop-in working code. The
abstract interface and the twin binding ARE complete and working.
"""
from abc import ABC, abstractmethod
from typing import Dict, Any, Tuple
import numpy as np


# ---------------------------------------------------------------------------
# Abstract interface that every backend (twin or real) implements
# ---------------------------------------------------------------------------
class MicroscopeBackend(ABC):
    """The operations an automation workflow is allowed to use. Write workflows
    against THIS interface so they are portable between the twin and a real tool.

    The instrument-control surface is deliberately identical across the twin and
    every vendor adapter (ThermoFisher, Nion, JEOL): same method names, same
    signatures, same units (microns / degrees / pA / kV / magnification). Only the
    binding underneath changes. This is what makes "test on the twin, deploy on the
    instrument" a genuine swap of the backend object."""

    @abstractmethod
    def get_stage(self) -> Dict[str, float]: ...
    @abstractmethod
    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False) -> None: ...
    @abstractmethod
    def get_beam(self) -> Dict[str, float]: ...
    @abstractmethod
    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True) -> None: ...
    @abstractmethod
    def set_mode(self, mode: str) -> None: ...           # "IMG" | "DIFF"
    @abstractmethod
    def set_fov_um(self, fov_um: float) -> None: ...
    @abstractmethod
    def get_magnification(self) -> Dict[str, float]: ...
    @abstractmethod
    def set_magnification(self, magnification: float) -> None: ...
    @abstractmethod
    def acquire_image(self) -> np.ndarray: ...
    @abstractmethod
    def autofocus(self) -> Dict[str, Any]: ...


# ---------------------------------------------------------------------------
# Binding 1: the digital twin (COMPLETE, working)
# ---------------------------------------------------------------------------
class TwinBackend(MicroscopeBackend):
    """Adapts the digital-twin control client to the abstract interface. This
    binding is complete: workflows written against MicroscopeBackend run unchanged
    here. It is constructed from a MicroscopeControlClient (the portable control
    surface) -- NOT the SimulationHarness -- which makes the layering explicit:
    the backend contract is implemented over instrument control only, never over
    simulation-only configuration."""

    def __init__(self, control_client, device="haadf"):
        # `control_client` is a MicroscopeControlClient (or the combined STEMClient,
        # which is also a MicroscopeControlClient). Simulation setup (load_sample,
        # set_environment, ...) is done separately via SimulationHarness and is not
        # visible to portable workflow code.
        self.stem = control_client
        self.device = device

    def get_stage(self):
        x, y, z, a, b = self.stem.get_stage()
        # twin stores x/y/z in metres; expose microns at the interface
        return {"x": x*1e6, "y": y*1e6, "z": z*1e6, "a": a, "b": b}

    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        # interface uses microns for x/y/z; twin expects metres
        sp = {}
        if x is not None: sp["x"] = x*1e-6
        if y is not None: sp["y"] = y*1e-6
        if z is not None: sp["z"] = z*1e-6
        if a is not None: sp["a"] = a
        if b is not None: sp["b"] = b
        self.stem.set_stage(sp, relative=relative)

    def get_beam(self):
        return self.stem.get_beam()

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        # Matches the ThermoFisher/JEOL adapters: beam changes are gated behind a
        # `disabled` flag so a workflow can't alter the beam by accident. Pass
        # disabled=False to actually apply the change.
        if disabled:
            print("Changing beam settings is disabled for safety. "
                  "To enable, call set_beam(..., disabled=False).")
            return
        bs = {}
        if current_pA is not None: bs["current_pA"] = current_pA
        if voltage_kV is not None: bs["voltage_kV"] = voltage_kV
        self.stem.set_beam(bs, relative=False)

    def set_mode(self, mode):
        self.stem.set_mode(mode)

    def set_fov_um(self, fov_um):
        self.stem.device_settings(self.device, field_of_view_um=float(fov_um))

    def get_magnification(self):
        return self.stem.get_magnification(device=self.device)

    def set_magnification(self, magnification):
        return self.stem.set_magnification(magnification, device=self.device)

    def acquire_image(self):
        return self.stem.acquire_image(self.device)

    def autofocus(self):
        return self.stem.autofocus(device=self.device)


# ---------------------------------------------------------------------------
# Binding 2: Thermo Fisher (FEI) via AutoScript / TEM Scripting  -- SKELETON
# ---------------------------------------------------------------------------
class ThermoFisherBackend(MicroscopeBackend):
    """Skeleton mapping to Thermo Fisher's AutoScript TEM Python API.

    The attribute paths below (microscope.specimen.stage, .optics, .acquisition...)
    follow the GENERAL structure of AutoScript but the EXACT names/units MUST be
    verified against your installed AutoScript version's documentation. Units in
    AutoScript are SI (metres, radians, amperes); we convert at the boundary so the
    workflow keeps using um / pA / kV / degrees.
    """
    def __init__(self, microscope):
        # `microscope` is the connected AutoScript Microscope() object
        self.m = microscope

    def get_stage(self):
        p = self.m.specimen.stage.position           # SI: m, rad
        return {"x": p.x*1e6, "y": p.y*1e6, "z": p.z*1e6,
                "a": np.degrees(p.a), "b": np.degrees(p.b)}

    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        from autoscript_tem_microscope_client.structures import StagePosition  # illustrative import
        target = StagePosition()
        if x is not None: target.x = x*1e-6
        if y is not None: target.y = y*1e-6
        if z is not None: target.z = z*1e-6
        if a is not None: target.a = np.radians(a)
        if b is not None: target.b = np.radians(b)
        if relative:
            self.m.specimen.stage.relative_move(target)
        else:
            self.m.specimen.stage.absolute_move(target)

    def get_beam(self):
        return {"current_pA": self.m.optics.beam_current*1e12,
                "voltage_kV": self.m.optics.high_tension/1e3}

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        if disabled:
            print('Changing beam settings is disabled for safety. To enable, please run set_beam(... disabled=False)')
        else:
            if current_pA is not None:
                self.m.optics.beam_current = current_pA*1e-12
            if voltage_kV is not None:
                self.m.optics.high_tension = voltage_kV*1e3

    def set_mode(self, mode):
        # Map to the instrument's imaging vs diffraction projection mode.
        self.m.optics.projection_mode = ("DIFFRACTION" if mode.upper()=="DIFF" else "IMAGING")

    def set_fov_um(self, fov_um):
        # AutoScript often controls FOV via magnification or HFW (horizontal field width)
        self.m.optics.horizontal_field_width = fov_um*1e-6

    def get_magnification(self):
        # AutoScript exposes magnification directly; if you drive FOV via HFW,
        # derive it from the same MAG_K calibration used by the twin.
        try:
            mag = float(self.m.optics.magnification)
        except Exception:
            hfw_um = float(self.m.optics.horizontal_field_width) * 1e6
            mag = (57000.0 * 1.6564523008e-6) / (hfw_um * 1e-6)
        hfw_um = float(getattr(self.m.optics, "horizontal_field_width", 0.0)) * 1e6
        return {"magnification": mag, "field_of_view_um": hfw_um}

    def set_magnification(self, magnification):
        # Prefer the instrument's native magnification control; verify the exact
        # attribute against your AutoScript version.
        self.m.optics.magnification = float(magnification)

    def acquire_image(self):
        from autoscript_tem_microscope_client.enumerations import DetectorType  # illustrative
        img = self.m.acquisition.acquire_stem_image()   # returns AdornedImage
        return np.asarray(img.data)

    def autofocus(self):
        # Many systems expose an auto-function; otherwise implement a Z sweep here.
        self.m.auto_functions.run_auto_focus()
        return {"converged": True, "reason": "vendor auto-focus (no diagnostic returned)"}


# ---------------------------------------------------------------------------
# Binding 3: Nion microscopes via the Nion Swift API  -- SKELETON
# ---------------------------------------------------------------------------
class NionBackend(MicroscopeBackend):
    """Skeleton mapping to the Nion Swift / nionswift-instrumentation API.

    Nion's API is Python-native and open; the calls below follow its general shape
    (stem_controller, scan, etc.) but EXACT names MUST be verified against the
    nionswift-instrumentation version you run.
    """
    def __init__(self, stem_controller, scan_controller):
        self.stem = stem_controller
        self.scan = scan_controller

    def get_stage(self):
        # Nion exposes control values via stem_controller.GetVal / TryGetVal
        gx = self.stem.TryGetVal("StageOutX"); gy = self.stem.TryGetVal("StageOutY")
        gz = self.stem.TryGetVal("StageOutZ")
        return {"x": (gx[1] if gx[0] else 0.0)*1e6, "y": (gy[1] if gy[0] else 0.0)*1e6,
                "z": (gz[1] if gz[0] else 0.0)*1e6, "a": 0.0, "b": 0.0}

    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        cur = self.get_stage() if relative else {"x":0,"y":0,"z":0}
        if x is not None: self.stem.SetVal("StageInX", (cur["x"]+x if relative else x)*1e-6)
        if y is not None: self.stem.SetVal("StageInY", (cur["y"]+y if relative else y)*1e-6)
        if z is not None: self.stem.SetVal("StageInZ", (cur["z"]+z if relative else z)*1e-6)
        # tilt (a/b) maps to the relevant Nion control if the stage supports it

    def get_beam(self):
        v = self.stem.TryGetVal("EHT")
        return {"current_pA": float("nan"), "voltage_kV": (v[1]/1e3 if v[0] else float("nan"))}

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        if disabled:
            print('Changing beam settings is disabled for safety. To enable, please run set_beam(... disabled=False)')
        else:
            if voltage_kV is not None:
                self.stem.SetVal("EHT", voltage_kV*1e3)
            # beam current on Nion is typically set via aperture / gun controls

    def set_mode(self, mode):
        # Nion is scan-based; "DIFF" would switch to a Ronchigram/diffraction camera
        pass

    def set_fov_um(self, fov_um):
        self.scan.set_frame_parameters({"fov_nm": fov_um*1e3})

    def get_magnification(self):
        # Nion is FOV-native; derive magnification from the same MAG_K calibration.
        fp = self.scan.get_frame_parameters()
        fov_um = float(fp.get("fov_nm", 0.0)) / 1e3
        mag = (57000.0 * 1.6564523008e-6) / (fov_um * 1e-6) if fov_um > 0 else float("nan")
        return {"magnification": mag, "field_of_view_um": fov_um}

    def set_magnification(self, magnification):
        fov_um = (57000.0 * 1.6564523008e-6) / float(magnification) * 1e6
        self.scan.set_frame_parameters({"fov_nm": fov_um*1e3})

    def acquire_image(self):
        data_and_metadata = self.scan.grab_next_to_finish()[0]
        return np.asarray(data_and_metadata.data)

    def autofocus(self):
        # Implement a Z-sweep using set_stage(z=...) + a sharpness metric, or call
        # a tuning routine if available.
        return {"converged": True, "reason": "implement Z-sweep or vendor tuning"}

# ---------------------------------------------------------------------------
# Binding 4: JEOL microscopes via the JEOL PyJEM API  -- SKELETON
# ---------------------------------------------------------------------------
class JEOLBackend(MicroscopeBackend):
    """Skeleton mapping to the JEOL PyJEM instrumentation API.

    JEOL's PyJEM API is Python-native and open.
    """
    def __init__(self, TEM3, detector): # TEM3 and detector are importable packages
        self.TEM3 = TEM3
        self.stage = TEM3.Stage3()
        self.deflector = TEM3.Def3()
        self.eos = TEM3.EOS3()
        self.feg = TEM3.FEG3()
        
        self.haadf = detector.Detector('HAADF')

    def get_stage(self):
        pos = self.stage.GetPos() # output in nm
        return {"x": pos[0], "y": pos[1], "z": pos[2], "a": pos[3], "b": pos[4]}
    
    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        # Todo: WHAT ARE THE UNITS OF THE SIMULATION? M OR NM?
        cur = self.get_stage() if relative else {"x":0,"y":0,"z":0}
        if x is not None: self.stage.SetX("StageInX", (cur["x"]+x if relative else x)*1e-9)
        if y is not None: self.stage.SetY("StageInY", (cur["y"]+y if relative else y)*1e-9)
        if z is not None: self.stage.SetZ("StageInZ", (cur["z"]+z if relative else z)*1e-9)
        self.stage.SetTiltXAngle(a) # not certain about units. check relative motion
        self.stage.SetTiltYAngle(b) # not certain about units. check relative motion
        # JEOL has motor / piezo switching

    def get_beam(self):
        return {"current_pA": self.TEM3.GUN3().GetHtCurrentValue(), # CHECK UNITS
                "voltage_kV": self.TEM3.HT3().GetHtValue()}

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        if disabled:
            print('Changing beam settings is disabled for safety. To enable, please run set_beam(... disabled=False)')
        else:
            if voltage_kV is not None:
                TEM3.HT3().GetHtValue(voltage_kV)
            # Todo: Add current changing

    def set_mode(self, mode):
        # Todo: Add
        pass

    def set_fov_um(self, fov_um):
        # Todo: Add
        pass

    def get_magnification(self):
        # JEOL exposes magnification via EOS3 (e.g. GetMagValue). Verify the exact
        # call/units against your PyJEM version.
        try:
            mag = float(self.eos.GetMagValue()[0])
        except Exception:
            mag = float("nan")
        fov_um = ((57000.0 * 1.6564523008e-6) / mag) * 1e6 if mag == mag and mag > 0 else float("nan")
        return {"magnification": mag, "field_of_view_um": fov_um}

    def set_magnification(self, magnification):
        # EOS3 typically sets magnification by index/selector; map your desired
        # magnification onto the instrument's mag table. Verify against PyJEM docs.
        # self.eos.SetSelector(index)  # illustrative
        pass

    def acquire_image(self):
        image_data = self.haadf.snapshot_rawdata()
        return image_data

    def autofocus(self):
        # Implement a Z-sweep using set_stage(z=...) + a sharpness metric, or call
        # a tuning routine if available.
        return {"converged": True, "reason": "implement Z-sweep or vendor tuning"}

# ---------------------------------------------------------------------------
# A workflow written against the ABSTRACT interface -- runs on ANY backend
# ---------------------------------------------------------------------------
def survey_then_focus(backend: MicroscopeBackend, fov_um=20.0, current_pA=100.0):
    """Trivial portability demo: same code drives the twin or a real microscope.
    Only the `backend` object differs at the call site."""
    backend.set_beam(current_pA=current_pA, disabled=False)  # beam safety opt-in
    backend.set_mode("IMG")
    backend.set_fov_um(fov_um)
    backend.set_stage(x=0, y=0, z=0, a=0, b=0, relative=False)
    af = backend.autofocus()
    img = backend.acquire_image()
    return {"autofocus": af, "image_shape": tuple(np.asarray(img).shape)}


In [ ]:
from microscope_backend import TwinBackend, survey_then_focus
backend = TwinBackend(control)
sim.set_environment("pristine")
sim.load_sample("au_dispersed", D=64, H=512, W=512)

# This workflow is backend-agnostic: the SAME call runs on a real microscope by
# passing ThermoFisherBackend(...) or NionBackend(...) instead of TwinBackend(control).
result = survey_then_focus(backend, fov_um=30.0, current_pA=100.0)
print("Portable workflow on the twin:", result)
print("\nDeploying to a real tool = swap the backend object; workflow code is unchanged.")


## 8. An ambiguously-specified workflow (where an agent earns its place)

"Survey the sample, find isolated features, and acquire detailed data on the most
promising ones." This is hard to fully pre-script: the features must be **detected**
from the image, and *which* are 'isolated' / 'promising' and *when to stop* are
judgement calls. The functions below expose those judgement points as discrete steps
an LLM agent can orchestrate; a simple built-in policy lets the demo run autonomously.

This directly addresses the concern that an LLM is superfluous for fully-specified
workflows (it is) — its value is in workflows like this one.

In [ ]:
%%writefile feature_finding.py
"""
feature_finding.py

An ambiguously-specified workflow: "survey the sample, find isolated features, and
acquire detailed data on the most promising ones." This is the class of task that
motivates an LLM/agent layer (review 5): the decision logic -- what counts as a
feature, which are 'isolated', which are 'promising', when to stop -- is awkward to
fully pre-script, and the features must be DETECTED from the acquired images, not
supplied as input (review 4).

The functions here are deliberately written as discrete decision steps so an agent
(LLM or otherwise) can orchestrate them. We provide a simple built-in policy so the
workflow also runs autonomously for demonstration; an LLM would replace the policy
with its own judgement (e.g. richer notions of 'promising', adaptive stopping).
"""
import numpy as np


def detect_features(image, bg_percentile=70, min_area_px=12):
    """Detect bright blobs in a survey image WITHOUT being told where they are.
    Returns a list of {centroid_px, area_px, mean_intensity, bbox}."""
    img = image.astype(np.float32)
    thr = np.percentile(img, bg_percentile)
    mask = img > thr
    # connected components via scipy if available, else a simple flood label
    try:
        from scipy import ndimage
        lab, n = ndimage.label(mask)
        feats = []
        for i in range(1, n+1):
            ys, xs = np.where(lab == i)
            if len(xs) < min_area_px:
                continue
            feats.append({
                "centroid_px": (float(xs.mean()), float(ys.mean())),
                "area_px": int(len(xs)),
                "mean_intensity": float(img[ys, xs].mean()),
                "bbox": (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
            })
        return feats
    except ImportError:
        return []


def score_isolation(feats, image_extent_px):
    """For each feature, compute how isolated it is (distance to nearest neighbour,
    normalized by image_extent_px). Higher = more isolated. A DECISION input."""
    if len(feats) < 2:
        for f in feats:
            f["isolation"] = 1.0
        return feats
    cents = np.array([f["centroid_px"] for f in feats])
    for i, f in enumerate(feats):
        d = np.hypot(cents[:,0]-cents[i,0], cents[:,1]-cents[i,1])
        d[i] = np.inf
        nn = d.min()
        f["isolation"] = float(nn / max(1.0, float(image_extent_px)))
    return feats


def rank_candidates(feats, prefer="isolated"):
    """Rank features by a policy. An LLM agent would supply its own ranking
    rationale here (e.g. 'isolated AND mid-sized AND high-contrast'). The built-in
    policies are intentionally simple."""
    if not feats:
        return []
    if prefer == "isolated":
        key = lambda f: f["isolation"]
    elif prefer == "braightest":
        key = lambda f: f["mean_intensity"]
    elif prefer == "largest":
        key = lambda f: f["area_px"]
    else:
        # composite: isolated, reasonably large, bright
        key = lambda f: f["isolation"] * np.sqrt(f["area_px"]) * f["mean_intensity"]
    return sorted(feats, key=key, reverse=True)


def feature_to_stage_um(feature, fov_um, image_shape, stage_center_um=(0.0, 0.0)):
    """Convert a feature's pixel centroid in the survey image to a stage position
    (um) so we can drive the beam to it for a detailed acquisition."""
    H, W = image_shape
    cx_px, cy_px = feature["centroid_px"]
    # pixel offset from image center -> fraction of FOV -> microns
    dx_um = (cx_px - W/2.0) / W * fov_um
    dy_um = (cy_px - H/2.0) / H * fov_um
    return (stage_center_um[0] + dx_um, stage_center_um[1] + dy_um)


def survey_and_characterize(backend, survey_fov_um=5.0, detail_fov_um=1.0,
                            max_targets=3, prefer="composite",
                            min_isolation=0.05, log=print):
    """The full ambiguous workflow, runnable on any MicroscopeBackend.

    Steps (each is a place an LLM agent could intervene):
      1. Low-mag survey acquisition.
      2. Detect features from the image (not supplied as input).
      3. Score isolation and rank by a policy.
      4. Decide which targets are worth detailed study (stopping/selection logic).
      5. Drive to each target and acquire a detailed image.
    Returns a record of decisions + detailed acquisitions.
    """
    backend.set_mode("IMG")
    backend.set_beam(current_pA=120, disabled=False)   # beam safety opt-in

    # 1. survey
    backend.set_fov_um(survey_fov_um)
    backend.set_stage(x=0, y=0, z=0, a=0, b=0, relative=False)
    survey = np.asarray(backend.acquire_image())
    log(f"[survey] acquired {survey.shape} at {survey_fov_um} um FOV")

    # 2. detect
    feats = detect_features(survey)
    log(f"[detect] found {len(feats)} candidate features")
    if not feats:
        return {"survey_shape": tuple(survey.shape), "features": [], "targets": []}

    # 3. score + rank
    feats = score_isolation(feats, max(survey.shape))
    ranked = rank_candidates(feats, prefer=prefer)

    # 4. SELECTION / STOPPING DECISION (agent would own this judgement)
    #    Here: keep only sufficiently-isolated features, take the top-N.
    selected = [f for f in ranked if f["isolation"] >= min_isolation][:max_targets]
    log(f"[select] {len(selected)} targets pass isolation>={min_isolation} "
        f"(of {len(ranked)} ranked); studying top {len(selected)}")

    # 5. detailed acquisition at each target
    details = []
    for idx, f in enumerate(selected):
        sx, sy = feature_to_stage_um(f, survey_fov_um, survey.shape)
        backend.set_stage(x=sx, y=sy, relative=False)   # interface uses microns
        backend.set_fov_um(detail_fov_um)
        af = backend.autofocus()
        dimg = np.asarray(backend.acquire_image())
        details.append({"target_index": idx,
                        "stage_um": (sx, sy),
                        "isolation": f["isolation"],
                        "area_px": f["area_px"],
                        "autofocus_converged": af.get("converged", None),
                        "detail_shape": tuple(dimg.shape)})
        log(f"[detail {idx}] stage=({sx:+.2f},{sy:+.2f})um isolation={f['isolation']:.3f} "
            f"AF={af.get('converged')}")
        # restore survey FOV for the next target hop
        backend.set_fov_um(survey_fov_um)

    return {"survey_shape": tuple(survey.shape),
            "n_features": len(feats),
            "n_selected": len(selected),
            "targets": details}


In [ ]:
from feature_finding import survey_and_characterize
sim.set_environment("pristine")
sim.load_sample("shape_assembly", params={"num_shapes": 35, "non_overlapping": True}, D=40, H=256, W=256)
control.device_settings("haadf", size=256, dwell_us=15.0)

backend = TwinBackend(control)
record = survey_and_characterize(backend, survey_fov_um=5.0, detail_fov_um=1.0,
                                 max_targets=3, prefer="composite", min_isolation=0.04)
print(f"\nDetected {record['n_features']} features; studied {record['n_selected']} in detail.")
for t in record["targets"]:
    print(f"  target {t['target_index']}: stage={tuple(round(v,2) for v in t['stage_um'])} um, "
          f"isolation={t['isolation']:.3f}, AF converged={t['autofocus_converged']}")

# An LLM agent would replace the fixed policy (isolation threshold, ranking,
# stopping rule) with its own judgement -- e.g. "prefer cubic, well-separated
# particles and stop once 3 good ones are characterized."


## 9. Loading your own atomistic structures (Atomsk / LAMMPS / XYZ / CIF)

The `atomsk_polycrystal` sample loads a real atomistic file and uses its atoms for
both imaging and diffraction. This is how you bring an *externally-generated*
structure (e.g. an Atomsk polycrystal, an MD snapshot, a relaxed defect cell) into
the twin.

**Supported formats:** `.xyz` (no dependencies), `.lmp`/`.data` (LAMMPS, Atomsk's
default), `.cfg`, `.cif`, `POSCAR`, `.xsf` (the non-XYZ formats need `ase` installed).

**Producing a file with Atomsk** (run on your own machine, then upload the output):
```
atomsk --create fcc 4.05 Au seed.cfg
atomsk --polycrystal seed.cfg poly.txt poly.xyz
```
Then upload `poly.xyz` to the Colab session (Files panel) and point the loader at it.

**Key parameters** (`sim.load_sample("atomsk_polycrystal", params={...})`):
- `file_path` — path to your structure file (default `sample_data/polycrystal.xyz`).
- `auto_fit` (default `True`) — stretch the structure's bounding box to fill the
  volume. Set `False` to use `scale_factor` for an explicit voxels-per-Angstrom map.
- `scale_factor` — voxels per Angstrom, only used when `auto_fit=False`.

The cell below **generates a small synthetic `.xyz`** (so the notebook runs without
Atomsk installed) and loads it. Replace `file_path` with your own upload to use real data.

In [ ]:
import os, numpy as np
os.makedirs("sample_data", exist_ok=True)

# --- Make a small synthetic polycrystal .xyz (stand-in for an Atomsk output) ---
# Three FCC Au grains at random orientations. Replace this whole block by simply
# uploading your own file and setting file_path to it.
a = 4.05
basis = np.array([[0,0,0],[0,.5,.5],[.5,0,.5],[.5,.5,0]])
rng = np.random.default_rng(3)
def rot(seed):
    r = np.random.default_rng(seed); ax = r.normal(size=3); ax/=np.linalg.norm(ax)
    th = r.uniform(0,2*np.pi)
    K = np.array([[0,-ax[2],ax[1]],[ax[2],0,-ax[0]],[-ax[1],ax[0],0]])
    return np.eye(3)+np.sin(th)*K+(1-np.cos(th))*(K@K)
atoms = []
for g in range(3):
    R = rot(10+g); centre = rng.uniform(-40, 40, 3)
    for i in range(7):
        for j in range(7):
            for k in range(7):
                for b in basis:
                    atoms.append(R @ ((np.array([i,j,k]) + b) * a) + centre)
atoms = np.array(atoms)
with open("sample_data/my_structure.xyz", "w") as f:
    f.write(f"{len(atoms)}\nsynthetic 3-grain FCC Au\n")
    for p in atoms:
        f.write(f"Au {p[0]:.3f} {p[1]:.3f} {p[2]:.3f}\n")
print(f"wrote sample_data/my_structure.xyz ({len(atoms)} atoms)")

# --- Load it into the twin ---
sim.load_sample("atomsk_polycrystal",
                params={"file_path": "sample_data/my_structure.xyz", "auto_fit": True},
                D=40, H=256, W=256)
print("current sample:", sim.get_current_sample()["name"])

# Image it, then take a diffraction pattern from a small region
control.set_mode("IMG"); control.device_settings("haadf", size=200, field_of_view_um=20.0, dwell_us=15.0)
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
img = control.acquire_image("haadf")
control.set_mode("DIFF"); control.set_diffraction_settings(aperture_um=2.0, depth_nm=20, thickness_nm=20)
diff = control.acquire_image("haadf")

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs[0].imshow(img, cmap="gray"); axs[0].set_title("Loaded structure (IMG)"); axs[0].axis("off")
axs[1].imshow(diff, cmap="inferno"); axs[1].set_title("Diffraction from the loaded atoms"); axs[1].axis("off")
plt.tight_layout(); plt.show()
print("To use YOUR data: upload a file and set params={'file_path': 'your_file.xyz'}.")


# 10. Worked examples — putting it together

The cells below combine the building blocks (samples, IMG/DIFF modes, environments,
stage control) into small end-to-end studies. They are meant as templates you can
copy and adapt. None of these are special-cased in the engine — each is just normal
use of the control + simulation API.

### Example 1 — Diffraction fingerprints across sample classes

The same diffraction engine produces qualitatively different patterns depending on
the specimen's structure: sharp spots (single crystal), a few rotated spot-sets or
ring tendency (polycrystal), a diffuse halo (amorphous), and a distorted spot
pattern (defected crystal). This is the visual signature a phase/structure
classifier would key on.

In [ ]:
gallery = [
    ("fcc_single_crystal",  20.0, 0.4,  "FCC single crystal"),
    ("polycrystal_grains",   2.0, 0.15, "Polycrystal (one grain)"),
    ("amorphous_film",       0.5, 0.4,  "Amorphous film"),
    ("dislocation_crystal",  0.5, 0.4,  "Dislocated crystal"),
]
sim.set_environment("pristine")
fig, axs = plt.subplots(2, 4, figsize=(18, 9))
for col, (name, fov, ap, label) in enumerate(gallery):
    sim.load_sample(name, D=40, H=256, W=256)
    control.set_mode("IMG")
    control.device_settings("haadf", size=160, field_of_view_um=fov, dwell_us=15.0)
    control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
    img = control.acquire_image("haadf")
    control.set_mode("DIFF")
    control.set_diffraction_settings(aperture_um=ap, depth_nm=20, thickness_nm=20)
    control.set_stage({"a":0,"b":0}, relative=False)
    dp = control.acquire_image("haadf")
    axs[0, col].imshow(img, cmap="gray"); axs[0, col].axis("off"); axs[0, col].set_title(label)
    axs[1, col].imshow(dp, cmap="inferno"); axs[1, col].axis("off")
axs[0, 0].text(-0.1, 0.5, "IMG", transform=axs[0,0].transAxes, rotation=90, va="center", fontsize=12)
axs[1, 0].text(-0.1, 0.5, "DIFF", transform=axs[1,0].transAxes, rotation=90, va="center", fontsize=12)
plt.suptitle("Structure -> diffraction fingerprint (same engine, different specimens)")
plt.tight_layout(); plt.show()


### Example 2 — Grain mapping by stage scan

Park a small selected-area aperture and step the stage across a polycrystal. As the
aperture crosses grain boundaries, the single-crystal pattern *rotates* to the next
grain's orientation. This is the core of an automated orientation-mapping routine —
and the patterns come from the same atomic model as the image.

In [ ]:
sim.set_environment("pristine")
sim.load_sample("polycrystal_grains", params={"n_grains": 4, "seed": 7}, D=40, H=256, W=256)

# survey image first
control.set_mode("IMG"); control.device_settings("haadf", size=200, field_of_view_um=2.0, dwell_us=15.0)
control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
survey = control.acquire_image("haadf")

# step a small aperture across x; record the diffraction at each stop
control.set_mode("DIFF"); control.set_diffraction_settings(aperture_um=0.12, depth_nm=20, thickness_nm=20)
xs_um = np.linspace(-0.7, 0.7, 5)
diffs = []
for x_um in xs_um:
    control.set_stage({"x": x_um*1e-6, "y": 0.0, "a": 0, "b": 0}, relative=False)
    diffs.append(control.acquire_image("haadf"))

fig = plt.figure(figsize=(18, 4.6))
ax0 = fig.add_subplot(1, 6, 1); ax0.imshow(survey, cmap="gray"); ax0.set_title("survey"); ax0.axis("off")
for i, (x_um, dp) in enumerate(zip(xs_um, diffs)):
    ax = fig.add_subplot(1, 6, i+2)
    ax.imshow(dp, cmap="inferno"); ax.axis("off"); ax.set_title(f"x={x_um:+.2f}um")
plt.suptitle("Grain mapping: the diffraction pattern changes as the aperture crosses grains")
plt.tight_layout(); plt.show()


### Example 3 — Dose-budget study on a beam-sensitive specimen

A practical trade-off: higher dose gives a cleaner image but damages the specimen.
We image the same beam-sensitive specimen at increasing dose and watch both effects —
image quality improves, then the specimen degrades. A real dose-management routine
has to find the sweet spot before the structure is lost.

In [ ]:
sim.set_environment("beam_sensitive")
fig, axs = plt.subplots(1, 4, figsize=(18, 4.7))
doses = [(20, 5.0), (80, 15.0), (300, 40.0), (800, 80.0)]
for ax, (pA, dwell) in zip(axs, doses):
    sim.reset_specimen()                       # fresh specimen for each dose
    sim.load_sample("fcc_single_crystal", D=40, H=256, W=256)
    control.set_mode("IMG")
    control.device_settings("haadf", size=160, field_of_view_um=12.0, dwell_us=dwell)
    control.set_beam({"current_pA": pA}, relative=False)
    control.set_stage({"x":0,"y":0,"z":0,"a":0,"b":0}, relative=False)
    # acquire a few frames so cumulative damage at high dose shows
    for _ in range(3):
        img = control.acquire_image("haadf")
    snr = img.mean() / (img.std() + 1e-9)
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title(f"{pA} pA, {dwell} us\nSNR proxy={snr:.2f}")
plt.suptitle("Dose budget: image improves with dose, but a sensitive specimen degrades")
plt.tight_layout(); plt.show()


### Example 4 — Zone-axis alignment as a workflow

Bring a crystal to its zone axis by stepping tilt and watching the diffraction
symmetry. Off-axis the pattern is sparse and asymmetric; on-axis the full symmetric
net appears. A real alignment routine would score symmetry and drive the tilt toward
the maximum — here we just show the progression the routine would follow.

In [ ]:
sim.set_environment("pristine")
sim.load_sample("fcc_single_crystal", D=40, H=256, W=256)
control.set_mode("DIFF")
control.set_diffraction_settings(aperture_um=0.4, depth_nm=20, thickness_nm=20)

tilts = [18, 12, 6, 0]   # walking toward the zone axis
fig, axs = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, a in zip(axs, tilts):
    control.set_stage({"a": a, "b": 0}, relative=False)
    dp = control.acquire_image("haadf").astype(float)
    # a crude "symmetry/intensity" score the routine might maximize
    score = (dp > 0.3*dp.max()).sum()
    ax.imshow(dp, cmap="inferno"); ax.axis("off")
    ax.set_title(f"tilt a={a}deg\nbright-spot count={score}")
plt.suptitle("Zone-axis alignment: sparse off-axis -> full symmetric net on-axis")
plt.tight_layout(); plt.show()


## What is realistic, and what is still simplified

**Now modelled:** Poisson dose statistics; finite probe / FOV-aware defocus & Cs blur;
accumulating mechanical drift + line jitter; **beam damage** (dose-driven signal loss)
and **contamination** (dwell-driven darkening); a **single atom-based diffraction engine**
giving spots (crystal), ring-like maxima (polycrystal) and diffuse halos (amorphous), with
Ewald-sphere geometry, structure-factor intensities, a thickness-dependent relrod, and
**stage-position-dependent (local) diffraction**; **simulation environments** bundling these
into named scenarios; and **failure-aware autofocus** that can diverge.

**Still simplified (and explicitly not claimed):**
- **No dynamical (multiple) scattering.** Diffraction and image contrast are kinematical.
  There are no extinction contours, thickness fringes, channelling-driven HAADF contrast,
  Kikuchi lines, or HOLZ rings. Workflows whose decisions depend on *dynamical intensities*
  (e.g. precise atom-counting from HAADF intensity, thickness from extinction) will not
  transfer quantitatively. **Future work:** couple the twin to a dynamical engine
  (abTEM, Prismatic, or py4DSTEM's simulation modules) for the diffraction/imaging channel,
  keeping this twin's acquisition/automation interface unchanged.
- **Atom budget capped at 100k**, so polycrystal patterns show correct ring *radii* but
  are not fully continuous powder rings (those need thousands of grains / millions of atoms,
  i.e. GPU).
- **Approximate detector calibration:** camera length is a relative zoom, not an absolute
  mm→mrad mapping; form factors use a simple Z-weighted envelope.

These are acceptable for the intended purpose — a **testing ground for acquisition and
decision logic** — where a workflow that behaves correctly here (handles drift, damage,
low dose, AF failure, locality) is far more likely to transfer to a real instrument than
one developed against an idealized simulator. Absolute physical constants will still differ.

## New / changed API quick reference

- `set_environment(name)` — `pristine` | `beam_sensitive` | `contaminating` | `thick_drifting` | `low_dose`
- `get_environment()`
- `set_specimen(beam_damage_enabled=, contamination_enabled=, damage_dose_threshold=, damage_rate=, contamination_rate=)`
- `get_specimen()` — includes accumulated-dose / contamination summaries
- `reset_specimen()` — fresh specimen (clears damage + contamination)
- `autofocus(...)` — now returns `converged`, `reason`, `curve_contrast`, `n_candidate_peaks`
- `set_diffraction_settings(camera_length_mm=, beamstop_radius_px=, thickness_nm=, aperture_um=, depth_nm=, use_local_atoms=)`
- Diffraction is atom-based for every sample; `aperture_um` (0=auto/FOV) sets the selected-area region, `depth_nm` (0=auto) the through-thickness extent.

## Portability & workflows (sections 7–8)

- `microscope_backend.MicroscopeBackend` — abstract interface; write workflows against it.
- `TwinBackend(stem)` — complete binding to the twin. `ThermoFisherBackend`, `NionBackend` — vendor adapter skeletons (verify SDK calls).
- `feature_finding.survey_and_characterize(backend, ...)` — an ambiguous workflow that detects features from images, ranks by isolation, and characterizes the most promising — the decision points an LLM agent would own.

## Adding crystals / structures

Any sample that implements `get_atoms_in_region(cx_um, cy_um, half_width_um, depth_nm)`
returning `(positions_Å, atomic_numbers)` automatically gets correct diffraction. Crystalline
samples build atoms by tiling a `CrystalLattice`; particle samples record their particle list
and let the base class fill them; defected/amorphous samples place atoms with the appropriate
displacement field or disorder.
